<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/NLP-2026/Lecture_5/%D0%9B%D0%B5%D0%BA%D1%86%D0%B8%D1%8F_5_1_%D0%92%D0%B2%D0%B5%D0%B4%D0%B5%D0%BD%D0%B8%D0%B5_%D0%B2_Retrieval_Augmented_Generation_(RAG).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лекция 5.1. Введение в Retrieval-Augmented Generation (RAG)



## Тема 1. Что такое RAG и зачем он нужен

### 1.1. Определение Retrieval-Augmented Generation

**Retrieval-Augmented Generation (RAG)** — это гибридная архитектура обработки естественного языка, которая объединяет два ключевых компонента: *информационный поиск (retrieval)* и *генерацию текста (generation)*. В отличие от классических поисковых систем, которые возвращают список документов или ссылок, RAG использует найденные документы как динамический контекст для генерации связного, фактологически обоснованного ответа на пользовательский запрос.

**Исторический контекст.** Концепция RAG была впервые предложена в 2020 году исследователями из Meta AI (тогда Facebook AI) в статье *«Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks»* (Lewis et al., 2020). Авторы показали, что комбинирование параметрической памяти (веса предобученной языковой модели) и непараметрической памяти (внешний индекс документов) позволяет значительно улучшить качество ответов на задачи, требующие глубоких предметных знаний. С тех пор RAG стал одним из наиболее востребованных подходов в области прикладного NLP, особенно после появления мощных открытых LLM и векторных баз данных.

**Три столпа RAG.** Архитектура RAG опирается на три фундаментальных компонента:

1. **Ретривер (Retriever)** — отвечает за поиск релевантных документов или фрагментов текста из внешнего корпуса по запросу пользователя. Обычно ретривер представляет собой модель эмбеддингов (например, *sentence‑transformers* или *BGE*), которая преобразует запрос и документы в векторное пространство, а затем выполняет поиск ближайших соседей с использованием косинусной близости или других метрик.

2. **Генератор (Generator)** — языковая модель (LLM), которая принимает на вход запрос и найденные ретривером документы (контекст) и генерирует итоговый ответ. Генератор может быть как локальной моделью (например, LLaMA, Qwen, Mistral), так и облачной (GPT‑4, Claude). Важно, чтобы модель имела достаточный размер контекстного окна для размещения всех релевантных фрагментов.

3. **Интеграция (Integration)** — механизм, связывающий ретривер и генератор. Он включает в себя формирование промпта, в который вставляются найденные документы (обычно с указанием источника), а также постобработку ответа (добавление ссылок, проверка фактов). Интеграция также может включать в себя дополнительные этапы, такие как переранжирование результатов, фильтрацию по метаданным, или даже многократные циклы поиска-генерации (как в Self‑RAG или Agentic RAG).

**Сравнение RAG с альтернативами.** Чтобы лучше понять место RAG в ландшафте технологий, сравним его с классическим поиском и с чистой LLM.

| Критерий | Классический поиск (например, Google) | Чистая LLM (без внешних данных) | RAG |
| :--- | :--- | :--- | :--- |
| **Источник знаний** | Индекс веб-страниц или внутренних документов | Исключительно веса модели (параметрическая память) | Внешние документы + параметрическая память |
| **Актуальность знаний** | Мгновенное обновление индекса (для веба – почти real‑time) | Фиксирована на момент обучения (срез данных) | Зависит от частоты обновления базы документов |
| **Формат ответа** | Список ссылок или кратких сниппетов | Связный текст, сгенерированный моделью | Связный текст с возможными ссылками на источники |
| **Галлюцинации** | Отсутствуют (поиск не генерирует факты) | Высокий риск галлюцинаций (особенно на узкие темы) | Значительно ниже, чем у чистой LLM, за счёт привязки к контексту |
| **Прозрачность** | Всегда видно источники | Невозможно проверить, откуда взята информация | Можно явно указать, из каких документов взят факт |
| **Стоимость на запрос** | Низкая (индекс уже построен) | Средняя (зависит от размера модели) | Выше, чем у LLM (из-за дополнительного поиска), но дешевле fine‑tuning |

Из таблицы видно, что RAG занимает «золотую середину»: он даёт связные, обоснованные ответы с возможностью проверки источников и обновления знаний без переобучения модели.

### 1.2. Проблемы, решаемые RAG

RAG был разработан для преодоления фундаментальных ограничений как классических поисковых систем, так и автономных языковых моделей. Рассмотрим шесть ключевых проблем, которые эффективно решает RAG, и приведём реальные примеры.

**1. Актуальность знаний.** LLM имеют «срез» знаний на момент окончания обучения. Например, GPT‑4 (сентябрь 2021) не знает о событиях после этой даты. Если пользователь спросит: «Какие изменения в налоговом кодексе РФ вступили в силу с 2025 года?», LLM либо даст устаревшую информацию, либо признается в незнании. RAG же может обратиться к актуальной базе законодательных документов, найти свежий текст поправок и сгенерировать ответ, основанный на них.

*Реальный пример:* юридическая компания внедряет RAG-систему, которая ежедневно индексирует новые судебные решения и законы. Адвокат задаёт вопрос о недавнем прецеденте – система находит нужное дело и выдаёт краткое резюме со ссылкой на источник.

**2. Приватность и безопасность данных.** Многие организации не могут отправлять свои внутренние документы в облачные API (из-за GDPR, коммерческой тайны или политик безопасности). Традиционный подход fine‑tuning требует передачи данных поставщику модели или их размещения на собственных GPU, что дорого и не всегда возможно. RAG позволяет хранить все документы локально (в векторной базе внутри корпоративного контура) и использовать локальную LLM (например, через Ollama или vLLM), что гарантирует, что данные никогда не покидают защищённую среду.

*Пример:* банк использует RAG для ответов на вопросы сотрудников о внутренних регламентах. Все документы хранятся на серверах банка, модель также запускается локально. Сотрудник спрашивает: «Какие лимиты на переводы для VIP-клиентов?» – система находит актуальный регламент и генерирует ответ, не раскрывая данные вовне.

**3. Специализированные домены.** В таких областях, как медицина, юриспруденция, техническая документация, LLM часто не имеет достаточной глубины знаний или оперирует общими сведениями. Например, редкое генетическое заболевание может быть описано в единичных клинических рекомендациях, которые не попали в обучающий корпус модели. RAG позволяет подключить специализированную базу (например, PubMed, коллекцию клинических протоколов) и получать точные ответы, основанные на экспертных источниках.

*Пример:* врач-онколог запрашивает рекомендации по лечению редкой саркомы. RAG ищет в базе клинических исследований за последние 5 лет, находит несколько подходящих статей и выдаёт обобщённые рекомендации с цитированием первоисточников.

**4. Прозрачность и проверяемость.** Одна из главных проблем LLM – «чёрный ящик»: невозможно узнать, на основании каких фактов модель выдала ответ. RAG решает эту проблему, предоставляя возможность показать пользователю, из каких конкретно документов была извлечена информация. Это повышает доверие, особенно в критических областях (медицина, юриспруденция, финансы).

*Пример:* студент пишет курсовую работу по истории и спрашивает: «Каковы причины Февральской революции 1917 года?» RAG находит несколько академических монографий, выдаёт ответ и в конце указывает: «Источники: Иванов А.А. 'Причины Февральской революции', 2010, стр. 45–47; Петров Б.В. 'Россия в 1917 году', 2015, стр. 112–115». Студент может проверить первоисточники.

**5. Экономическая эффективность.** Полная тонкая настройка (full fine‑tuning) модели, особенно большой (70B+), требует десятков и сотен тысяч долларов на вычислительные ресурсы и специалистов. RAG же требует лишь однократной индексации документов (что сравнительно дёшево) и выполнение поиска на каждый запрос (операция с вычислительной сложностью O(log N) для ANN-индексов). По оценкам, стоимость одного RAG-запроса на 10–100 раз дешевле, чем обучение модели на новых данных.

*Пример:* стартап создаёт чат-бота для поддержки клиентов интернет-магазина. Вместо того чтобы дообучать модель на сотнях тысяч тикетов, они индексируют базу знаний (статьи, инструкции, политики) и используют RAG. Это позволяет быстро запустить систему и легко обновлять знания при появлении новых товаров или правил, без затрат на переобучение.

**6. Динамическое обновление знаний.** В мире, где информация меняется ежедневно, поддерживать актуальность модели через переобучение практически невозможно. RAG позволяет обновлять знания простым добавлением, удалением или изменением документов в базе. Например, если вышла новая версия технического регламента, достаточно загрузить новый PDF-файл в систему, и уже через несколько минут модель сможет отвечать на вопросы по нему.

*Пример:* производитель программного обеспечения выпускает еженедельные патчи и обновления документации. Техподдержка использует RAG-систему: инженеры добавляют новые релиз-ноуты в базу, и чат-бот сразу начинает давать корректные ответы о новых функциях и исправленных багах, без какого-либо переобучения.

### 1.3. Основные сценарии применения RAG

RAG находит применение в самых разных отраслях. Рассмотрим шесть наиболее распространённых сценариев с пояснением, какие именно проблемы RAG решает в каждом случае.

**1. Корпоративные системы поиска по внутренней документации.** Крупные компании имеют тысячи страниц политик, инструкций, технической документации, отчётов. Сотрудникам часто трудно найти нужную информацию, даже при наличии поиска. RAG позволяет задать вопрос на естественном языке и получить точный ответ с указанием источника. *Решаемые проблемы:* специализированный домен, приватность, прозрачность.

**2. Интеллектуальные FAQ и чат‑боты для поддержки клиентов.** Традиционные FAQ статичны и не покрывают все возможные вопросы. Чат-боты на основе LLM часто галлюцинируют, если не имеют доступа к актуальной базе знаний. RAG-чат-бот подключается к базе статей поддержки, форумов, руководств и выдаёт точные ответы, снижая нагрузку на операторов. *Решаемые проблемы:* актуальность знаний, экономическая эффективность, динамическое обновление.

**3. Юридические ассистенты.** Юристы и адвокаты работают с огромным объёмом законов, постановлений, судебных прецедентов. RAG позволяет быстро находить релевантные нормы и прецеденты, формулировать правовые заключения, проверять согласованность документов. *Решаемые проблемы:* специализированный домен, прозрачность, актуальность.

**4. Медицинские ассистенты.** Врачи могут использовать RAG для поиска по клиническим рекомендациям, фармакологическим справочникам, результатам клинических испытаний. Система помогает быстро принимать решения, особенно в редких или сложных случаях. *Решаемые проблемы:* специализированный домен, проверяемость, актуальность (если база регулярно обновляется).

**5. Образовательные платформы.** Студенты и преподаватели могут задавать вопросы по учебникам, лекциям, научным статьям. RAG не только даёт ответ, но и указывает, где в учебнике эта тема освещена, что способствует самостоятельному изучению. *Решаемые проблемы:* прозрачность, динамическое обновление (при добавлении новых материалов).

**6. Научно-исследовательские системы.** Исследователи тратят до 40% времени на обзор литературы. RAG-система может помочь найти релевантные статьи, сравнить результаты экспериментов, выделить ключевые методы. Интеграция с arXiv, PubMed и другими базами делает этот процесс значительно эффективнее. *Решаемые проблемы:* специализированный домен, актуальность, прозрачность.

### 1.4. Ключевые преимущества RAG перед альтернативами

Подход RAG имеет ряд существенных преимуществ, подтверждённых как теоретическими исследованиями, так и практическим опытом.

**1. Снижение галлюцинаций на 50–70%.** В исследованиях (например, работах по Self‑RAG и CRAG) показано, что использование внешнего контекста уменьшает вероятность фактологических ошибок. Например, в бенчмарке *Natural Questions* RAG-модели достигают точности на 15–20% выше, чем чистая LLM. При этом генерация с привязкой к документам даёт меньше вымысла. Цифра 50–70% — это усреднённое улучшение по метрикам faithfulness (верность фактам) в различных экспериментах.

**2. Обновление знаний без переобучения.** В отличие от fine‑tuning, где для обновления знаний нужно заново обучать модель (что требует времени и ресурсов), RAG позволяет просто добавить новые документы в индекс. Это делает систему готовой к работе с новыми данными в течение минут.

**3. Возможность ссылаться на источники – повышение доверия.** Возможность указать, из какого документа взята информация, критична для корпоративных, юридических и медицинских приложений. Пользователь может проверить первоисточник, что значительно повышает доверие к системе и снижает риски неправильных решений.

**4. Гибкость и модульность архитектуры.** RAG построен на заменяемых компонентах: ретривер, модель эмбеддингов, векторная БД, LLM, промпт-инжиниринг – каждый из них можно обновлять или заменять независимо. Например, можно перейти с open‑source модели на GPT‑4, не меняя остальную систему, или заменить Chroma на Milvus для масштабирования.

**5. Экономия вычислительных ресурсов.** Обучение или дообучение большой модели требует тысяч GPU‑часов и стоит десятки тысяч долларов. RAG же требует только однократной индексации документов (которая выполняется на CPU) и лёгких поисковых запросов. Стоимость одного RAG-запроса (с учётом поиска и генерации) на порядок ниже, чем у fine‑tuned модели.

### 1.5. Визуализация: архитектура RAG и сравнение с альтернативами

**Схема «RAG vs LLM vs Search» (текстовое описание):**

- **Поиск (Search):** Пользователь → запрос → поисковый движок → список документов/ссылок → пользователь (без генерации связного ответа).
- **Чистая LLM:** Пользователь → запрос → LLM (генерирует из весов) → ответ (без ссылок, возможны галлюцинации).
- **RAG:** Пользователь → запрос → ретривер → поиск в векторной БД → контекст (документы) → формирование промпта (контекст + запрос) → LLM → ответ со ссылками → пользователь.

Ниже представлена диаграмма в формате Mermaid, показывающая взаимодействие компонентов как для этапа индексации (offline), так и для этапа инференса (online).

```mermaid
flowchart TD
    subgraph Offline["Этап индексации (offline)"]
        A[Документы] --> B[Извлечение текста и очистка]
        B --> C[Разбиение на чанки]
        C --> D[Генерация эмбеддингов]
        D --> E[Сохранение в векторную БД]
        E --> F[(Векторная база данных)]
    end

    subgraph Online["Этап инференса (online)"]
        Q[Запрос пользователя] --> G[Генерация эмбеддинга запроса]
        G --> H[Поиск ближайших соседей в векторной БД]
        H --> I[Извлечение топ-k чанков]
        I --> J[Переранжирование / фильтрация]
        J --> K[Формирование промпта с контекстом]
        K --> L[LLM генерация ответа]
        L --> M[Постобработка / добавление ссылок]
        M --> R[Ответ пользователю]
    end

    F --> H
    style F fill:#f9f,stroke:#333,stroke-width:2px
```

На схеме видно, что этап индексации и этап инференса разделены: индексация выполняется один раз (или периодически), а инференс — каждый запрос.

### 1.6. Сквозные примеры работы RAG

**Пример 1. Налоговый кодекс.** Пользователь (бухгалтер) спрашивает: *«Какие изменения по налогу на прибыль вступили в силу с января 2025 года?»*. Система RAG:
1. Преобразует запрос в эмбеддинг.
2. Ищет в векторной БД, содержащей все актуальные законодательные документы (например, тексты законов, постановлений, разъяснений Минфина).
3. Находит несколько релевантных фрагментов: текст поправок, даты вступления, комментарии.
4. Формирует промпт: *«Контекст: [фрагмент 1], [фрагмент 2]. Вопрос: ...»*.
5. LLM генерирует ответ: *«С 1 января 2025 года ставка налога на прибыль для ИТ-компаний снижена с 20% до 17%. Также изменён порядок расчёта амортизации по нематериальным активам (см. ст. 259.3 НК РФ). Источник: Федеральный закон № 123-ФЗ от 20.12.2024, ст. 1, п. 3»*.
В ответе указана конкретная статья и документ, что позволяет бухгалтеру проверить информацию.

**Пример 2. Редкое заболевание.** Врач-онколог спрашивает: *«Какие схемы лечения саркомы Юинга рекомендуются для детей младше 5 лет?»*. Система RAG:
1. Ищет в базе клинических рекомендаций (например, NCCN, PubMed, локальные протоколы).
2. Находит несколько релевантных статей и клинических случаев.
3. Формирует промпт с контекстом из 3–5 источников.
4. LLM генерирует ответ: *«Согласно клиническим рекомендациям NCCN 2024, для детей до 5 лет предпочтительна схема VAC (винкристин, актиномицин, циклофосфамид) с уменьшенными дозировками. Также рекомендуется проведение локальной лучевой терапии после 3 циклов химиотерапии (источники: NCCN Guidelines, версия 2.2024, стр. 12; статья Smith et al., JCO 2023)»*.
Врач получает конкретные рекомендации с источниками, что повышает уверенность в правильности решения.

### 1.7. Математические основы (интуиция)

На данном этапе мы не будем углубляться в детали, но дадим интуитивное понимание ключевой математической операции RAG — оценки релевантности.

В основе векторного поиска лежит преобразование текста в векторы фиксированной размерности (эмбеддинги) с помощью нейросетевых моделей (например, *sentence‑transformers*). Затем релевантность между запросом и документом измеряется через **косинусное расстояние** (или косинусную близость):

$$
\text{cosine\_similarity}(q, d) = \frac{\mathbf{q} \cdot \mathbf{d}}{\|\mathbf{q}\| \cdot \|\mathbf{d}\|}
$$

где $\mathbf{q}$ — вектор запроса, $\mathbf{d}$ — вектор документа. Значение близости лежит в диапазоне $[-1, 1]$, причём значение, близкое к 1, означает высокую релевантность (векторы сонаправлены). Поиск ближайших соседей выполняется с использованием приближённых алгоритмов (ANN), таких как HNSW или IVF, чтобы обеспечить миллисекундную задержку даже для миллиардов документов. Более подробно математика будет рассмотрена в следующих темах.

### 1.8. Контрольные вопросы и задания

**Вопросы для самопроверки (с ответами):**

1. *Чем RAG отличается от тонкой настройки (fine‑tuning) языковой модели?*  
   **Ответ:** RAG не изменяет веса модели, а использует внешний поиск для предоставления актуального контекста на каждый запрос. Fine‑tuning изменяет веса модели на основе размеченных данных, что требует значительных вычислительных ресурсов и переобучения при обновлении знаний. RAG дешевле, проще в обновлении и обеспечивает прозрачность источников.

2. *Какие проблемы решает RAG, которые не решаются классическим поиском?*  
   **Ответ:** Классический поиск возвращает документы, но не генерирует связный ответ, не может переформулировать информацию и не адаптируется к контексту диалога. RAG же генерирует структурированный, понятный ответ, извлекает ключевую информацию из нескольких источников и может указывать ссылки.

3. *Почему RAG считается более экономически эффективным, чем fine‑tuning?*  
   **Ответ:** Fine‑tuning требует дорогостоящих GPU-часов (например, дообучение модели 70B может стоить >100 000 $). RAG требует лишь однократной индексации документов (выполняется на CPU) и лёгких поисковых операций на каждый запрос, что на несколько порядков дешевле при большом количестве запросов.

**Практические задания:**

1. Найдите в интернете не менее трёх примеров реального внедрения RAG в компаниях (можно поискать кейсы на сайтах Pinecone, Weaviate, Qdrant, а также в блогах по AI). Опишите каждое внедрение: какая задача решалась, какие компоненты использовались (ретривер, векторная БД, LLM), какие результаты были достигнуты. Сделайте краткий обзор (1–2 страницы).

2. Напишите эссе (объём 1–2 страницы) на тему: *«Почему в 2024–2026 годах RAG стал более популярным подходом, чем тонкая настройка моделей?»*. В эссе обязательно затроньте: а) рост популярности векторных баз данных; б) появление качественных open‑source LLM; в) требования к приватности и актуальности знаний; г) экономические факторы; д) сравнение с другими методами (fine‑tuning, промпт-инжиниринг). Аргументируйте свою точку зрения.

### 1.9. Список литературы для углублённого изучения

1. **Lewis, P., Perez, E., Piktus, A., et al. (2020).** *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks*. – arXiv:2005.11401. Оригинальная статья, заложившая основы RAG.

2. **Gao, Y., Xiong, Y., Gao, X., et al. (2023).** *Retrieval-Augmented Generation for Large Language Models: A Survey*. – arXiv:2312.10997. Обзор современных подходов к RAG, включая продвинутые архитектуры.

3. **Asai, A., et al. (2023).** *Self-RAG: Learning to Retrieve, Generate, and Critique through Self-Reflection*. – ICLR 2024. Статья о Self‑RAG, одном из самых влиятельных расширений RAG.

4. **Yan, S., et al. (2024).** *Corrective Retrieval Augmented Generation*. – Работа по CRAG, описывающая механизмы исправления ошибок поиска.

5. **Hugging Face Blog.** *Retrieval-Augmented Generation (RAG) – A Comprehensive Guide*. – https://huggingface.co/blog/rag. Практическое руководство по реализации RAG с открытым кодом.

6. **Pinecone Documentation.** *RAG with LLMs*. – https://docs.pinecone.io/docs/rag. Описание практических подходов с использованием Pinecone.

7. **Weaviate Blog.** *The Complete Guide to Retrieval-Augmented Generation*. – https://weaviate.io/blog. Цикл статей с примерами кода и архитектурными решениями.




## Тема 2. Архитектура RAG-системы

После того как мы определили, что такое RAG и какие задачи он решает, необходимо погрузиться в его внутреннее устройство. Понимание архитектуры — ключ к грамотному проектированию, оптимизации и отладке систем, работающих в реальных условиях. В этой теме мы детально разберём все компоненты RAG, этапы его работы, основные типы архитектур, сравним RAG с альтернативными подходами и дадим практические рекомендации по выбору.

---

### 2.1. Основные компоненты архитектуры

Любая RAG-система состоит из пяти функциональных модулей. Их можно представить как последовательный конвейер, в котором каждый модуль выполняет строго определённую задачу, а также как набор независимых блоков, которые можно заменять, улучшать или масштабировать.

**1. Модуль индексации (Ingestion).** Это «фабрика знаний» системы. Он отвечает за подготовку документов к поиску и выполняется в фоновом режиме (offline). Основные шаги:

- **Загрузка** — получение документов из различных источников (локальные папки, базы данных, веб-API, облачные хранилища). Инструменты: `pypdf`, `python-docx`, `beautifulsoup4`, `pandas`, а также фреймворки `LangChain` и `LlamaIndex`, предоставляющие унифицированные загрузчики (`DirectoryLoader`, `WebBaseLoader`).
- **Предобработка и очистка** — удаление шума, нормализация текста, исправление кодировок, извлечение основного содержимого (без рекламы, навигации и т.д.). Часто применяются библиотеки `re`, `ftfy`, `textwrap`, а для работы с HTML — `readability-lxml`.
- **Чанкинг (разбиение на фрагменты)** — деление длинных документов на смысловые блоки фиксированного размера или по предложениям/абзацам. От размера чанка зависит качество поиска: слишком маленький чанк теряет контекст, слишком большой — вносит шум. Инструменты: `RecursiveCharacterTextSplitter` (LangChain), `SentenceTransformersTokenTextSplitter`.
- **Генерация эмбеддингов** — преобразование текстовых чанков в числовые векторы с помощью предобученных моделей эмбеддингов. Популярные модели: `all‑MiniLM‑L6‑v2` (быстрая, 384d), `BAAI/bge‑base‑en‑v1.5` (высокое качество, 768d), `intfloat/multilingual‑e5‑large` (многоязычная, 1024d). Реализация: `sentence‑transformers`, `HuggingFace Embedding` классы.
- **Сохранение в векторной базе данных** — запись полученных векторов вместе с метаданными (имя документа, номер страницы, дата и т.д.) в специализированное хранилище. Популярные ВБД: `Chroma` (для прототипов), `FAISS` (высокопроизводительная библиотека, но без встроенного хранения метаданных), `Qdrant`, `Weaviate`, `Milvus`, `Pinecone`.

**2. Модуль поиска (Retriever).** Это «глаза» системы. Он получает запрос пользователя и находит наиболее релевантные фрагменты в индексированном корпусе. Поиск может быть:

- **Векторный (семантический)** — использует эмбеддинги запроса и документов, вычисляет косинусную близость. Работает хорошо для запросов, сформулированных на естественном языке, и улавливает смысловые нюансы.
- **Лексический (на основе ключевых слов)** — использует инвертированные индексы и алгоритмы типа BM25 (Okapi BM25) или TF‑IDF. Хорош для точных совпадений, кодов, номеров, имён собственных, но игнорирует семантику.
- **Гибридный** — комбинирует результаты векторного и лексического поиска (например, взвешенная сумма баллов или реранкинг). Даёт наилучшее качество в широком спектре запросов.

Инструменты: для векторного поиска — `FAISS`, `Chroma`; для лексического — `rank_bm25`, `elasticsearch`; для гибридного — `Weaviate` (с поддержкой BM25 + векторного), а также ручная реализация с объединением результатов.

**3. Модуль генерации (Generator).** Это «голос» системы. Он принимает на вход запрос и контекст (найденные документы) и генерирует связный ответ. Генератор — это языковая модель (LLM), которая может быть:

- **Локальной** — запущенной на собственных серверах (например, `Qwen2.5‑7B`, `Llama‑3.2‑3B`, `Mistral‑7B` через `Ollama`, `vLLM`, `llama.cpp`). Обеспечивает приватность, но требует вычислительных ресурсов.
- **Облачной** — доступ к API от OpenAI (GPT‑4, GPT‑4o), Anthropic (Claude), Cohere (Command), Google (Gemini). Даёт высокое качество, но платное и требует передачи данных.

Важно, чтобы модель имела достаточное контекстное окно для размещения всех релевантных чанков (обычно 4–8 чанков по 256–512 токенов).

**4. Модуль интеграции (Integration).** Это «мозг», связывающий поиск и генерацию. Он включает:

- **Формирование промпта** — создание текста, который подаётся на вход LLM. Промпт обычно содержит системную инструкцию (задающую роль модели), контекст (найденные чанки с возможными ссылками), и сам вопрос пользователя. Важно правильно структурировать контекст, чтобы модель не перепутала источник и не начала галлюцинировать.
- **Управление длиной контекста** — если общая длина чанков превышает допустимый лимит, применяется усечение (например, оставляем только топ‑K самых релевантных) или суммаризация.
- **Постобработка ответа** — добавление ссылок на источники, проверка согласованности, форматирование (Markdown, таблицы, списки).

Инструменты: ручное шаблонирование с `f‑strings` или использование `LangChain` (класс `PromptTemplate`), `ChatPromptTemplate`.

**5. Модуль обратной связи (Feedback).** Это система сбора данных для непрерывного улучшения. Он включает:

- **Логирование** всех запросов, найденных документов, сгенерированных ответов, времён выполнения.
- **Сбор метрик** качества — автоматических (BLEU, ROUGE, faithfulness) и пользовательских (лайки/дизлайки, оценки). Инструменты: `RAGAS`, `DeepEval`, `TruLens`.
- **Анализ ошибок** — выявление частых проблем (например, частые случаи «не найдено»), чтобы скорректировать чанкинг или поиск.
- **Итеративное улучшение** — обновление индекса, переобучение модели эмбеддингов, настройка параметров реранкинга на основе собранной обратной связи.

---

### 2.2. Этапы работы RAG (offline и online)

Процесс функционирования RAG-системы чётко делится на две фазы: подготовительную (индексацию) и исполнительную (инференс). Понимание каждого этапа важно для оптимизации.

#### Offline (индексация)

Выполняется один раз (или периодически) перед началом работы системы.

| Шаг | Описание | Ключевые параметры, влияющие на качество/скорость |
| :--- | :--- | :--- |
| **1. Загрузка документов** | Сбор всех исходных документов из различных источников. | Формат, объём, необходимость обновления. |
| **2. Предобработка** | Очистка, извлечение основного текста, удаление шумов. | Качество парсинга (важно для PDF и HTML). |
| **3. Чанкинг** | Разбиение текста на фрагменты (например, по 500 токенов с overlap 50). | `chunk_size`, `chunk_overlap`, стратегия (recursive, semantic). Влияют на recall и precision. |
| **4. Генерация эмбеддингов** | Преобразование каждого чанка в вектор фиксированной размерности. | Модель эмбеддингов, размерность, использование GPU/CPU. |
| **5. Построение индекса** | Сохранение векторов и метаданных в ВБД с построением индекса (HNSW, IVF). | Тип индекса, параметры (M, ef_construction для HNSW), влияют на скорость поиска и точность. |

#### Online (инференс)

Выполняется для каждого пользовательского запроса.

| Шаг | Описание | Ключевые параметры, влияющие на качество/скорость |
| :--- | :--- | :--- |
| **1. Запрос пользователя** | Пользователь вводит вопрос или сообщение. | Язык, длина, сложность. |
| **2. (Опционально) Переформулировка** | Улучшение запроса (например, добавление синонимов, уточнение местоимений). | Модель для переформулировки, качество исходного вопроса. |
| **3. Поиск (Retrieval)** | Поиск по индексу: вычисляется эмбеддинг запроса, выполняется ANN-поиск. | `top_k` (число возвращаемых чанков), тип индекса, фильтрация по метаданным. |
| **4. Реранкинг (опционально)** | Уточнение порядка найденных чанков с помощью cross‑encoder. | Модель cross‑encoder, количество чанков для реранкинга. |
| **5. Формирование промпта** | Сборка системной инструкции, контекста из релевантных чанков и вопроса пользователя. | Шаблон промпта, максимальная длина контекста. |
| **6. Генерация (LLM)** | Подача промпта в LLM и получение ответа. | Модель LLM, параметры генерации (temperature, top_p, max_tokens). |
| **7. Постобработка** | Добавление ссылок на источники, форматирование, проверка фактов. | Правила добавления ссылок, шаблоны вывода. |

---

### 2.3. Типы RAG-систем

Архитектуры RAG можно классифицировать по уровню сложности и функциональности.

**Naive RAG** — это базовый вариант, в котором есть ровно один шаг поиска и один шаг генерации. Нет переранжирования, гибридного поиска, фильтрации. Подходит для простых FAQ, где документы хорошо структурированы, а запросы типовые. Недостатки: может возвращать нерелевантные чанки, не учитывает синонимы, не использует сложную логику.

**Advanced RAG** — включает улучшения:
- **Гибридный поиск** — комбинация векторного и BM25 для учета и семантики, и точных совпадений.
- **Реранкинг** — применение cross‑encoder для уточнения порядка чанков.
- **Фильтрация по метаданным** — например, только документы за последний год, или только от определённого автора.
- **Адаптивный чанкинг** — например, семантическое разбиение.
Такой подход даёт значительный прирост качества (на 5–15% по метрикам) ценой небольшого увеличения времени ответа.

**Modular RAG** — архитектура, где каждый компонент (парсер, сплиттер, модель эмбеддингов, ВБД, реранкер, LLM) является независимым модулем с чётким интерфейсом. Модули можно заменять, комбинировать, переиспользовать в разных проектах. Это самый гибкий подход, рекомендуемый для продакшена, особенно когда система будет развиваться.

**Гибридный поиск** часто выделяют как отдельную разновидность, так как сочетание BM25 и векторного поиска — один из самых эффективных способов увеличить recall. Реализуется либо через объединение ранжированных списков, либо через использование специальных ВБД с поддержкой гибридных запросов (например, Weaviate).

**Agentic RAG** — следующий уровень эволюции, где LLM выступает в роли агента, который сам принимает решения: когда искать, что именно искать, в каком порядке выполнять операции. Агент может разбить сложный запрос на несколько подзапросов, выполнять поиск в разных базах, проверять полученные факты и, если нужно, уточнять запрос. Этот подход реализуется с использованием фреймворков типа `LangGraph` или `AutoGen` и позволяет решать самые сложные задачи, но требует больше вычислительных ресурсов и тщательной отладки.

**Сравнительная таблица типов RAG-систем**

| Критерий | Naive RAG | Advanced RAG | Modular RAG | Agentic RAG |
| :--- | :--- | :--- | :--- | :--- |
| **Сложность реализации** | Низкая | Средняя | Выше средней | Высокая |
| **Качество ответа** | Среднее (зависит от данных) | Хорошее (выше на 10–20%) | Очень хорошее (гибкость) | Потенциально наилучшее (адаптивность) |
| **Время ответа (латенси)** | Низкое | Умеренное (реранкинг + гибрид) | Умеренное | Может быть высоким (много шагов) |
| **Гибкость** | Низкая (всё зафиксировано) | Средняя | Высокая (замена модулей) | Очень высокая (решения агента) |
| **Стоимость (на запрос)** | Низкая | Средняя | Средняя | Высокая (больше вызовов LLM) |
| **Обновляемость** | Простая (обновление индекса) | Простая | Простая | Сложная (логика агента) |

---

### 2.4. Сравнение RAG с другими подходами

Чтобы правильно выбрать технологию для своей задачи, полезно сравнить RAG с альтернативными методами.

**RAG vs Fine‑tuning (дообучение).** Fine‑tuning изменяет веса модели под конкретную задачу, что требует дорогих GPU-ресурсов и размеченных данных. Оно даёт хорошее качество, если задача требует изменения стиля или манеры ответа (например, моделирование личности). Однако fine‑tuning «замораживает» знания: для обновления информации нужно переобучать модель. RAG же позволяет использовать свежие данные без переобучения, дешевле и прозрачнее. **Рекомендация:** для фактологических вопросов → RAG; для изменения стиля/формата → fine‑tuning; в идеале — комбинация (fine‑tuning для стиля + RAG для фактов).

**RAG vs Prompt Engineering.** Промпт-инжиниринг (разработка шаблонов вопросов, инструкций) — это бесплатный способ улучшить ответы LLM, но он не добавляет модели новых знаний. Промпт только меняет форму ответа или способ рассуждения. RAG же снабжает модель конкретными фактами. Промпт-инжиниринг всегда используется вместе с RAG (для правильного оформления контекста), но сам по себе не решает проблему актуальности знаний.

**RAG vs Semantic Search (семантический поиск).** Семантический поиск возвращает список документов, отсортированных по релевантности. Это полезно для исследователей, но не даёт готового ответа. RAG идёт дальше, используя документы как контекст для генерации связного ответа. Это особенно важно в диалоговых системах и для пользователей, которые не хотят читать много документов.

**Рекомендации по выбору:**

| Тип задачи | Рекомендуемый подход |
| :--- | :--- |
| Ответы на фактологические вопросы, работа с документами, поддержка клиентов | **RAG** |
| Требуется изменить стиль общения или жанр (формальный → неформальный) | **Fine‑tuning** (или комбинация) |
| Нет данных, но нужно улучшить логику ответов | **Prompt Engineering** |
| Пользователь хочет сам изучить документы | **Semantic Search** |
| Сложные, многошаговые задачи с необходимостью принятия решений | **Agentic RAG** |

---

### 2.5. Визуализация архитектуры

#### Диаграмма потока данных (offline/online) в Mermaid

```mermaid
flowchart TD
    subgraph Offline["Offline: Индексация"]
        A[Документы] --> B[Очистка и извлечение текста]
        B --> C[Чанкинг]
        C --> D[Генерация эмбеддингов]
        D --> E[Сохранение в векторную БД]
        E --> F[(Векторная БД с индексом)]
    end

    subgraph Online["Online: Инференс"]
        G[Запрос пользователя] --> H[Переформулировка запроса (опционально)]
        H --> I[Вычисление эмбеддинга запроса]
        I --> J[Поиск в векторной БД]
        J --> K[Реранкинг (опционально)]
        K --> L[Формирование промпта\n(система + контекст + вопрос)]
        L --> M[LLM генерация]
        M --> N[Постобработка\n(добавление ссылок, формат)]
        N --> O[Ответ пользователю]
    end

    F --> J
    style F fill:#f9f,stroke:#333,stroke-width:2px
```

#### Схема архитектуры с компонентами

Ниже представлена схема, показывающая взаимодействие компонентов в RAG-системе.

```mermaid
flowchart LR
    subgraph Ingestion["Модуль индексации"]
        direction LR
        P[Parser] --> S[Splitter]
        S --> E[Embedder]
        E --> V[Vector DB]
    end

    subgraph Retrieval["Модуль поиска"]
        direction TB
        Q[Запрос] --> QE[Embedder]
        QE --> VS[Векторный поиск]
        VS --> RR[Реранкер] --> R[Результаты]
    end

    subgraph Generation["Модуль генерации"]
        direction TB
        R --> PF[Промпт-инжиниринг]
        PF --> LLM[LLM] --> Post[Постобработка]
    end

    subgraph Feedback["Модуль обратной связи"]
        direction LR
        Metrics[Сбор метрик] --> Logs[Логирование]
        Logs --> Analysis[Анализ] --> Update[Обновление параметров]
    end

    Ingestion --> Retrieval
    Retrieval --> Generation
    Generation --> Feedback
    Feedback --> Ingestion
    Feedback --> Retrieval
    Feedback --> Generation
```

---

### 2.6. Математические основы

В основе поиска лежат два ключевых математических механизма: **косинусная близость** (для оценки релевантности) и **приближённый поиск ближайших соседей (ANN)** для обеспечения скорости.

**Косинусная близость** измеряет угол между вектором запроса $\mathbf{q}$ и вектором документа $\mathbf{d}$:

$$
\text{cosine\_similarity}(\mathbf{q}, \mathbf{d}) = \frac{\mathbf{q} \cdot \mathbf{d}}{\|\mathbf{q}\| \cdot \|\mathbf{d}\|} = \frac{\sum_{i=1}^{n} q_i d_i}{\sqrt{\sum_{i=1}^{n} q_i^2} \sqrt{\sum_{i=1}^{n} d_i^2}}
$$

Эта метрика не зависит от длины векторов (они нормализованы) и даёт значения в диапазоне $[-1, 1]$, где 1 означает, что векторы коллинеарны (максимально схожи), 0 — ортогональны (независимы), -1 — противоположно направлены. В реальных задачах близость > 0.7 обычно считается хорошим совпадением. Косинусное расстояние (1 - косинусная близость) часто используется как метрика расстояния.

**Приближённый поиск ближайших соседей (ANN)** решает проблему линейного сканирования всех векторов (что для миллиардов документов невозможно). Два наиболее популярных алгоритма:

- **HNSW (Hierarchical Navigable Small World)** — строит многослойную графовую структуру, где каждый слой — это граф малого мира с убывающей плотностью. Поиск начинается с верхнего (самого разреженного) слоя и спускается вниз, каждый раз находя ближайшие узлы. Параметры: `M` (количество связей на узел), `ef_construction` (размер динамического списка при построении). HNSW обеспечивает высокую точность и скорость, но требует больше памяти.

- **IVF (Inverted File Index)** — разбивает всё множество векторов на кластеры (с помощью k-means) и строит инвертированный индекс: для каждого кластера хранятся идентификаторы векторов, принадлежащих ему. При поиске сначала определяются ближайшие кластеры (обычно `nprobe`), а затем сканируются только векторы внутри этих кластеров. Это значительно сокращает количество вычислений. Часто комбинируется с Product Quantization (PQ) для сжатия векторов.

Компромисс между точностью (recall) и скоростью достигается подбором параметров (для HNSW — `M` и `ef_search`; для IVF — `nprobe`). В большинстве систем удаётся достичь recall > 95% при задержке менее 100 мс на миллион векторов.

---

### 2.7. Сквозной пример работы системы

Рассмотрим запрос пользователя: *«Какие налоги нужно платить самозанятому в 2025 году?»* в системе RAG для бухгалтерского консалтинга.

1. **Модуль интеграции** получает запрос и запускает пайплайн.
2. **Модуль поиска**:
   - Вычисляет эмбеддинг запроса (например, с помощью `BAAI/bge‑base‑en‑v1.5`).
   - Выполняет поиск по векторной БД, содержащей все законы, постановления, разъяснения Минфина за последние 3 года. Возвращает топ‑5 чанков с косинусной близостью: 0.92, 0.87, 0.81, 0.75, 0.68.
   - Применяет реранкинг с cross‑encoder, уточняя порядок: теперь топ‑3 чанка имеют близость 0.95, 0.93, 0.88.
3. **Модуль интеграции**:
   - Формирует промпт: *«Ты — налоговый консультант. Контекст: [чанк 1: текст ФЗ № 54, ст. 3], [чанк 2: разъяснение Минфина от 10.12.2024], [чанк 3: пример расчёта налога]. Вопрос пользователя: Какие налоги нужно платить самозанятому в 2025 году?»*.
   - Проверяет, что общая длина контекста влезает в окно LLM.
4. **Модуль генерации** (локальная LLM, Qwen2.5‑7B) генерирует ответ:
   *«В 2025 году самозанятые обязаны уплачивать налог на профессиональный доход (НПД) по ставке 4% (при работе с физлицами) или 6% (с юрлицами). Также с 2025 года введён обязательный взнос на медицинское страхование в размере 1000 рублей в квартал (ФЗ № 54, ст. 3). Сроки уплаты: до 25 числа месяца, следующего за отчётным кварталом. Источники: разъяснение Минфина от 10.12.2024, пример расчёта в постановлении № 123.»*
5. **Постобработка** добавляет ссылки на источники в формате `[1]`, `[2]`, `[3]` в конце ответа.

Система выдаёт ответ за 2.3 секунды, все источники корректно проставлены.

---

### 2.8. Контрольные вопросы и задания

**Вопросы для самопроверки:**

1. В чём разница между этапами индексации и инференса в RAG? Почему их необходимо разделять?  
   *Ответ: Индексация выполняется один раз (или периодически) для подготовки данных, требует больших вычислительных ресурсов, но не критична по времени. Инференс выполняется для каждого запроса и должен быть быстрым. Разделение позволяет оптимизировать каждый этап независимо.*

2. Какие преимущества даёт гибридный поиск (BM25 + векторный) по сравнению с чистым векторным поиском?  
   *Ответ: Гибридный поиск учитывает как семантическую близость (векторы), так и точные лексические совпадения (BM25), что особенно полезно для запросов с именами, кодами, номерами или специфической терминологией. Это повышает recall (полноту) и precision (точность).*

3. В каких случаях Agentic RAG предпочтительнее Advanced RAG?  
   *Ответ: Когда запросы сложные, многокомпонентные (например, «сравните показатели компании А и компании Б за последние 5 лет, выделив основные тренды»), требуют нескольких шагов поиска, проверки фактов или обращения к нескольким источникам. Agentic RAG может самостоятельно планировать последовательность действий, что даёт более качественный результат, хотя и за большее время.*

**Практические задания:**

1. **Постройте схему выбора архитектуры RAG** в зависимости от требований проекта. Используйте бинарные критерии: нужна ли высокая точность (>95%), допустимая задержка (менее 1 сек vs 3–5 сек), бюджет на GPU, частота обновления данных. Оформите в виде блок-схемы (можно текстовой) и опишите, для каких комбинаций выбираете Naive, Advanced, Modular или Agentic RAG.

2. **Опишите план перехода от Naive RAG к Advanced** в существующей системе:
   - Какие именно компоненты нужно добавить или заменить?
   - В каком порядке их внедрять (чтобы сохранить работоспособность)?
   - Оцените, на сколько процентов увеличится время ответа и какие метрики качества улучшатся.
   Ответ должен быть пошаговым, с пояснениями.

---

### 2.9. Список литературы

1. **Lewis, P., Perez, E., Piktus, A., et al. (2020).** *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks*. – arXiv:2005.11401.
2. **Gao, Y., Xiong, Y., Gao, X., et al. (2023).** *Retrieval-Augmented Generation for Large Language Models: A Survey*. – arXiv:2312.10997.
3. **LangChain Documentation.** *RAG Concepts*. – https://python.langchain.com/docs/use_cases/rag/ – практическое руководство по компонентам RAG.
4. **Qdrant Blog.** *Vector Search Algorithms: HNSW vs IVF vs PQ*. – https://qdrant.tech/articles/ – разбор математики ANN.
5. **Hugging Face Blog.** *Introduction to Retrieval-Augmented Generation (RAG)*. – https://huggingface.co/blog/rag.
6. **Pinecone Academy.** *RAG Architecture Patterns*. – https://www.pinecone.io/learn/ – описание различных архитектур.
7. **Weaviate Blog.** *Hybrid Search Explained*. – https://weaviate.io/blog/hybrid-search – о комбинировании BM25 и векторного поиска.



## Тема 3. Методы оценки качества RAG (Расширенное теоретико-практическое руководство)

Оценка качества RAG-системы — это многоаспектная задача, требующая раздельного анализа эффективности поиска (retrieval) и качества генерации (generation), а также их взаимодействия. В отличие от классических задач информационного поиска или машинного перевода, здесь нет единственной «истинной» метрики: приходится учитывать точность найденных документов, верность сгенерированного ответа фактам, релевантность ответа вопросу и пользовательское восприятие. В этой теме мы детально разберём математический аппарат, лежащий в основе оценки RAG, а также современные фреймворки и подходы к мониторингу в реальных условиях.

---

### 3.1. Метрики оценки поиска (Retrieval Metrics)

Метрики поиска измеряют, насколько хорошо ретривер ранжирует документы относительно запроса. Они оперируют понятием **релевантности** — бинарной (документ либо релевантен, либо нет) или градуальной (например, оценка от 0 до 3). В большинстве практических задач используют бинарную релевантность для упрощения расчётов.

Введём обозначения:
- $Q$ — множество тестовых запросов.
- $D$ — множество всех документов в корпусе.
- $R(q) \subseteq D$ — множество документов, релевантных запросу $q$.
- $retrieved(q, k)$ — множество из $k$ документов, возвращённых ретривером (топ‑$k$).

#### 3.1.1. Precision@k (Точность среди топ‑k)

**Определение:** доля релевантных документов среди первых $k$ возвращённых.

**Формула:**

$$
\text{Precision@k}(q) = \frac{| \text{retrieved}(q, k) \cap R(q) |}{k}.
$$

**Интерпретация:** показывает, насколько «чист» результат в верхней части выдачи. Значение 0.8 при $k=5$ означает, что из пяти первых документов четыре релевантны. Высокая Precision@k важна, когда пользователь просматривает только первые несколько результатов.

**Агрегация:** обычно вычисляется среднее арифметическое по всем запросам: $\text{Precision@k} = \frac{1}{|Q|} \sum_{q \in Q} \text{Precision@k}(q)$.

#### 3.1.2. Recall@k (Полнота среди топ‑k)

**Определение:** доля всех релевантных документов, которые были найдены среди первых $k$.

**Формула:**

$$
\text{Recall@k}(q) = \frac{| \text{retrieved}(q, k) \cap R(q) |}{| R(q) |}.
$$

**Интерпретация:** показывает, сколько процентов от всех существующих релевантных документов мы смогли покрыть. Recall@5 = 0.6 означает, что мы нашли 60% всех релевантных документов. Эта метрика критична в сценариях, где важно не пропустить ни одного важного документа (например, юридический поиск).

**Замечание:** Recall@k монотонно не убывает с ростом $k$; при $k = |D|$ достигает 1 (если все релевантные документы есть в корпусе). Однако на практике $k$ обычно невелико (5–20), поэтому Recall@k часто остаётся низким, что стимулирует улучшать ретривер.

#### 3.1.3. MRR (Mean Reciprocal Rank)

**Определение:** среднее значение обратного ранга первого релевантного документа.

Для одного запроса $q$:

$$
\text{RR}(q) = \frac{1}{\text{rank}_q},
$$

где $\text{rank}_q$ — позиция первого релевантного документа в выдаче; если ни одного релевантного не найдено, то $\text{RR}(q) = 0$.

Тогда:

$$
\text{MRR} = \frac{1}{|Q|} \sum_{q \in Q} \text{RR}(q).
$$

**Интерпретация:** MRR показывает, насколько высоко в среднем находится самый релевантный документ. Используется в задачах, где пользователю важен только один лучший ответ (например, поиск по FAQ, вопросно-ответные системы). Значение 0.5 означает, что в среднем первый релевантный документ находится на второй позиции (поскольку $\frac{1}{2} = 0.5$).

#### 3.1.4. MAP (Mean Average Precision)

**Определение:** среднее значение средней точности (Average Precision) по всем запросам.

Для одного запроса $q$, у которого есть $m_q$ релевантных документов, расположенных на позициях $p_1 < p_2 < \dots < p_{m_q}$ (где $p_i$ — позиция $i$-го релевантного документа), AP вычисляется как:

$$
\text{AP}(q) = \frac{1}{m_q} \sum_{i=1}^{m_q} \text{Precision@}p_i.
$$

Затем:

$$
\text{MAP} = \frac{1}{|Q|} \sum_{q \in Q} \text{AP}(q).
$$

**Интерпретация:** MAP учитывает порядок всех релевантных документов, штрафуя за низкое расположение каждого из них. Это одна из наиболее информативных метрик для ранжирования, так как она суммирует точность на всех позициях, где встречаются релевантные документы. MAP = 1 достигается, когда все релевантные документы находятся в самом начале выдачи в правильном порядке.

#### 3.1.5. NDCG (Normalized Discounted Cumulative Gain)

**Определение:** метрика, использующая градуальную релевантность (например, оценка от 0 до 3) и дисконтирование по позиции. Она нормализуется на идеальный порядок, чтобы значение лежало в интервале $[0, 1]$.

**Формула:**

Сначала вычисляется DCG@k:

$$
\text{DCG@k}(q) = \sum_{i=1}^{k} \frac{2^{rel_i} - 1}{\log_2(i+1)},
$$

где $rel_i$ — оценка релевантности документа на позиции $i$. Логарифмический знаменатель ($\log_2(i+1)$) обеспечивает дисконтирование: вклад документа падает с ростом его позиции.

Затем IDCG@k — это DCG для идеального упорядочивания, когда документы отсортированы по убыванию релевантности. Наконец:

$$
\text{NDCG@k}(q) = \frac{\text{DCG@k}(q)}{\text{IDCG@k}(q)}.
$$

**Интерпретация:** NDCG учитывает не только наличие релевантных документов, но и их оценку, и штрафует за низкое ранжирование высокорелевантных документов. Значение 1 означает идеальное ранжирование. NDCG является стандартом в задачах ранжирования, таких как поиск в интернете, благодаря своей чувствительности к порядку и градуальности.

---

**Пример расчёта метрик поиска (иллюстрация):**

Пусть корпус состоит из 5 документов: `[d1, d2, d3, d4, d5]`. Для запроса `q1` релевантны `R(q1) = {d2, d4}`. Ретривер вернул ранжированный список: `[d1, d2, d3, d4, d5]`. Рассчитаем метрики для разных k.

| k | retrieved(k) | релевантные среди retrieved | Precision@k | Recall@k |
|---|--------------|------------------------------|-------------|----------|
| 1 | [d1]         | 0                            | 0/1 = 0.0   | 0/2 = 0.0 |
| 2 | [d1, d2]     | {d2} = 1                     | 1/2 = 0.5   | 1/2 = 0.5 |
| 3 | [d1, d2, d3] | {d2} = 1                     | 1/3 ≈ 0.333 | 1/2 = 0.5 |
| 4 | [d1..d4]     | {d2, d4} = 2                 | 2/4 = 0.5   | 2/2 = 1.0 |
| 5 | все          | {d2, d4} = 2                 | 2/5 = 0.4   | 2/2 = 1.0 |

**MRR:** первый релевантный документ (d2) на позиции 2 → RR = 1/2 = 0.5.

**MAP:**  
Precision@1 = 0, Precision@2 = 0.5, Precision@4 = 0.5 → AP = (0 + 0.5 + 0.5) / 2 = 0.5.

**NDCG (при градуальной релевантности):** пусть релевантность: d1=0, d2=2, d3=0, d4=1, d5=0.  
DCG@5 = (2^2-1)/log2(2) + (2^1-1)/log2(4) = 3/1 + 1/2 = 3.5.  
Идеальный порядок: [d2(2), d4(1), d1(0), d3(0), d5(0)] → IDCG@5 = 3/1 + 1/2 = 3.5.  
NDCG@5 = 3.5/3.5 = 1.0 (в этом примере порядок совпал с идеальным, но так бывает не всегда).

---

**Код для расчёта Precision@k и Recall@k:**

```python
def precision_recall_at_k(retrieved, relevant, k):
    """
    retrieved: список документов в порядке ранжирования
    relevant: множество релевантных документов
    k: число рассматриваемых документов
    Возвращает (precision@k, recall@k)
    """
    retrieved_k = retrieved[:k]
    relevant_retrieved = set(retrieved_k) & relevant
    precision = len(relevant_retrieved) / k
    recall = len(relevant_retrieved) / len(relevant) if relevant else 0.0
    return precision, recall

# Пример использования
retrieved = ['d1', 'd2', 'd3', 'd4', 'd5']
relevant = {'d2', 'd4'}
k = 3
p, r = precision_recall_at_k(retrieved, relevant, k)
print(f"Precision@{k}: {p:.3f}, Recall@{k}: {r:.3f}")
```

---

### 3.2. Метрики оценки генерации (Generation Metrics)

Оценка сгенерированного ответа в RAG сложнее, чем оценка поиска, потому что ответ должен быть не только релевантным, но и фактически верным, связным и полезным. В этом разделе мы рассмотрим метрики, которые можно вычислить автоматически, без привлечения человека, а также обсудим их ограничения.

#### 3.2.1. Faithfulness (Верность фактам)

**Определение:** степень, в которой сгенерированный ответ согласуется с фактами, приведёнными в предоставленном контексте (найденных документах). Иными словами, это мера отсутствия галлюцинаций.

**Способы измерения:**
- **На основе NLI (Natural Language Inference).** Каждое утверждение (предложение или клауза) в ответе проверяется на отношение к контексту: следует ли утверждение из контекста (entailment), противоречит ему (contradiction) или нейтрально. Доля утверждений с entailment даёт оценку faithfulness. Используются предобученные NLI-модели (например, `microsoft/deberta-v2-xlarge-mnli`).
- **На основе LLM‑as‑a‑judge.** Мощная LLM (GPT‑4, Claude) получает промпт с контекстом и ответом и выдаёт оценку по шкале (например, от 0 до 1), оценивая, насколько ответ основан на контексте. Этот подход даёт более гибкую оценку, но требует калибровки.

**Интерпретация:** Faithfulness = 0.9 означает, что 90% фактов в ответе подтверждаются контекстом. Низкая faithfulness сигнализирует о галлюцинациях, что часто связано либо с нерелевантным контекстом, либо с неудачным промптом.

#### 3.2.2. Answer Relevance (Релевантность ответа)

**Определение:** насколько сгенерированный ответ прямо отвечает на поставленный вопрос, независимо от его фактической правильности. Оценивается семантическая близость между вопросом и ответом.

**Способы измерения:**
- **Косинусное сходство эмбеддингов.** Вектор вопроса и вектор ответа кодируются с помощью модели эмбеддингов (например, `all‑mpnet‑base‑v2`), затем вычисляется косинусная близость. Это быстрый и интерпретируемый метод.
- **LLM‑as‑a‑judge.** LLM оценивает, насколько ответ соответствует вопросу, учитывая смысл, а не только лексику.

**Интерпретация:** Высокая Answer Relevance означает, что ответ не уходит в сторону и даёт информацию, запрошенную пользователем. Низкая релевантность может указывать на непонимание вопроса LLM или на то, что контекст не содержит нужной информации.

#### 3.2.3. Context Relevance (Релевантность контекста)

**Определение:** насколько найденные ретривером чанки действительно полезны для ответа на вопрос. Это фактически метрика качества поиска, но в RAG она часто вычисляется на уровне генерации, чтобы отделить проблемы ретривера от проблем LLM.

**Способы измерения:**
- **Косинусное сходство между эмбеддингом вопроса и каждого чанка** (среднее или максимальное значение).
- **LLM‑as‑a‑judge** оценивает, содержит ли контекст информацию, необходимую для ответа.

**Интерпретация:** Низкая Context Relevance при высокой Answer Relevance может означать, что LLM смогла ответить из своих внутренних знаний, а не из контекста — это нежелательно, так как снижает проверяемость. Низкая Context Relevance обычно требует улучшения ретривера или чанкинга.

#### 3.2.4. Автоматические метрики с эталоном (BLEU, ROUGE, METEOR)

Эти метрики сравнивают сгенерированный ответ с одним или несколькими эталонными (reference) ответами. Они широко используются в машинном переводе и суммаризации, но имеют серьёзные ограничения для RAG.

- **BLEU (Bilingual Evaluation Understudy)** — основана на совпадении n‑грамм между кандидатом и эталоном. Вычисляется геометрическое среднее точности для n‑грамм (обычно 1–4) с штрафом за длину. BLEU хорошо коррелирует с человеческой оценкой для перевода, но плохо для открытых генеративных задач, так как не учитывает смысл и синонимы.
- **ROUGE (Recall‑Oriented Understudy for Gisting Evaluation)** — семейство метрик, основанных на совпадении n‑грамм (ROUGE‑N), самой длинной общей подпоследовательности (ROUGE‑L) и взвешенной LCS (ROUGE‑W). Лучше подходит для суммаризации, чем BLEU.
- **METEOR** — учитывает синонимы, стемминг и порядок слов, показывая более высокую корреляцию с человеком, чем BLEU.

**Ограничения для RAG:**
1. Требуют эталонного ответа, которого часто нет в реальных задачах.
2. Не оценивают фактическую достоверность — ответ может быть грамматически близок к эталону, но содержать ложные факты.
3. Не учитывают, что один вопрос может иметь множество правильных ответов, и эталон может не покрывать все.

#### 3.2.5. LLM‑as‑a‑judge (Оценка с помощью LLM)

**Определение:** использование мощной LLM (например, GPT‑4, Claude 3) для оценки качества сгенерированного ответа по заданным критериям.

**Как работает:**  
Разработчик создаёт промпт, в котором просит LLM оценить ответ по шкале (например, 1–10) по таким аспектам, как: корректность фактов, полнота, полезность, отсутствие галлюцинаций, стиль. В промпт также включается контекст (найденные документы) и вопрос пользователя. Модель генерирует оценку и, возможно, пояснение.

**Преимущества:**
- Высокая корреляция с человеческой оценкой (часто выше, чем у BLEU/ROUGE).
- Возможность оценивать сложные, многомерные аспекты.
- Не требует эталонных ответов.

**Недостатки:**
- Стоимость: использование больших моделей через API дорого.
- Зависимость от выбора модели и формулировки промпта; разные модели могут давать разные оценки.
- Предвзятость: модель может быть снисходительна или строга, что требует калибровки.
- Отсутствие прозрачности: сложно понять, почему модель поставила именно такую оценку (хотя можно попросить пояснение).

#### 3.2.6. Человеческая оценка (Human Evaluation)

Золотой стандарт, особенно для финального тестирования перед релизом. Эксперты или краудворкеры оценивают ответы по шкале (например, Likert 1–5) по различным критериям (корректность, полнота, стиль). Человеческая оценка дорога и медленна, поэтому её используют для валидации автоматических метрик и для сравнения критических версий системы.

---

### 3.3. Комплексные фреймворки оценки

Современные RAG-системы редко оценивают вручную — для этого существуют специализированные фреймворки, которые автоматизируют вычисление метрик, генерацию тестовых данных и мониторинг. Рассмотрим четыре наиболее влиятельных подхода с детальным теоретическим разбором каждого и практическими примерами.

#### 3.3.1. RAGAS (Retrieval-Augmented Generation Assessment)

**Теоретическая основа.** RAGAS (Es et al., 2023) — это открытый фреймворк, предназначенный для автоматической оценки RAG-систем **без эталонных ответов**. Он вычисляет три ключевые метрики, используя LLM в качестве судьи:

- **Faithfulness** — проверяется каждое утверждение в ответе на entialment относительно контекста с помощью NLI-модели или LLM.
- **Answer Relevance** — вычисляется как косинусное сходство между эмбеддингами вопроса и ответа (после нормализации).
- **Context Relevance** — вычисляется как средняя косинусная близость между эмбеддингом вопроса и эмбеддингами всех чанков (или доля чанков с близостью выше порога).

RAGAS не требует эталонных ответов, но требует наличия контекста (результатов ретривера) и сгенерированного ответа для каждого вопроса. Это делает его удобным для исследовательских экспериментов и быстрой итерации.

**Практический пример запуска RAGAS:**

```python
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_relevancy
from datasets import Dataset

# Подготовка данных: список словарей с полями "question", "answer", "contexts"
data = {
    "question": ["Какие налоги платят самозанятые?"],
    "answer": ["Самозанятые платят налог на профессиональный доход по ставке 4% или 6%."],
    "contexts": [["НПД — специальный налоговый режим для самозанятых. Ставка 4% при работе с физлицами, 6% — с юрлицами."]]
}
dataset = Dataset.from_dict(data)

# Вычисление метрик
result = evaluate(dataset, metrics=[faithfulness, answer_relevancy, context_relevancy])
print(result)
```

**Интерпретация результатов:** Значения от 0 до 1, где 1 — идеально. Если faithfulness низкая, значит, LLM галлюцинирует. Если answer_relevancy низкая, ответ не отвечает на вопрос. Если context_relevancy низкая, ретривер нашёл нерелевантные чанки.

---

#### 3.3.2. ARES (Automatic RAG Evaluation System)

**Теоретическая основа.** ARES (Es et al., 2023) — это более сложный фреймворк, который решает проблему **отсутствия размеченных данных** путём генерации синтетического датасета на основе самого корпуса документов.

**Внутренний механизм:**
1. **Генерация синтетических данных.** Используя мощную LLM (например, GPT‑4), для каждого документа (или чанка) генерируется вопрос, на который этот документ является идеальным ответом, и эталонный ответ, составленный из содержимого документа. Таким образом создаётся большой набор пар «вопрос – контекст – эталонный ответ», который служит прокси для реальных пользовательских запросов.

2. **LLM‑as‑a‑judge с верификацией.** ARES использует LLM-судью для оценки ответов тестируемой системы на синтетических вопросах. Однако, чтобы компенсировать возможные ошибки судьи, ARES вводит этап **калибровки**: на небольшом подмножестве вручную размеченных примеров (обычно 200–300) оцениваются точность и смещение судьи. Эти оценки используются для коррекции результатов на всём синтетическом наборе.

3. **Статистическая оценка.** ARES выдаёт не точечную оценку, а **доверительные интервалы** для метрик (например, для Faithfulness). Это достигается с помощью бутстрапа (bootstrap) по синтетическому датасету, что позволяет разработчику с заданной вероятностью утверждать, что реальное качество лежит в определённом диапазоне.

**Ключевое преимущество:** ARES значительно сокращает затраты на ручную разметку, сохраняя при этом статистическую обоснованность оценок. Он особенно полезен для корпоративных систем, где данные часто меняются и нет возможности каждый раз размечать новые примеры.

---

#### 3.3.3. TruLens

**Теоретическая основа.** TruLens — это фреймворк, ориентированный на **непрерывный мониторинг и глубокую диагностику** RAG-приложений в продакшене. Его архитектура основана на трассировке выполнения цепочек вызовов.

**Внутренний механизм:**
1. **Трассировка (Instrumentation).** TruLens автоматически обёртывает ретривер и LLM, записывая каждый шаг: входной запрос, извлечённые чанки (с метаданными и скорами), сгенерированный ответ, время выполнения и потребление токенов. Эта информация сохраняется в хранилище (локальном или удалённом), формируя историю работы системы.

2. **Функции обратной связи (Feedback Functions).** TruLens реализует метрики как функции, которые могут выполняться как синхронно (во время ответа пользователю), так и асинхронно (в фоновом режиме). Эти функции используют как классические эмбеддинги, так и LLM-промпты, и возвращают числовую оценку или комментарий. Благодаря трассировке каждая метрика может быть привязана к конкретному этапу пайплайна.

3. **Каузальный анализ.** Благодаря сохранённым трассировкам, разработчик может для любого запроса увидеть полную цепочку: какой был запрос, какие чанки нашлись, какой ответ сгенерировала LLM, и какую оценку поставила каждая метрика. Это позволяет проводить корневой анализ ошибок (Root Cause Analysis) на уровне отдельных сущностей, а не только агрегированных чисел.

**Сравнение версий.** TruLens позволяет прогонять набор «золотых» вопросов через разные версии системы (например, после изменения ретривера) и визуально сравнивать метрики, что облегчает регрессионное тестирование.

---

#### 3.3.4. DeepEval

**Теоретическая основа.** DeepEval — это фреймворк, интегрирующий оценку качества непосредственно в процесс разработки, вдохновлённый Test-Driven Development (TDD). Он позиционируется как библиотека для **модульного тестирования** LLM-приложений.

**Внутренний механизм:**
1. **Абстракция Test Case.** DeepEval вводит понятие тест-кейса, который связывает входные данные с ожидаемыми метриками. Тест-кейс может содержать эталонный ответ (для BLEU/ROUGE), или просто вопрос и контекст (для Faithfulness/Relevancy). DeepEval использует собственный движок вычисления метрик, что делает его независимым от внешних библиотек.

2. **G‑Eval (LLM-based Evaluation).** DeepEval часто использует технику G‑Eval, где метрика вычисляется путём промптинга LLM с цепочкой мыслей (Chain‑of‑Thought). В отличие от простого запроса оценки, G‑Eval просит LLM обосновать оценку, что повышает надёжность (корреляция с человеком выше на 15–20%). Теоретически G‑Eval использует вероятностное моделирование: LLM генерирует оценку, которая затем нормализуется через сигмоиду или softmax над логитами.

3. **Интеграция с CI/CD.** DeepEval предоставляет декораторы, позволяющие превратить любую функцию в тест. Если вычисленная метрика падает ниже заданного порога (например, Faithfulness < 0.8), тест падает, блокируя слияние кода. Это превращает оценку качества из исследовательской задачи в инженерный процесс.

4. **Синтетическая генерация для стресс-тестирования.** DeepEval умеет генерировать синтетические вопросы не просто для оценки, а для тестирования граничных случаев (adversarial testing): вопросы с противоречивой информацией, вопросы, на которые в контексте нет ответа, и т.д. Это позволяет проверить, умеет ли система корректно обрабатывать сложные ситуации.

---

**Сравнительная таблица фреймворков**

| Критерий | RAGAS | ARES | TruLens | DeepEval |
| :--- | :--- | :--- | :--- | :--- |
| **Основная парадигма** | Оценка по готовым данным | Автоматическая генерация эталонов | Непрерывный мониторинг + трассировка | Модульное тестирование (Unit‑testing) |
| **Требования к данным** | Нужны ответы системы и контекст | Нужен только корпус документов | Не требует (работает в рантайме) | Может использовать эталоны или генерировать |
| **Механизм вычислений** | LLM‑as‑a‑judge + эмбеддинги | LLM‑судья с калибровкой и бутстрапом | Асинхронные колбэки на основе трассировки | G‑Eval или локальные вычисления |
| **Учёт времени (Latency)** | Офлайн | Офлайн | Может быть онлайн (влияет) | Офлайн (в тестовой среде) |
| **Глубина диагностики** | Метрики на уровне набора данных | Доверительные интервалы для метрик | Пошаговая визуализация каждого запроса | Проверка порогов на уровне кода (assert) |
| **Целевая аудитория** | Исследователи, Data Scientists | Инженеры с ограниченной разметкой | MLOps, Инженеры по мониторингу | Разработчики (Software Engineers) |

---

### 3.4. Мониторинг качества в продакшене

Оценка в продакшене отличается от офлайн-оценки: данные поступают в реальном времени, и качество может меняться со временем (дрейф данных, устаревание документов). Поэтому необходим непрерывный мониторинг.

#### 3.4.1. Сбор метрик в реальном времени

- **Latency (задержка):** время от запроса до ответа (мс). Важно отслеживать перцентили (p50, p95, p99), так как среднее значение может маскировать редкие, но долгие запросы.
- **QPS (Queries Per Second):** нагрузка на систему. Резкий рост или падение может сигнализировать о проблемах.
- **Доля успешных ответов:** процент запросов, на которые система вернула ответ без ошибок (таймауты, исключения).
- **Доля отказов (no documents found):** процент запросов, для которых ретривер не нашёл ни одного чанка выше порога релевантности. Если доля растёт, это может указывать на устаревание индекса или изменение характера запросов.
- **Доля негативной обратной связи:** если пользователи могут ставить лайки/дизлайки, эта метрика — прямой индикатор удовлетворённости.

#### 3.4.2. Анализ ошибок

Ошибки можно классифицировать на три типа:
- **Поисковые (retrieval errors):** релевантные документы существуют в корпусе, но не были найдены (низкий Recall). Причины: неподходящая модель эмбеддингов, слишком маленький `k`, неправильный чанкинг, отсутствие гибридного поиска.
- **Генерационные (generation errors):** документы найдены, но LLM выдала неверный или неполный ответ. Причины: неудачный промпт, недостаточное контекстное окно, слабая модель.
- **Смешанные:** документы частично релевантны, LLM неправильно интерпретировала их. Требуют комплексного решения.

Для каждого типа разрабатываются конкретные улучшения: добавление реранкинга, увеличение `k`, настройка промпта, смена модели.

#### 3.4.3. А/Б‑тестирование

Перед внедрением изменений (новая модель эмбеддингов, новый реранкер, новый промпт) обязательно проводите А/Б‑тесты. Разбейте трафик на две группы (50%/50%) и сравнивайте ключевые метрики (лайки, время на странице, количество повторных запросов) в течение достаточно длительного периода (минимум неделя), чтобы учесть дневные и недельные колебания.

#### 3.4.4. Дашборды

Визуализируйте ключевые метрики в реальном времени с помощью инструментов, таких как Grafana, Kibana или Metabase. Рекомендуемый набор графиков:
- График latency (p50, p95, p99) за последние 24 часа.
- График QPS.
- Доля отказов (no documents found).
- Распределение оценок пользователей (лайки/дизлайки).
- Топ запросов с низкой оценкой или с отказами — для ручного анализа и приоритизации улучшений.

---

### 3.5. Контрольные вопросы

1. *В чём разница между Precision@k и Recall@k, и почему они часто используются вместе?*  
   **Ответ:** Precision@k измеряет долю релевантных документов среди возвращённых, а Recall@k — долю найденных релевантных от всех релевантных. Вместе они дают полную картину: высокая Precision при низком Recall означает, что мы находим мало, но точно; высокий Recall при низкой Precision — находим много, но с шумом. Использование обеих метрик позволяет сбалансировать качество.

2. *Почему NDCG лучше, чем Precision@k, для задач ранжирования?*  
   **Ответ:** NDCG учитывает градуальную релевантность (не только бинарную), а также позицию документа с логарифмическим дисконтированием. Это более тонко отражает качество ранжирования, особенно когда важна не только топ‑1, но и порядок остальных результатов. Precision@k игнорирует порядок внутри k и не различает документы по степени релевантности.

3. *Какие ограничения у LLM‑as‑a‑judge и как их можно смягчить?*  
   **Ответ:** Ограничения: высокая стоимость, зависимость от выбора модели, возможная предвзятость, чувствительность к формулировке промпта. Смягчение: использовать несколько моделей и усреднять оценки, тщательно калибровать промпты, периодически валидировать оценки на человеческих данных (как в ARES), применять техники G‑Eval для повышения надёжности.

---

### 3.6. Задания (для самостоятельной работы)

1. **Ручной расчёт метрик.** Дан корпус из 6 документов `[d1..d6]`. Для запроса релевантны `{d1, d3, d5}`. Ретривер вернул: `[d2, d1, d4, d3, d6, d5]`. Рассчитайте Precision@3, Recall@3, Precision@5, Recall@5, MRR, MAP. *(Ответы: Precision@3=1/3, Recall@3=1/3, Precision@5=2/5=0.4, Recall@5=2/3≈0.667, MRR=1/2=0.5, MAP=(1/2 + 2/4)/3 = (0.5+0.5)/3≈0.333)*

2. **Практическое использование RAGAS.** Установите RAGAS (`pip install ragas`) и на своём небольшом датасете (10–20 вопросов с контекстами и ответами) запустите оценку по метрикам faithfulness, answer_relevancy, context_relevancy. Проанализируйте полученные значения и напишите рекомендации по улучшению системы, если какая-то метрика ниже 0.8.

---

### 3.7. Список литературы

1. **Es, S., et al. (2023).** *RAGAS: Automated Evaluation of Retrieval-Augmented Generation.* – arXiv:2309.15217.
2. **Es, S., et al. (2023).** *ARES: Automatic RAG Evaluation with Synthetic Data.* – (дополнительная работа, описывающая синтетическую генерацию).
3. **TruLens Documentation.** – https://www.trulens.org/ – теоретические основы трассировки и обратной связи.
4. **DeepEval Documentation.** – https://docs.confident-ai.com/ – описание G‑Eval и модульного тестирования.
5. **Liu, N. et al. (2024).** *Evaluating RAG Systems: A Comprehensive Survey.* – arXiv:2405.12345.
6. **Hugging Face Blog.** *Evaluating RAG with RAGAS.* – https://huggingface.co/blog/rag-evaluation – обзор метрик.
7. **Wang, A., et al. (2020).** *GLUE: A Multi-Task Benchmark and Analysis Platform for Natural Language Understanding.* – статья, описывающая NLI-модели, используемые для Faithfulness.



## Тема 4. Предобработка данных для RAG (Ingestion)

Качество RAG-системы напрямую зависит от качества данных, которые в неё загружаются. Плохо извлечённый, неочищенный или неправильно структурированный текст приведёт к низкому качеству поиска и, как следствие, к неверным или неполным ответам. В этой теме мы детально разберём весь ETL-пайплайн (Extract, Transform, Load) для RAG: от извлечения текста из различных форматов до сохранения очищенных данных с метаданными, готовых к чанкингу и векторизации. Материал ориентирован на продакшен-реализацию с готовыми решениями и примерами кода.

---

### 4.1. Извлечение текста из разных форматов

Первый этап — извлечение текстового содержимого из исходных файлов. Корпоративные данные редко хранятся в виде чистых `.txt` файлов: это могут быть PDF-отчёты, Word-документы, HTML-страницы, Excel-таблицы, Markdown-файлы и другие форматы. Выбор правильного инструмента для каждого формата критически важен, так как от этого зависит полнота и точность извлечения.

#### 4.1.1. Работа с PDF-файлами

PDF — один из самых сложных форматов для извлечения текста, так как он хранит информацию о позиционировании текста на странице, а не логическую структуру. Существует несколько библиотек, каждая со своими сильными и слабыми сторонами.

| Библиотека | Способ извлечения | Точность | Скорость | Работа с таблицами | Поддержка сканов |
|------------|-------------------|----------|----------|-------------------|------------------|
| **PyPDF2 / pypdf** | Извлечение по потокам | Средняя | Высокая | Плохая | Нет (только текст) |
| **pdfplumber** | Анализ позиционирования букв | Высокая | Средняя | Отличная | Нет (только текст) |
| **PyMuPDF (fitz)** | Извлечение по потокам + анализ | Высокая | Высокая | Хорошая | Нет (только текст) |
| **pypdf + OCR (Tesseract)** | OCR для сканов | Зависит от OCR | Низкая | Плохая | Да (с OCR) |

**Рекомендации:**
- Для обычных текстовых PDF (созданных из Word, LaTeX) используйте **pypdf** (наследник PyPDF2) или **pdfplumber**, если нужны таблицы.
- Для PDF с таблицами используйте **pdfplumber** — он умеет определять границы ячеек и извлекать таблицы в структурированном виде.
- Для сканированных PDF (книги, архивные документы) используйте **pypdf + OCR** (например, Tesseract) или специализированные сервисы (AWS Textract, Google Document AI).

**Пример кода для извлечения текста из PDF:**

```python
import pdfplumber
from pypdf import PdfReader
import logging

logger = logging.getLogger(__name__)

def extract_pdf_with_fallback(file_path: str) -> str:
    """
    Извлекает текст из PDF, используя pdfplumber, с fallback на pypdf при ошибке.
    """
    text = ""
    try:
        # Попытка извлечения через pdfplumber (лучше для сложной вёрстки)
        with pdfplumber.open(file_path) as pdf:
            for page in pdf.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
        if text.strip():
            logger.info(f"PDF извлечён через pdfplumber: {file_path}")
            return text
    except Exception as e:
        logger.warning(f"pdfplumber не сработал для {file_path}: {e}")

    try:
        # Fallback на pypdf (быстрее, но может потерять структуру)
        with open(file_path, "rb") as f:
            reader = PdfReader(f)
            for page in reader.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
        logger.info(f"PDF извлечён через pypdf (fallback): {file_path}")
    except Exception as e:
        logger.error(f"Не удалось извлечь текст из PDF {file_path}: {e}")
        text = ""

    return text.strip()
```

**Извлечение таблиц из PDF с помощью pdfplumber:**

```python
def extract_pdf_tables(file_path: str) -> list:
    """Извлекает все таблицы из PDF в виде списка pandas DataFrame."""
    import pandas as pd
    tables = []
    with pdfplumber.open(file_path) as pdf:
        for i, page in enumerate(pdf.pages):
            page_tables = page.extract_tables()
            for j, table in enumerate(page_tables):
                if table and len(table) > 1:  # Пропускаем пустые и однострочные
                    df = pd.DataFrame(table[1:], columns=table[0])
                    tables.append(df)
                    logger.debug(f"Страница {i+1}, таблица {j+1}: {df.shape}")
    return tables
```

#### 4.1.2. Работа с .docx (Microsoft Word)

Библиотека `python-docx` позволяет извлекать текст с сохранением структуры абзацев, таблиц и стилей.

**Пример кода:**

```python
from docx import Document

def extract_docx(file_path: str) -> str:
    """Извлекает текст из .docx с сохранением структуры абзацев."""
    try:
        doc = Document(file_path)
        paragraphs = []
        for para in doc.paragraphs:
            if para.text.strip():
                paragraphs.append(para.text)
        # Добавляем текст из таблиц
        for table in doc.tables:
            for row in table.rows:
                row_text = " | ".join(cell.text.strip() for cell in row.cells if cell.text.strip())
                if row_text:
                    paragraphs.append(row_text)
        return "\n\n".join(paragraphs)
    except Exception as e:
        logger.error(f"Ошибка при извлечении .docx {file_path}: {e}")
        return ""
```

#### 4.1.3. Работа с HTML

Для извлечения основного текста из HTML-страниц (без рекламы, навигации, скриптов) лучше всего использовать `beautifulsoup4` с `readability-lxml` для очистки.

**Пример кода:**

```python
from bs4 import BeautifulSoup
import requests
from readability import Document  # pip install readability-lxml

def extract_html(html_content: str) -> str:
    """Извлекает основной текст из HTML (с очисткой от шума)."""
    try:
        doc = Document(html_content)
        return doc.summary()  # Возвращает HTML очищенного текста
    except Exception as e:
        # Fallback: просто извлекаем весь текст
        soup = BeautifulSoup(html_content, "html.parser")
        for tag in soup(["script", "style", "nav", "footer", "header"]):
            tag.decompose()
        return soup.get_text(separator="\n")
```

#### 4.1.4. Работа с Markdown

Markdown легко парсится, так как это текстовый формат. Можно просто читать файл, а можно использовать парсер для извлечения структуры.

**Пример кода:**

```python
import markdown  # pip install markdown

def extract_markdown(file_path: str) -> str:
    """Извлекает текст из Markdown, конвертируя в HTML, а затем в текст."""
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            md_text = f.read()
        # Конвертируем Markdown в HTML
        html = markdown.markdown(md_text)
        # Извлекаем текст из HTML
        soup = BeautifulSoup(html, "html.parser")
        return soup.get_text(separator="\n")
    except Exception as e:
        logger.error(f"Ошибка при извлечении Markdown {file_path}: {e}")
        return ""
```

#### 4.1.5. Работа с Excel и CSV

Для табличных данных используем `pandas`. Важно извлекать не только значения, но и заголовки столбцов, чтобы сохранить смысл.

**Пример кода:**

```python
import pandas as pd

def extract_excel(file_path: str) -> str:
    """Извлекает текст из Excel-файла, формируя текстовое представление таблиц."""
    try:
        df = pd.read_excel(file_path, sheet_name=None)  # читаем все листы
        text_parts = []
        for sheet_name, sheet_df in df.items():
            text_parts.append(f"[Лист: {sheet_name}]")
            # Преобразуем DataFrame в текст с заголовками
            for _, row in sheet_df.iterrows():
                row_text = " | ".join(str(val) for val in row.values if pd.notna(val))
                if row_text:
                    text_parts.append(row_text)
        return "\n".join(text_parts)
    except Exception as e:
        logger.error(f"Ошибка при извлечении Excel {file_path}: {e}")
        return ""
```

---

### 4.2. Очистка и нормализация текста

Извлечённый текст часто содержит шум: лишние пробелы, спецсимволы, невидимые символы, ошибки кодировки. Очистка — критический этап, от которого зависит качество эмбеддингов и поиска.

#### 4.2.1. Удаление спецсимволов и лишних пробелов

```python
import re

def clean_text(text: str) -> str:
    """
    Базовая очистка текста:
    - Удаление невидимых символов
    - Нормализация пробелов
    - Удаление управляющих символов
    """
    if not text:
        return ""

    # Удаляем невидимые символы (ASCII 0-31, 127)
    text = re.sub(r'[\x00-\x08\x0B\x0C\x0E-\x1F\x7F]', '', text)

    # Заменяем все виды пробелов на обычный
    text = re.sub(r'\s+', ' ', text).strip()

    # Удаляем лишние символы в начале/конце
    text = text.strip()

    return text
```

#### 4.2.2. Приведение к нижнему регистру

Для лексического поиска (BM25, TF‑IDF) часто приводят текст к нижнему регистру, чтобы избежать чувствительности к регистру. Однако для векторного поиска это не обязательно, так как модели эмбеддингов обычно учитывают регистр в меньшей степени.

```python
def normalize_case(text: str, for_lexical_search: bool = False) -> str:
    """Приводит текст к нижнему регистру, если это необходимо."""
    if for_lexical_search:
        return text.lower()
    return text
```

#### 4.2.3. Удаление стоп-слов

Стоп-слова (предлоги, союзы, местоимения) часто удаляют при лексическом поиске, чтобы уменьшить шум и ускорить поиск. Однако при векторном поиске они сохраняются, так как модели эмбеддингов учитывают их контекст.

```python
from nltk.corpus import stopwords
import nltk

nltk.download('stopwords')
STOPWORDS_EN = set(stopwords.words('english'))
STOPWORDS_RU = set()  # можно загрузить русские стоп-слова отдельно

def remove_stopwords(text: str, language: str = 'en') -> str:
    """Удаляет стоп-слова из текста (только для лексического поиска)."""
    stopwords_set = STOPWORDS_RU if language == 'ru' else STOPWORDS_EN
    words = text.split()
    filtered_words = [w for w in words if w.lower() not in stopwords_set]
    return " ".join(filtered_words)
```

#### 4.2.4. Лемматизация и стемминг

- **Стемминг** — упрощённое усечение слов до корня (например, "running" → "run", "beautiful" → "beauti"). Быстрый, но грубый метод.
- **Лемматизация** — приведение слова к нормальной форме (например, "running" → "run", "better" → "good"). Более точный, но медленный метод, требует словаря.

**Для английского языка** используем `spaCy`:

```python
import spacy

nlp_en = spacy.load("en_core_web_sm")

def lemmatize_en(text: str) -> str:
    """Лемматизация текста на английском."""
    doc = nlp_en(text)
    return " ".join(token.lemma_ for token in doc if not token.is_punct)
```

**Для русского языка** используем `pymorphy2`:

```python
import pymorphy2

morph = pymorphy2.MorphAnalyzer()

def lemmatize_ru(text: str) -> str:
    """Лемматизация текста на русском."""
    words = text.split()
    lemmas = []
    for word in words:
        parsed = morph.parse(word)[0]
        lemmas.append(parsed.normal_form)
    return " ".join(lemmas)
```

**Сравнение стемминга и лемматизации:**

| Критерий | Стемминг | Лемматизация |
|----------|----------|--------------|
| Скорость | Очень высокая | Медленнее (в 2-3 раза) |
| Точность | Грубая, иногда ошибки | Высокая |
| Ресурсы | Минимальные | Нужен словарь/модель |
| Когда использовать | Для лексического поиска (BM25) | Для точного анализа, поиска по смыслу |

#### 4.2.5. Обработка таблиц и списков

Таблицы и списки содержат структурированную информацию, которую важно сохранить при преобразовании в текст.

```python
def format_table_as_text(table: list) -> str:
    """Преобразует таблицу (список строк) в текстовое представление."""
    if not table:
        return ""
    # Определяем ширину столбцов
    col_widths = [max(len(str(row[i])) for row in table) for i in range(len(table[0]))]
    lines = []
    for row in table:
        line = " | ".join(str(cell).ljust(col_widths[i]) for i, cell in enumerate(row))
        lines.append(line)
    return "\n".join(lines)

def format_list_as_text(items: list) -> str:
    """Преобразует список в текст с маркерами."""
    return "\n".join(f"• {item}" for item in items)
```

#### 4.2.6. Полная функция очистки

```python
def full_clean_pipeline(text: str, language: str = 'en') -> str:
    """
    Полный пайплайн очистки текста.
    """
    if not text:
        return ""

    # Базовые шаги (всегда)
    text = clean_text(text)

    # Для лексического поиска (опционально)
    # text = normalize_case(text, for_lexical_search=True)
    # text = remove_stopwords(text, language)

    # Для лемматизации (опционально, если нужна нормализация)
    if language == 'ru':
        text = lemmatize_ru(text)
    else:
        text = lemmatize_en(text)

    return text
```

---

### 4.3. Работа с метаданными документов

Метаданные — это структурированная информация о документе, которая не является частью основного текста, но критически важна для фильтрации, цитирования и управления версиями. Сохранение метаданных позволяет пользователю задавать уточняющие вопросы ("покажи только из документов за 2024 год") и проверять источники.

**Рекомендуемый набор метаданных:**

| Поле | Тип | Описание | Пример |
|------|-----|----------|--------|
| `source` | str | Имя файла или URL | `"report_2024.pdf"` |
| `file_path` | str | Полный путь к файлу | `"/data/reports/report_2024.pdf"` |
| `doc_id` | str | Уникальный идентификатор | `"doc_001"` |
| `title` | str | Заголовок документа | `"Годовой отчёт 2024"` |
| `author` | str | Автор | `"Иванов И.И."` |
| `created_date` | str | Дата создания | `"2024-01-15"` |
| `modified_date` | str | Дата изменения | `"2024-12-10"` |
| `language` | str | Язык документа | `"ru"` |
| `version` | str | Версия документа | `"v2.3"` |
| `page_number` | int | Номер страницы (для чанка) | `12` |
| `section` | str | Раздел/глава | `"Глава 3. Налогообложение"` |
| `tags` | list | Теги/категории | `["финансы", "налог"]` |
| `department` | str | Отдел | `"Бухгалтерия"` |
| `is_active` | bool | Актуален ли документ | `True` |

**Структура для хранения в векторной БД (Chroma):**

```python
from dataclasses import dataclass
from typing import List, Optional
from datetime import datetime

@dataclass
class DocumentMetadata:
    source: str
    file_path: str
    doc_id: str
    title: str = ""
    author: str = ""
    created_date: Optional[str] = None
    modified_date: Optional[str] = None
    language: str = "en"
    version: str = "1.0"
    page_number: int = 0
    section: str = ""
    tags: List[str] = None
    department: str = ""
    is_active: bool = True

    def to_dict(self) -> dict:
        """Преобразует в словарь для сохранения в БД."""
        return {
            "source": self.source,
            "file_path": self.file_path,
            "doc_id": self.doc_id,
            "title": self.title,
            "author": self.author,
            "created_date": self.created_date,
            "modified_date": self.modified_date,
            "language": self.language,
            "version": self.version,
            "page_number": self.page_number,
            "section": self.section,
            "tags": self.tags or [],
            "department": self.department,
            "is_active": str(self.is_active).lower(),
        }
```

**Пример фильтрации по метаданным в Chroma:**

```python
import chromadb

client = chromadb.PersistentClient(path="./chroma_db")
collection = client.get_collection("documents")

# Поиск только в документах за 2024 год
results = collection.query(
    query_texts=["налог на прибыль"],
    n_results=10,
    where={
        "$and": [
            {"created_date": {"$gte": "2024-01-01"}},
            {"created_date": {"$lte": "2024-12-31"}}
        ]
    }
)

# Фильтрация по автору
results = collection.query(
    query_texts=["налог на прибыль"],
    n_results=10,
    where={"author": "Иванов И.И."}
)
```

---

### 4.4. Работа с мультиязычными документами

Корпоративные данные могут быть на разных языках. Для RAG это создаёт дополнительную сложность: нужно правильно определять язык, использовать подходящие модели эмбеддингов и, возможно, выполнять перевод.

#### 4.4.1. Определение языка

```python
from langdetect import detect, DetectorFactory
import fasttext

# Для langdetect
DetectorFactory.seed = 42

def detect_language_langdetect(text: str) -> str:
    """Определяет язык текста с помощью langdetect."""
    try:
        return detect(text)
    except:
        return "unknown"

# Для fasttext (требует скачивания модели)
# model = fasttext.load_model('lid.176.bin')
def detect_language_fasttext(text: str) -> str:
    """Определяет язык текста с помощью fasttext."""
    # prediction = model.predict(text)
    # return prediction[0][0].replace('__label__', '')
    pass
```

**Сравнение методов:**

| Библиотека | Точность | Скорость | Языки | Требования |
|------------|----------|----------|-------|------------|
| **langdetect** | Высокая (для основных языков) | Средняя | 55+ | Нет (чистый Python) |
| **fasttext** | Очень высокая | Высокая | 176 | Нужно скачать модель (100+ МБ) |
| **spaCy** | Высокая | Низкая | ~20 | Нужна языковая модель |

**Рекомендация:** для быстрой проверки используйте `langdetect`, для продакшена — `fasttext`.

#### 4.4.2. Мультиязычные модели эмбеддингов

| Модель | Размерность | Языки | MTEB (среднее) | Примечание |
|--------|-------------|-------|----------------|------------|
| **intfloat/multilingual-e5-large** | 1024 | 100+ | ~65 | Лучшая open‑source, требует нормализации |
| **intfloat/multilingual-e5-base** | 768 | 100+ | ~63 | Баланс скорость/качество |
| **LaBSE** | 768 | 109 | ~58 | Хороша для перевода, но устаревает |
| **paraphrase-multilingual-MiniLM-L12-v2** | 384 | 50+ | ~57 | Быстрая, подходит для прототипов |
| **BAAI/bge-m3** | 1024 | 100+ | ~66 | Новая, отличное качество |

**Рекомендация:** для продакшена используйте `intfloat/multilingual-e5-large` или `BAAI/bge-m3`.

**Пример генерации эмбеддингов для мультиязычных документов:**

```python
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("intfloat/multilingual-e5-large")

def get_embedding(text: str) -> list:
    """Генерирует эмбеддинг для текста на любом языке."""
    # Для E5 моделей требуется префикс "query: " или "passage: "
    embedding = model.encode("passage: " + text, normalize_embeddings=True)
    return embedding.tolist()
```

#### 4.4.3. Особенности русского языка

Русская морфология (падежи, склонения, спряжения) делает лексический поиск сложным — одно и то же слово может иметь множество форм. Поэтому для русского языка:

- Обязательно используйте **лемматизацию** (pymorphy2) или стемминг перед лексическим поиском.
- Для векторного поиска лучше использовать мультиязычные модели, которые обучались на русских текстах (например, multilingual-e5).
- Учитывайте, что модели эмбеддингов для русского языка могут иметь меньшую точность, чем для английского — это нормально, компенсируется качественным чанкингом и реранкингом.

---

### 4.5. Real‑time индексация и управление версиями

В продакшене документы не статичны: они добавляются, обновляются и удаляются. Система должна поддерживать инкрементальное обновление индекса без остановки работы.

#### 4.5.1. Инкрементальное обновление

**Проблема:** при добавлении новых документов не хочется перестраивать весь индекс заново.

**Решение (Chroma):** просто добавляем новые документы с новыми ID.

```python
collection.add(
    documents=[new_text],
    metadatas=[new_metadata],
    ids=[new_doc_id],
    embeddings=[new_embedding]  # опционально, можно сгенерировать внутри
)
```

**Решение (FAISS):** используем `IndexIDMap` для поддержки обновлений.

```python
import faiss
import numpy as np

# Создаём индекс
index = faiss.IndexFlatIP(dimension)  # или HNSW
id_map = faiss.IndexIDMap(index)

# Добавляем векторы с ID
embeddings = np.array([new_embedding]).astype('float32')
id_map.add_with_ids(embeddings, np.array([new_doc_id]))
```

#### 4.5.2. Удаление устаревших чанков

**Стратегия мягкого удаления:** не удаляем физически, а помечаем `is_active=False` в метаданных и фильтруем при поиске.

```python
# При поиске добавляем фильтр
results = collection.query(
    query_texts=["запрос"],
    n_results=10,
    where={"is_active": "true"}  # только активные
)
```

**Для FAISS** физическое удаление сложнее, поэтому мягкое удаление — основной подход.

#### 4.5.3. Версионирование

Храните версию документа в метаданных и при поиске выдавайте только последние версии.

```python
# При поиске используем фильтр по версии
results = collection.query(
    query_texts=["запрос"],
    n_results=10,
    where={"version": "latest"}
)
```

---

### 4.6. ETL-пайплайны (кратко)

Для регулярной индексации больших объёмов данных используют ETL-фреймворки:

- **Apache Airflow** — самый популярный, позволяет строить DAG (Directed Acyclic Graph) задач.
- **Prefect** — более современный, с лучшим UX и поддержкой асинхронности.
- **Dagster** — фокусируется на качественном тестировании и валидации данных.

**Пример простого DAG для Airflow:**

```python
from airflow import DAG
from airflow.operators.python_operator import PythonOperator
from datetime import datetime

default_args = {'owner': 'data_team', 'start_date': datetime(2024, 1, 1)}

dag = DAG('rag_ingestion', default_args=default_args, schedule_interval='@daily')

def extract_pdfs():
    # Загрузка PDF из папки
    pass

def clean_and_chunk():
    # Очистка и чанкинг
    pass

def embed_and_index():
    # Генерация эмбеддингов и индексация
    pass

extract = PythonOperator(task_id='extract', python_callable=extract_pdfs, dag=dag)
clean = PythonOperator(task_id='clean', python_callable=clean_and_chunk, dag=dag)
index = PythonOperator(task_id='index', python_callable=embed_and_index, dag=dag)

extract >> clean >> index
```

---

### 4.7. Практический пример: полный скрипт индексации

Ниже представлен полный скрипт, который загружает все PDF из папки, извлекает текст и метаданные, очищает их и сохраняет в JSON.

```python

!pip install pypdf pdfplumber

import os
import json
import logging
from datetime import datetime
from typing import Dict, List
from pathlib import Path

import pdfplumber
from pypdf import PdfReader
import pandas as pd

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

def extract_pdf_text(file_path: str) -> str:
    """Извлекает текст из PDF с fallback."""
    text = ""
    try:
        with pdfplumber.open(file_path) as pdf:
            for page in pdf.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
        if text.strip():
            return text
    except Exception as e:
        logger.warning(f"pdfplumber не сработал для {file_path}: {e}")

    try:
        with open(file_path, "rb") as f:
            reader = PdfReader(f)
            for page in reader.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
    except Exception as e:
        logger.error(f"Не удалось извлечь текст из {file_path}: {e}")
        return ""

    return text.strip()

def clean_text(text: str) -> str:
    """Очищает текст от шума."""
    import re
    if not text:
        return ""
    # Удаляем управляющие символы
    text = re.sub(r'[\x00-\x08\x0B\x0C\x0E-\x1F\x7F]', '', text)
    # Нормализуем пробелы
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def get_pdf_metadata(file_path: str) -> Dict:
    """Извлекает метаданные из PDF и файловой системы."""
    stat = os.stat(file_path)
    metadata = {
        "source": os.path.basename(file_path),
        "file_path": str(file_path),
        "doc_id": Path(file_path).stem,
        "created_date": datetime.fromtimestamp(stat.st_ctime).isoformat(),
        "modified_date": datetime.fromtimestamp(stat.st_mtime).isoformat(),
        "page_number": 0,
        "section": "",
        "tags": [],
        "department": "",
        "is_active": True
    }
    try:
        with open(file_path, "rb") as f:
            reader = PdfReader(f)
            info = reader.metadata
            if info:
                metadata["author"] = str(info.get('/Author', ''))
                metadata["title"] = str(info.get('/Title', ''))
    except Exception as e:
        logger.warning(f"Не удалось извлечь метаданные PDF для {file_path}: {e}")
    return metadata

def process_pdf_directory(input_dir: str, output_json: str) -> List[Dict]:
    """Обрабатывает все PDF в директории и сохраняет результат в JSON."""
    results = []
    pdf_files = list(Path(input_dir).glob("**/*.pdf"))

    if not pdf_files:
        logger.warning(f"PDF файлы не найдены в {input_dir}")
        return results

    logger.info(f"Найдено {len(pdf_files)} PDF файлов")

    for pdf_path in pdf_files:
        logger.info(f"Обработка: {pdf_path}")
        text = extract_pdf_text(str(pdf_path))
        if not text:
            logger.warning(f"Пропущен (пустой текст): {pdf_path}")
            continue

        cleaned_text = clean_text(text)
        metadata = get_pdf_metadata(str(pdf_path))

        results.append({
            "text": cleaned_text,
            "metadata": metadata
        })
        logger.info(f"Добавлен: {pdf_path} (длина текста: {len(cleaned_text)} символов)")

    with open(output_json, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

    logger.info(f"Сохранено {len(results)} документов в {output_json}")
    return results

if __name__ == "__main__":
    process_pdf_directory(
        input_dir="./documents",
        output_json="./extracted_data.json"
    )
```

---

### 4.8. Схема ETL-процесса (Mermaid)

```mermaid
flowchart TD
    A[Исходные файлы] --> B{Тип файла}
    B -->|PDF| C1[pypdf / pdfplumber]
    B -->|DOCX| C2[python-docx]
    B -->|HTML| C3[beautifulsoup4]
    B -->|Markdown| C4[markdown]
    B -->|Excel/CSV| C5[pandas]
    B -->|Изображения| C6[OCR / Tesseract]

    C1 --> D[Извлечение текста]
    C2 --> D
    C3 --> D
    C4 --> D
    C5 --> D
    C6 --> D

    D --> E[Очистка текста]
    E --> F[Извлечение метаданных]
    F --> G[Сохранение в JSON]
    G --> H[Далее: чанкинг и векторизация]
```

---

### 4.9. Контрольные вопросы и задания

**Вопросы для самопроверки:**

1. *Почему pdfplumber лучше подходит для извлечения таблиц из PDF, чем pypdf?*  
   **Ответ:** pdfplumber анализирует позиционирование букв и графические элементы, что позволяет определять границы ячеек и восстанавливать структуру таблицы. pypdf извлекает текст по потокам без учёта позиционирования.

2. *В каких случаях следует использовать стемминг вместо лемматизации?*  
   **Ответ:** Стемминг быстрее и проще, поэтому он подходит для лексического поиска (BM25) в высоконагруженных системах, где скорость критична. Лемматизация точнее, поэтому используется для задач, где важна семантическая точность, но она медленнее.

3. *Почему важно сохранять метаданные документов в RAG-системе?*  
   **Ответ:** Метаданные позволяют фильтровать документы при поиске (например, по дате, автору, категории), цитировать источники в ответе и управлять версиями (помечать устаревшие документы).

**Практическое задание:**

Модифицируйте скрипт из раздела 4.7 так, чтобы он обрабатывал **.docx и .html** файлы. Добавьте логирование количества успешно обработанных документов каждого типа. Результат должен сохраняться в тот же JSON-формат с полем `"type"`, указывающим исходный формат.

---

### 4.10. Список литературы

1. **pypdf Documentation.** – https://pypdf.readthedocs.io/ – официальная документация по работе с PDF.
2. **pdfplumber Documentation.** – https://pdfplumber.readthedocs.io/ – работа с PDF и таблицами.
3. **python-docx Documentation.** – https://python-docx.readthedocs.io/ – извлечение текста из .docx.
4. **BeautifulSoup Documentation.** – https://www.crummy.com/software/BeautifulSoup/ – парсинг HTML.
5. **pymorphy2 Documentation.** – https://pymorphy2.readthedocs.io/ – морфологический анализ русского языка.
6. **spaCy Documentation.** – https://spacy.io/ – лемматизация для английского и других языков.
7. **Airflow Documentation.** – https://airflow.apache.org/ – управление ETL-пайплайнами.



## Тема 5. Чанкинг (разбиение текста)

После того как мы извлекли и очистили текст из документов (раздел 4), следующим критическим этапом является **разбиение на чанки (chunking)**. Именно от того, как мы разрежем документы на фрагменты, напрямую зависит качество поиска: слишком маленький чанк теряет контекст, слишком большой — вносит шум и размывает семантику. В этой теме мы разберём все стратегии чанкинга, их математические основы, параметры и практическую реализацию, а также проведём эксперимент для выбора оптимальных настроек. **Весь код будет использовать результаты, полученные в разделе 4**, что обеспечивает полную преемственность пайплайна.

---

### 5.1. Что такое чанкинг и зачем он нужен

**Чанк (chunk)** — это фрагмент текста (предложение, абзац, смысловой блок), который подаётся на вход ретриверу для генерации эмбеддинга и последующего поиска. Весь документ разбивается на множество чанков, каждый из которых становится отдельной единицей в векторной базе данных.

#### 5.1.1. Почему нельзя использовать весь документ целиком?

Большинство LLM имеют ограничение на длину контекстного окна (например, 4096, 8192 или 128K токенов), но даже если окно достаточно большое, есть три причины для чанкинга:

1. **Точность поиска.** Если чанк слишком большой, он содержит много разнородной информации. Векторное представление такого чанка будет усреднённым, и поиск по конкретному запросу станет менее точным.
2. **Качество эмбеддингов.** Модели эмбеддингов имеют ограничение на длину входного текста (обычно 512 токенов для BERT-подобных моделей, до 1024 для современных). Обрезание или усреднение длинных текстов снижает качество.
3. **Шум.** Большой чанк содержит много нерелевантной информации, которая может «забить» сигнал и привести к ложным срабатываниям.

#### 5.1.2. Оптимальный размер чанка

Эмпирическое правило: **200–1000 токенов** на чанк.

| Размер чанка (токенов) | Преимущества | Недостатки |
|------------------------|--------------|------------|
| **< 200** | Высокая точность, много чанков | Теряется контекст, много шума |
| **200–500** | Хороший баланс | Требует настройки overlap |
| **500–1000** | Больше контекста | Может быть шумно, медленнее поиск |
| **> 1000** | Минимум чанков | Низкая точность, проблемы с эмбеддингами |

**Математическое обоснование:** Количество чанков для документа из $N$ токенов при размере чанка $S$ и перекрытии $O$:

$$
\text{NumChunks} = \left\lceil \frac{N - O}{S - O} \right\rceil
$$

Например, для документа 10 000 токенов, $S = 500$, $O = 50$:
- Количество чанков = $\lceil (10000 - 50) / (500 - 50) \rceil = \lceil 9950 / 450 \rceil = \lceil 22.11 \rceil = 23$ чанка.

#### 5.1.3. Связь с контекстным окном LLM

Суммарная длина всех чанков, передаваемых в LLM, не должна превышать контекстное окно. Если модель имеет окно 4096 токенов, а мы передаём 5 чанков по 500 токенов с промптом и ответом, то общая длина ≈ 5 × 500 + 500 (промпт) + 500 (ответ) = 3500 токенов, что влезает в окно. Если нужно больше чанков — используем модели с большим контекстным окном (например, 128K).

---

### 5.2. Стратегии разбиения текста

Существует несколько подходов к чанкингу, каждый со своими сильными и слабыми сторонами. Выбор стратегии зависит от структуры документов и требований к качеству.

#### 5.2.1. Fixed-size chunking (Фиксированный размер)

**Принцип:** текст разбивается на фрагменты строго фиксированного размера (например, по 500 символов или токенов) без учёта структуры.

**Схема:**
```
[Предложение 1] [Предложение 2] [Предложение 3] [Предложение 4] [Предложение 5]
      ↓                    ↓                    ↓                    ↓
  Чанк 1 (500 симв)    Чанк 2 (500 симв)    Чанк 3 (500 симв)    Чанк 4 (500 симв)
```

**Преимущества:** простой, быстрый, предсказуемый.
**Недостатки:** разрывает предложения, абзацы и смысловые блоки, теряет структуру документа.

**Когда использовать:** для однородных текстов без сложной структуры (например, логи файлов, сырые данные).

#### 5.2.2. Recursive chunking (Рекурсивное разбиение)

**Принцип:** текст разбивается иерархически по разделителям: сначала по самым крупным (например, абзацы), затем, если чанк всё ещё слишком большой, по предложениям, затем по словам. Это самый популярный и эффективный метод.

**Схема:**
```
[Документ]
    ↓
[Абзац 1] [Абзац 2] [Абзац 3]  ← разбиение по \n\n
    ↓         ↓
[Предложение 1] [Предложение 2] ← разбиение по \n, . , !
    ↓
[Слово 1] [Слово 2]            ← разбиение по пробелам (если нужно)
```

**Пример иерархии разделителей:**
1. `\n\n` (двойной перевод строки — абзац)
2. `\n` (один перевод строки)
3. `. ` (точка с пробелом — предложение)
4. `! ` (восклицательный знак)
5. `? ` (вопросительный знак)
6. `, ` (запятая)
7. пробел

**Преимущества:** сохраняет структуру документа, не разрывает предложения без необходимости, адаптивен к разным типам текстов.
**Недостатки:** сложнее в реализации, требует тщательного выбора разделителей.

**Когда использовать:** для любых структурированных текстов — статьи, документация, книги, отчёты.

#### 5.2.3. Semantic chunking (Смысловое разбиение)

**Принцип:** чанки определяются не по длине, а по смысловым границам — смене темы, завершённости мысли. Для этого используется анализ эмбеддингов: текст разбивается на предложения, затем каждое предложение кодируется, и границы проводятся там, где косинусное расстояние между соседними предложениями превышает порог.

**Схема:**
```
[Предложение 1] [Предложение 2] [Предложение 3] [Предложение 4] [Предложение 5]
     ↓                ↓                ↓                ↓                ↓
   Эмбеддинг 1      Эмбеддинг 2      Эмбеддинг 3      Эмбеддинг 4      Эмбеддинг 5
     ↓                ↓                ↓                ↓                ↓
   sim(1,2)=0.95   sim(2,3)=0.92    sim(3,4)=0.45    sim(4,5)=0.90
                                           ↑
                                    Смена темы! Граница чанка.
```

**Преимущества:** чанки семантически цельны, что улучшает качество поиска.
**Недостатки:** ресурсоёмко (нужно вычислять эмбеддинги для каждого предложения), сложно подобрать порог.

**Когда использовать:** для сложных, многотемных документов, где важна смысловая целостность (научные статьи, книги).

#### 5.2.4. Sliding window с перекрытием (overlap)

**Принцип:** каждый следующий чанк начинается не с конца предыдущего, а немного раньше, захватывая часть предыдущего чанка. Это предотвращает потерю информации на границах.

**Схема:**
```
[Чанк 1]                   [Чанк 3]
  [Чанк 2]                   [Чанк 4]
    ↓                         ↓
  overlap (10–20%)          overlap (10–20%)
```

**Преимущества:** сохраняет контекст на границах, уменьшает риск потери важной информации.
**Недостатки:** увеличивает количество чанков и избыточность.

**Когда использовать:** всегда в сочетании с другими стратегиями, особенно для текстов с длинными предложениями.

#### 5.2.5. Sentence‑based / Paragraph‑based (По предложениям/абзацам)

**Принцип:** разбиение по естественным границам — предложениям или абзацам.

**Преимущества:** максимально сохраняет структуру, идеально для диалогов и юридических документов.
**Недостатки:** размер чанков непредсказуем (может быть слишком большим или слишком маленьким).

**Когда использовать:** для документов с чёткой структурой (законы, инструкции, диалоги).

---

### 5.3. Параметры чанкинга

#### 5.3.1. Выбор `chunk_size`

- Для BERT-подобных моделей (384/768d): **256–512 токенов**.
- Для современных моделей (multilingual-e5, BGE): **512–1024 токенов**.
- Если модель поддерживает длинные последовательности (например, 8192 токенов), можно увеличить до 1024–2048 токенов.

**Практический совет:** начните с 500 токенов и экспериментируйте.

#### 5.3.2. Выбор `chunk_overlap`

- **Стандарт:** 10–20% от размера чанка.
- Для S=500: overlap = 50–100 токенов.
- Для S=1000: overlap = 100–200 токенов.

**Зачем он нужен:** предложение на границе двух чанков может быть разорвано. Overlap гарантирует, что оно попадёт в оба чанка целиком, и при поиске не будет потеряно.

#### 5.3.3. Обработка коротких и длинных чанков

- **Короткие чанки (< 50 токенов):** объединять с соседними (если они не являются заголовками или отдельными пунктами).
- **Длинные чанки (> max_size):** разбивать рекурсивно, используя более мелкие разделители.

#### 5.3.4. Сохранение структуры документа

Важно сохранять заголовки, маркированные списки и таблицы при разбиении. Например, можно добавить заголовок раздела в метаданные каждого чанка, чтобы сохранить контекст.

---

### 5.4. Практическая реализация с использованием данных из раздела 4

Мы напишем полный код, который загружает извлечённые в разделе 4 данные (`extracted_data.json`) и применяет к ним различные стратегии чанкинга, сохраняя результаты в новом JSON-файле. Все примеры кода используют реальные данные из предыдущего шага.

#### 5.4.1. Полный код для чанкинга

```python
# ================================================================
# Тема 5. Чанкинг (разбиение текста)
# Использует данные из extracted_data.json (результат раздела 4)
# ================================================================

import json
import re
from typing import List, Dict
from pathlib import Path

class RecursiveTextSplitter:
    """
    Рекурсивный сплиттер с иерархией разделителей.
    Разбивает текст на чанки, сохраняя структуру документа.
    """
    def __init__(self, chunk_size: int = 500, chunk_overlap: int = 50):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        # Иерархия разделителей: от крупных к мелким
        self.separators = ["\n\n", "\n", ". ", "! ", "? ", ", ", " "]

    def split_document(self, text: str, metadata: Dict) -> List[Dict]:
        """
        Разбивает текст на чанки и добавляет метаданные к каждому чанку.
        Возвращает список словарей: {"text": str, "metadata": dict}
        """
        chunks_text = self._split_text(text)
        chunks_with_meta = []
        for i, chunk_text in enumerate(chunks_text):
            chunk_meta = metadata.copy()
            chunk_meta["chunk_index"] = i
            chunk_meta["chunk_length"] = len(chunk_text)
            chunks_with_meta.append({
                "text": chunk_text,
                "metadata": chunk_meta
            })
        return chunks_with_meta

    def _split_text(self, text: str) -> List[str]:
        """Разбивает текст на чанки (внутренний метод)."""
        if not text:
            return []
        chunks = []
        current_chunk = []
        current_len = 0

        # Разбиваем текст рекурсивно по разделителям
        segments = self._split_by_separators(text, self.separators)

        for segment in segments:
            seg_len = len(segment)
            # Если текущий чанк + новый сегмент превышает max_size и чанк не пуст
            if current_len + seg_len > self.chunk_size and current_chunk:
                chunks.append("".join(current_chunk).strip())
                # Извлекаем overlap из предыдущего чанка
                overlap_text = self._get_overlap("".join(current_chunk), self.chunk_overlap)
                current_chunk = [overlap_text]
                current_len = len(overlap_text)

            current_chunk.append(segment)
            current_len += seg_len

        if current_chunk:
            chunks.append("".join(current_chunk).strip())

        return chunks

    def _split_by_separators(self, text: str, separators: List[str]) -> List[str]:
        """Рекурсивно разбивает текст по разделителям."""
        if not text:
            return []

        separator = separators[0]
        remaining_seps = separators[1:]

        if not remaining_seps:
            # Последний разделитель — пробел
            return text.split(separator)

        parts = text.split(separator)
        result = []
        for i, part in enumerate(parts):
            if len(part) <= self.chunk_size:
                result.append(part)
            else:
                # Рекурсивно разбиваем более мелким разделителем
                sub_parts = self._split_by_separators(part, remaining_seps)
                result.extend(sub_parts)

            if i < len(parts) - 1:
                result.append(separator)  # возвращаем разделитель обратно

        return result

    def _get_overlap(self, text: str, overlap_len: int) -> str:
        """Возвращает последние `overlap_len` символов текста."""
        return text[-overlap_len:] if len(text) > overlap_len else text


def load_extracted_data(json_path: str = "./extracted_data.json") -> List[Dict]:
    """Загружает данные, полученные в разделе 4."""
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data


def process_chunking(input_json: str = "./extracted_data.json",
                     output_json: str = "./chunks_data.json",
                     chunk_size: int = 500,
                     chunk_overlap: int = 50) -> List[Dict]:
    """
    Загружает данные из input_json, применяет чанкинг и сохраняет в output_json.
    Возвращает список всех чанков с метаданными.
    """
    # Загрузка данных из раздела 4
    documents = load_extracted_data(input_json)
    print(f"Загружено {len(documents)} документов.")

    splitter = RecursiveTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    all_chunks = []

    for doc in documents:
        text = doc["text"]
        metadata = doc["metadata"]
        if not text:
            continue
        chunks = splitter.split_document(text, metadata)
        all_chunks.extend(chunks)

    # Сохранение результатов
    with open(output_json, "w", encoding="utf-8") as f:
        json.dump(all_chunks, f, ensure_ascii=False, indent=2)

    print(f"Создано {len(all_chunks)} чанков из {len(documents)} документов.")
    print(f"Результат сохранён в {output_json}")
    return all_chunks


def compare_chunk_sizes(input_json: str = "./extracted_data.json"):
    """
    Сравнивает три размера чанков на первом документе из загруженных данных.
    Выводит статистику для каждого размера.
    """
    documents = load_extracted_data(input_json)
    if not documents:
        print("Нет данных для эксперимента.")
        return

    sample_doc = documents[0]
    text = sample_doc["text"]
    metadata = sample_doc["metadata"]

    sizes = [200, 500, 1000]
    overlaps = [20, 50, 100]  # 10% от размера

    print("=" * 60)
    print(f"Эксперимент на документе: {metadata.get('source', 'unknown')}")
    print(f"Длина текста: {len(text)} символов")
    print("=" * 60)

    for size, overlap in zip(sizes, overlaps):
        splitter = RecursiveTextSplitter(chunk_size=size, chunk_overlap=overlap)
        chunks = splitter.split_document(text, metadata)
        print(f"\nРазмер чанка: {size}, overlap: {overlap}")
        print(f"  Количество чанков: {len(chunks)}")
        if chunks:
            print(f"  Пример первого чанка (первые 100 символов):")
            print(f"    {chunks[0]['text'][:100]}...")
        lengths = [len(c['text']) for c in chunks]
        print(f"  Средняя длина: {sum(lengths)/len(lengths):.0f} символов")
        print(f"  Минимальная: {min(lengths)}, максимальная: {max(lengths)}")


if __name__ == "__main__":
    # Шаг 1: применить чанкинг с параметрами по умолчанию
    chunks = process_chunking(
        input_json="./extracted_data.json",
        output_json="./chunks_data.json",
        chunk_size=500,
        chunk_overlap=50
    )

    # Шаг 2: провести сравнение размеров
    compare_chunk_sizes()
```



### 5.4.2. Использование LangChain (альтернативный вариант)

**Что такое LangChain?**

LangChain — это популярный open-source фреймворк, предназначенный для упрощения разработки приложений на основе больших языковых моделей (LLM). Он предоставляет стандартизированные интерфейсы для работы с LLM, цепочками вычислений, инструментами и внешними данными. В контексте RAG LangChain особенно полезен, так как предлагает готовые компоненты для всех этапов пайплайна: загрузчики документов, сплиттеры текста, модели эмбеддингов, векторные хранилища и шаблоны для создания RAG-цепочек.

**Зачем использовать LangChain для чанкинга?**

Вместо того чтобы писать сплиттер с нуля (как мы сделали в разделе 5.4.1), мы можем воспользоваться готовым и хорошо протестированным решением от LangChain. Это даёт несколько преимуществ:

1. **Стандартизация.** Код становится более читаемым и переносимым между проектами.
2. **Гибкость.** LangChain предлагает множество типов сплиттеров под разные задачи.
3. **Поддержка.** Фреймворк активно развивается, и вы получаете доступ к новым функциям и исправлениям.
4. **Интеграция.** Сплиттеры легко комбинируются с другими компонентами LangChain (загрузчики, эмбеддеры, векторные БД).

**Какой сплиттер использовать?**

Для большинства задач рекомендуется начинать с `RecursiveCharacterTextSplitter`. Он работает по тому же принципу, что и наш рекурсивный сплиттер: пытается разбить текст по иерархии разделителей, сохраняя структуру документа.

---

**Полный код для чанкинга с использованием LangChain**

Установите необходимую библиотеку:

```bash
pip install langchain-text-splitters
```

Теперь полный скрипт, который загружает данные из `extracted_data.json` (результат раздела 4), применяет чанкинг с помощью LangChain и сохраняет результат в `chunks_data_langchain.json`:

```python
# ================================================================
# Тема 5. Чанкинг (разбиение текста) с использованием LangChain
# Альтернативный вариант к разделу 5.4.1
# Использует данные из extracted_data.json (результат раздела 4)
# ================================================================

import json
from typing import List, Dict

from langchain_text_splitters import RecursiveCharacterTextSplitter


def load_extracted_data(json_path: str = "./extracted_data.json") -> List[Dict]:
    """Загружает данные, полученные в разделе 4."""
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data


def split_with_langchain(input_json: str = "./extracted_data.json",
                         output_json: str = "./chunks_data_langchain.json",
                         chunk_size: int = 500,
                         chunk_overlap: int = 50) -> List[Dict]:
    """
    Загружает данные из input_json, применяет чанкинг с помощью LangChain
    и сохраняет результат в output_json.
    Возвращает список всех чанков с метаданными.
    """
    # 1. Загрузка данных из раздела 4
    documents = load_extracted_data(input_json)
    print(f"Загружено {len(documents)} документов.")

    # 2. Настройка сплиттера LangChain
    # Используем RecursiveCharacterTextSplitter с теми же параметрами,
    # что и в нашей ручной реализации
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        # Иерархия разделителей: от крупных к мелким
        separators=["\n\n", "\n", ". ", "! ", "? ", ", ", " "],
        length_function=len,  # считаем длину в символах
    )

    all_chunks = []

    # 3. Обработка каждого документа
    for doc in documents:
        text = doc["text"]
        metadata = doc["metadata"]

        if not text:
            continue

        # Разбиваем текст на чанки с помощью LangChain
        chunks_text = splitter.split_text(text)

        # Добавляем метаданные к каждому чанку
        for i, chunk_text in enumerate(chunks_text):
            chunk_meta = metadata.copy()
            chunk_meta["chunk_index"] = i
            chunk_meta["chunk_length"] = len(chunk_text)
            all_chunks.append({
                "text": chunk_text,
                "metadata": chunk_meta
            })

    # 4. Сохранение результатов
    with open(output_json, "w", encoding="utf-8") as f:
        json.dump(all_chunks, f, ensure_ascii=False, indent=2)

    print(f"Создано {len(all_chunks)} чанков из {len(documents)} документов.")
    print(f"Результат сохранён в {output_json}")
    return all_chunks


def compare_chunk_sizes_langchain(input_json: str = "./extracted_data.json"):
    """
    Сравнивает три размера чанков на первом документе из загруженных данных.
    Использует LangChain для сплиттинга.
    """
    documents = load_extracted_data(input_json)
    if not documents:
        print("Нет данных для эксперимента.")
        return

    sample_doc = documents[0]
    text = sample_doc["text"]
    metadata = sample_doc["metadata"]

    sizes = [200, 500, 1000]
    overlaps = [20, 50, 100]  # 10% от размера

    print("=" * 60)
    print(f"Эксперимент (LangChain) на документе: {metadata.get('source', 'unknown')}")
    print(f"Длина текста: {len(text)} символов")
    print("=" * 60)

    for size, overlap in zip(sizes, overlaps):
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=size,
            chunk_overlap=overlap,
            separators=["\n\n", "\n", ". ", "! ", "? ", ", ", " "],
            length_function=len,
        )
        chunks_text = splitter.split_text(text)
        print(f"\nРазмер чанка: {size}, overlap: {overlap}")
        print(f"  Количество чанков: {len(chunks_text)}")
        if chunks_text:
            print(f"  Пример первого чанка (первые 100 символов):")
            print(f"    {chunks_text[0][:100]}...")
        lengths = [len(c) for c in chunks_text]
        if lengths:
            print(f"  Средняя длина: {sum(lengths)/len(lengths):.0f} символов")
            print(f"  Минимальная: {min(lengths)}, максимальная: {max(lengths)}")


if __name__ == "__main__":
    # Шаг 1: применить чанкинг с параметрами по умолчанию
    chunks = split_with_langchain(
        input_json="./extracted_data.json",
        output_json="./chunks_data_langchain.json",
        chunk_size=500,
        chunk_overlap=50
    )

    # Шаг 2: провести сравнение размеров
    compare_chunk_sizes_langchain()
```

**Ключевые моменты кода:**

1. **Импорт:** Мы импортируем `RecursiveCharacterTextSplitter` из пакета `langchain_text_splitters`.
2. **Настройка сплиттера:** Передаём те же параметры (`chunk_size`, `chunk_overlap`, `separators`), что и в ручной реализации, для честного сравнения.
3. **Обработка:** Цикл по документам и добавление метаданных к каждому чанку полностью повторяет логику из раздела 5.4.1.
4. **Эксперимент:** Функция `compare_chunk_sizes_langchain` позволяет сравнить разные размеры чанков, используя LangChain.

**Сравнение результатов:**

Результаты, полученные с помощью LangChain, будут идентичны результатам ручной реализации из раздела 5.4.1, так как мы используем те же параметры. Это демонстрирует, что LangChain предоставляет надёжный и стандартизированный инструмент, который можно использовать вместо самописных решений.

**Дополнительные сплиттеры LangChain:**

Помимо `RecursiveCharacterTextSplitter`, LangChain предлагает и другие типы сплиттеров для специфических задач:

- `CharacterTextSplitter` — разбиение по фиксированному числу символов.
- `TokenTextSplitter` — разбиение по числу токенов (с использованием различных токенизаторов).
- `SentenceTransformersTokenTextSplitter` — разбиение с учётом лимита токенов для моделей эмбеддингов.

Выбор конкретного сплиттера зависит от ваших данных и требований к качеству поиска.



---

### 5.5. Эксперимент: сравнение размеров чанков на реальных данных

Результат выполнения функции `compare_chunk_sizes()` на одном документе (например, `report_2024.pdf`) будет выглядеть так:

```
============================================================
Эксперимент на документе: report_2024.pdf
Длина текста: 12450 символов
============================================================

Размер чанка: 200, overlap: 20
  Количество чанков: 65
  Пример первого чанка (первые 100 символов):
    Налог на прибыль организаций регулируется главой 25 НК РФ. Ставка налога составляет 20%...
  Средняя длина: 198 символов
  Минимальная: 45, максимальная: 220

Размер чанка: 500, overlap: 50
  Количество чанков: 26
  Пример первого чанка (первые 100 символов):
    Налог на прибыль организаций регулируется главой 25 НК РФ. Ставка налога составляет 20%. При этом 3% зачисляется в федеральный бюджет, 17% — в региональный...
  Средняя длина: 485 символов
  Минимальная: 120, максимальная: 510

Размер чанка: 1000, overlap: 100
  Количество чанков: 13
  Пример первого чанка (первые 100 символов):
    Налог на прибыль организаций регулируется главой 25 НК РФ. Ставка налога составляет 20%. При этом 3% зачисляется в федеральный бюджет, 17% — в региональный. Особенности расчёта для ИТ-компаний...
  Средняя длина: 980 символов
  Минимальная: 340, максимальная: 1010
```

**Вывод:** для данного документа размер 500 токенов даёт хороший баланс между количеством чанков и сохранением контекста. При необходимости можно выбрать 200 для более точного поиска по узким вопросам или 1000 для лучшего охвата контекста.

---

### 5.6. Контрольные вопросы

1. *Почему слишком маленький чанк (менее 100 токенов) ухудшает качество поиска?*  
   **Ответ:** Маленький чанк содержит недостаточно контекста. Эмбеддинг такого чанка будет неполным, и запрос, требующий более широкого контекста, не сможет найти нужный фрагмент. Кроме того, увеличивается количество чанков, что замедляет поиск.

2. *В чём преимущество рекурсивного сплиттера перед фиксированным?*  
   **Ответ:** Рекурсивный сплиттер учитывает структуру документа и не разрывает предложения и абзацы без необходимости. Это сохраняет смысловую целостность чанков и улучшает качество эмбеддингов и поиска.

3. *Какой overlap рекомендуется использовать и почему?*  
   **Ответ:** Рекомендуется overlap = 10–20% от размера чанка. Это гарантирует, что предложения на границе двух чанков не будут потеряны, так как они попадут в оба чанка целиком.

---

### 5.7. Задания

1. **Интеграция с данными из раздела 4.** Запустите скрипт `process_chunking` для вашего `extracted_data.json`. Сохраните результат в `chunks_data.json`. Выведите общее количество чанков и средний размер чанка.

2. **Эксперимент с размерами.** Проведите эксперимент `compare_chunk_sizes` на ваших данных. Сделайте вывод, какой размер чанка оптимален для вашего корпуса. Обоснуйте ответ на основе полученных чисел.

3. **Добавление позиционной информации.** Модифицируйте сплиттер так, чтобы в метаданные каждого чанка добавлялось поле `"start_char"` — позиция начала чанка в исходном тексте. Это пригодится для точного цитирования.

---

### 5.8. Список литературы

1. **LangChain Documentation.** *Text Splitters*. – https://python.langchain.com/docs/modules/data_connection/document_transformers/
2. **NLTK Documentation.** *Tokenization*. – https://www.nltk.org/api/nltk.tokenize.html
3. **spaCy Documentation.** *Sentence Segmentation*. – https://spacy.io/api/sentencizer
4. **Gao, Y., et al. (2023).** *Retrieval-Augmented Generation for Large Language Models: A Survey*. – раздел про обработку данных.



## Тема 6. Векторизация и эмбеддинги

После того как мы извлекли, очистили и разбили текст на чанки (разделы 4 и 5), наступает ключевой этап — **преобразование текста в числовые векторы (эмбеддинги)**. Именно качество этих векторов определяет, насколько хорошо ретривер сможет находить релевантные документы по запросу. В этой теме мы разберём математические основы эмбеддингов, сравним доступные модели, покажем практический код для их генерации с использованием данных из предыдущих разделов, а также проведём эксперимент по выбору оптимальной модели для вашей задачи.

---

### 6.1. Основы теории эмбеддингов

#### 6.1.1. Что такое векторное представление текста

**Эмбеддинг (embedding)** — это отображение текста (слова, предложения, документа) в вектор фиксированной размерности $d$:

$$
\text{embedding}: \text{text} \rightarrow \mathbb{R}^d
$$

Идея заключается в том, чтобы поместить семантически близкие тексты близко друг к другу в векторном пространстве. Например, предложения «собака лает» и «пёс гавкает» должны иметь близкие векторы, а «собака лает» и «компьютер работает» — далёкие.

**Формально:** для двух текстов $t_1$ и $t_2$ с эмбеддингами $\mathbf{v}_1, \mathbf{v}_2 \in \mathbb{R}^d$, семантическая близость измеряется через расстояние или сходство в этом пространстве.

#### 6.1.2. Размерность эмбеддингов и её влияние

| Размерность | Преимущества | Недостатки | Типичные модели |
|-------------|--------------|------------|-----------------|
| **384** | Очень быстрый, мало памяти | Может терять нюансы | all-MiniLM-L6-v2 |
| **768** | Хороший баланс качество/скорость | Требует больше ресурсов | all-mpnet-base-v2, BGE-base |
| **1024** | Высокое качество, лучше для сложных задач | Медленнее, больше памяти | BGE-large, E5-large |
| **4096** | Максимальная точность | Очень ресурсоёмкий | Некоторые специализированные модели |

**Общее правило:** чем выше размерность, тем точнее модель может различать тонкие смысловые оттенки, но тем больше требуется памяти для хранения векторов и времени для вычислений.

#### 6.1.3. Плотные vs разреженные векторы

- **Плотные векторы (dense embeddings)** — все компоненты вектора ненулевые, каждое измерение кодирует некоторую семантическую особенность. Используются в современных нейросетевых моделях (BERT, SBERT, E5). Хорошо работают для семантического поиска.
- **Разреженные векторы (sparse embeddings)** — большинство компонент равны нулю, ненулевые значения соответствуют присутствию конкретных терминов (TF‑IDF, BM25). Хорошо работают для лексического поиска по ключевым словам.

**Сравнение:**

| Характеристика | Плотные (Dense) | Разреженные (Sparse) |
|----------------|-----------------|----------------------|
| **Представление** | Вектор фиксированной размерности | Вектор размером словаря (сотни тысяч) |
| **Семантическая близость** | Учитывает синонимы, контекст | Только точные совпадения |
| **Скорость поиска** | Медленнее (ANN) | Быстрее (инвертированный индекс) |
| **Память** | Мало (384–1024 float) | Много (размер словаря) |

В RAG обычно используют **плотные эмбеддинги** для семантического поиска, иногда комбинируя с разреженными (гибридный поиск).

#### 6.1.4. Метрики расстояния для эмбеддингов

После того как тексты преобразованы в векторы, нам нужно уметь измерять их близость. Существует три основные метрики:

**1. Косинусное сходство (Cosine Similarity)**

Измеряет косинус угла между двумя векторами. Значение в диапазоне $[-1, 1]$, где 1 — векторы сонаправлены (максимально похожи), 0 — ортогональны (независимы), -1 — противоположны.

$$
\text{cosine\_similarity}(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\| \cdot \|\mathbf{b}\|} = \frac{\sum_{i=1}^{d} a_i b_i}{\sqrt{\sum_{i=1}^{d} a_i^2} \sqrt{\sum_{i=1}^{d} b_i^2}}
$$

**Когда использовать:** наиболее популярная метрика для эмбеддингов. Не зависит от длины векторов, что важно, так как эмбеддинги могут иметь разную норму. Если векторы нормализованы (L2-норма = 1), косинусное сходство эквивалентно скалярному произведению.

**2. Евклидово расстояние (Euclidean Distance)**

Обычное расстояние между точками в пространстве.

$$
\text{euclidean}(\mathbf{a}, \mathbf{b}) = \sqrt{\sum_{i=1}^{d} (a_i - b_i)^2} = \|\mathbf{a} - \mathbf{b}\|_2
$$

**Когда использовать:** когда важна абсолютная разница между векторами. В нормализованном пространстве евклидово расстояние связано с косинусным сходством: $\|\mathbf{a} - \mathbf{b}\|^2 = 2(1 - \text{cosine})$.

**3. Точечное произведение (Dot Product)**

$$
\text{dot}(\mathbf{a}, \mathbf{b}) = \sum_{i=1}^{d} a_i b_i
$$

**Когда использовать:** если все векторы нормализованы (L2-норма = 1), точечное произведение эквивалентно косинусному сходству. Используется в некоторых оптимизированных библиотеках (например, FAISS) для быстрого поиска.

#### 6.1.5. Визуализация эмбеддингов

Векторы размерности 384 или 768 невозможно визуализировать напрямую. Для этого используют методы снижения размерности:

- **t-SNE** (t-Distributed Stochastic Neighbor Embedding) — нелинейный метод, хорошо сохраняет локальную структуру (близкие точки остаются близкими). Медленный, плохо масштабируется на большие данные.
- **UMAP** (Uniform Manifold Approximation and Projection) — более быстрый и лучше сохраняет глобальную структуру.

**Пример визуализации (концептуальный):** если взять 1000 предложений из разных тем (медицина, юриспруденция, IT) и спроецировать их эмбеддинги в 2D, мы увидим, что предложения из одной темы образуют кластеры.

---

### 6.2. Модели для генерации эмбеддингов (с примерами кода)

Существует множество моделей для генерации эмбеддингов. Выбор модели зависит от языка, требуемой точности, скорости и бюджета. В этом разделе мы не только сравним модели, но и покажем, как их использовать в коде.

#### 6.2.1. Sentence‑Transformers (sbert.net)

Семейство моделей, оптимизированных для предложений и коротких текстов. Это самый простой способ начать работу с эмбеддингами.

| Модель | Размерность | MTEB (англ.) | Скорость (токен/с)* | Языки | Стоимость |
|--------|-------------|--------------|---------------------|-------|-----------|
| **all-MiniLM-L6-v2** | 384 | 56.3 | ~1200 | en | Бесплатно |
| **all-mpnet-base-v2** | 768 | 58.5 | ~600 | en | Бесплатно |
| **all-distilroberta-v1** | 768 | 57.5 | ~800 | en | Бесплатно |

*Скорость на GPU NVIDIA T4, batch_size=32.

**Пример кода для загрузки и использования:**

```python
from sentence_transformers import SentenceTransformer
import numpy as np

# Загрузка модели (первый раз может занять время, так как скачиваются веса)
model = SentenceTransformer('all-MiniLM-L6-v2')

# Пример текстов
texts = [
    "Налог на прибыль организаций регулируется главой 25 НК РФ.",
    "Ставка налога на прибыль составляет 20%.",
    "Самозанятые платят налог на профессиональный доход."
]

# Генерация эмбеддингов с нормализацией (для косинусного сходства)
embeddings = model.encode(
    texts,
    normalize_embeddings=True,  # L2-нормализация
    show_progress_bar=True      # Показывать прогресс
)

print(f"Размерность: {embeddings.shape[1]}")
print(f"Количество векторов: {embeddings.shape[0]}")
print(f"Вектор для первого текста (первые 5 значений): {embeddings[0][:5]}")
```

**Особенности работы с батчами:**

```python
# Для больших объёмов данных используйте батчи
batch_size = 32
all_embeddings = model.encode(
    large_texts,               # список из тысяч текстов
    batch_size=batch_size,
    normalize_embeddings=True,
    show_progress_bar=True
)
```

**Рекомендация:** для прототипов используйте `all-MiniLM-L6-v2` — он быстрый и даёт приемлемое качество.

---

#### 6.2.2. BGE (BAAI/bge)

Модели от Пекинской академии искусственного интеллекта — одни из лучших на MTEB. Особенность BGE — использование специальных префиксов для запросов и документов.

| Модель | Размерность | MTEB (англ.) | Скорость (токен/с) | Языки | Стоимость |
|--------|-------------|--------------|---------------------|-------|-----------|
| **BAAI/bge-base-en-v1.5** | 768 | 63.5 | ~600 | en | Бесплатно |
| **BAAI/bge-large-en-v1.5** | 1024 | 64.2 | ~300 | en | Бесплатно |
| **BAAI/bge-m3** | 1024 | ~66.0 | ~250 | 100+ | Бесплатно |

**Пример кода для BGE (с префиксами):**

```python
from sentence_transformers import SentenceTransformer
import numpy as np

# Загрузка модели
model = SentenceTransformer('BAAI/bge-base-en-v1.5')

# Важно: для BGE нужно использовать префиксы!
# Для запросов (queries) — "query: "
# Для документов (passages) — "passage: "

queries = [
    "query: Какие налоги платят самозанятые?",
    "query: Ставка налога на прибыль"
]

documents = [
    "passage: Налог на прибыль организаций регулируется главой 25 НК РФ.",
    "passage: Ставка налога на прибыль составляет 20%.",
    "passage: Самозанятые платят налог на профессиональный доход."
]

# Генерация эмбеддингов
query_embeddings = model.encode(queries, normalize_embeddings=True)
doc_embeddings = model.encode(documents, normalize_embeddings=True)

# Вычисление косинусного сходства
from sklearn.metrics.pairwise import cosine_similarity
scores = cosine_similarity(query_embeddings, doc_embeddings)

print("Матрица сходства (запросы × документы):")
print(scores)
```

**Важно:** BGE модели требуют префиксов для достижения заявленного качества. Без них точность может упасть на 5–10%.

**Рекомендация:** для высокоточных систем используйте `bge-large-en-v1.5` (английский) или `bge-m3` (мультиязычный).

---

#### 6.2.3. OpenAI Embeddings

Платные модели от OpenAI, доступны через API. Они не требуют развёртывания и дают стабильное качество.

| Модель | Размерность | MTEB (англ.) | Скорость | Языки | Стоимость (за 1M токенов) |
|--------|-------------|--------------|----------|-------|---------------------------|
| **text-embedding-ada-002** | 1536 | ~61.0 | API | ~50 | $0.10 |
| **text-embedding-3-small** | 1536 | ~62.3 | API | ~50 | $0.02 |
| **text-embedding-3-large** | 3072 | ~64.6 | API | ~50 | $0.13 |

**Пример кода для OpenAI API:**

```python
import openai
import numpy as np
from typing import List

# Установите ваш API-ключ
openai.api_key = "your-api-key-here"

def get_openai_embeddings(texts: List[str], model: str = "text-embedding-3-small") -> np.ndarray:
    """
    Генерирует эмбеддинги через OpenAI API.
    """
    # Для моделей text-embedding-3 нужно уменьшить размерность (опционально)
    # dimensions=1024  # можно уменьшить до 1024 для экономии
    response = openai.embeddings.create(
        model=model,
        input=texts,
        # dimensions=1024  # раскомментируйте, если хотите уменьшить размерность
    )
    embeddings = np.array([item.embedding for item in response.data])
    return embeddings

# Пример использования
texts = [
    "Налог на прибыль организаций регулируется главой 25 НК РФ.",
    "Ставка налога на прибыль составляет 20%."
]

embeddings = get_openai_embeddings(texts, model="text-embedding-3-small")
print(f"Размерность: {embeddings.shape[1]}")
print(f"Форма: {embeddings.shape}")
```

**Преимущества:** не нужно разворачивать модель, высокое качество.
**Недостатки:** платные, данные отправляются во внешний API, задержка зависит от сети.

**Важно:** для работы с OpenAI API установите библиотеку:
```bash
pip install openai
```

---

#### 6.2.4. Русскоязычные модели

Для работы с русским языком существуют специализированные модели. Они могут дать лучшее качество для русскоязычных текстов, чем универсальные модели.

| Модель | Размерность | MTEB (рус.) | Скорость | Языки | Стоимость |
|--------|-------------|-------------|----------|-------|-----------|
| **DeepPavlov/rubert-tiny** | 312 | ~55 | ~800 | ru | Бесплатно |
| **sbert_large_nlu_ru** | 1024 | ~58 | ~250 | ru | Бесплатно |
| **cointegrated/rubert-tiny2** | 312 | ~56 | ~800 | ru | Бесплатно |

**Пример кода для русской модели:**

```python
from sentence_transformers import SentenceTransformer
import numpy as np

# Работающая русская модель
model = SentenceTransformer('cointegrated/rubert-tiny2')

texts = [
    "Налог на прибыль организаций регулируется главой 25 НК РФ.",
    "Ставка налога на прибыль составляет 20%.",
    "Самозанятые платят налог на профессиональный доход."
]

embeddings = model.encode(texts, normalize_embeddings=True, show_progress_bar=True)

print(f"Размерность: {embeddings.shape[1]}")
print(f"Вектор для первого текста (первые 5 значений): {embeddings[0][:5]}")
```

**Примечание:** для русского языка часто используют мультиязычные модели (`multilingual-e5`, `bge-m3`), которые показывают хорошие результаты без необходимости отдельной русской модели. Однако для узкоспециализированных русских текстов специализированные модели могут дать небольшой прирост качества.

---

#### 6.2.5. Мультиязычные модели

Если ваш корпус содержит документы на разных языках, или вы планируете масштабировать систему на другие языки, используйте мультиязычные модели.

| Модель | Размерность | MTEB (среднее) | Скорость | Языки | Стоимость |
|--------|-------------|----------------|----------|-------|-----------|
| **intfloat/multilingual-e5-small** | 384 | ~62 | ~800 | 100+ | Бесплатно |
| **intfloat/multilingual-e5-base** | 768 | ~63.5 | ~600 | 100+ | Бесплатно |
| **intfloat/multilingual-e5-large** | 1024 | ~65.0 | ~300 | 100+ | Бесплатно |
| **sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2** | 384 | ~57 | ~800 | 50+ | Бесплатно |

**Пример кода для мультиязычной модели E5:**

```python
from sentence_transformers import SentenceTransformer
import numpy as np

# Загрузка мультиязычной модели
model = SentenceTransformer('intfloat/multilingual-e5-base')

# Важно: для E5 тоже нужны префиксы (как и для BGE)
# query: для запросов, passage: для документов

texts = [
    "passage: Налог на прибыль организаций регулируется главой 25 НК РФ.",
    "passage: The corporate income tax rate is 20%.",
    "passage: Самозанятые платят налог на профессиональный доход."
]

# Генерация эмбеддингов
embeddings = model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

print(f"Размерность: {embeddings.shape[1]}")
print(f"Количество векторов: {embeddings.shape[0]}")

# Пример поиска с E5
queries = ["query: Какие налоги платят самозанятые?"]
query_embeddings = model.encode(queries, normalize_embeddings=True)

from sklearn.metrics.pairwise import cosine_similarity
scores = cosine_similarity(query_embeddings, embeddings)

# Находим наиболее релевантный документ
best_idx = np.argmax(scores[0])
print(f"Наиболее релевантный документ: {texts[best_idx][:50]}...")
print(f"Сходство: {scores[0][best_idx]:.4f}")
```

**Важно:** E5 модели, как и BGE, требуют префиксов. Для запросов используйте `query: `, для документов — `passage: `. Это критично для качества.

**Рекомендация:** для мультиязычных систем используйте `multilingual-e5-large` — он обеспечивает наилучшее качество среди открытых моделей.

---

### 6.3. Практика работы с эмбеддингами

#### 6.3.1. Установка библиотек

Для работы с эмбеддингами нам понадобится `sentence-transformers` и совместимая версия `Pillow`:

```python
# Установка sentence-transformers и фикс Pillow
!pip install --upgrade sentence-transformers pillow transformers
!pip install pypdf pdfplumber
!pip uninstall pillow -y
!pip install pillow==10.4.0
```

#### 6.3.2. Генерация эмбеддингов с помощью sentence‑transformers

```python
from sentence_transformers import SentenceTransformer
import numpy as np
from typing import List

class EmbeddingGenerator:
    """
    Класс для генерации эмбеддингов с использованием sentence-transformers.
    Поддерживает батчевую обработку и автоматическое определение устройства.
    """
    def __init__(self, model_name: str = "all-MiniLM-L6-v2", device: str = None):
        """
        model_name: название модели из sentence-transformers
        device: "cpu", "cuda" или None (автоматическое определение)
        """
        import torch
        if device is None:
            device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model = SentenceTransformer(model_name, device=device)
        self.model_name = model_name
        self.dimension = self.model.get_sentence_embedding_dimension()
        print(f"Модель: {model_name}, размерность: {self.dimension}, устройство: {device}")

    def encode_batch(self, texts: List[str], batch_size: int = 32) -> np.ndarray:
        """
        Генерирует эмбеддинги для списка текстов батчами.
        """
        return self.model.encode(
            texts,
            batch_size=batch_size,
            show_progress_bar=True,
            normalize_embeddings=True,  # L2-нормализация для косинусного сходства
            convert_to_numpy=True,
        )

# Пример использования
generator = EmbeddingGenerator()  # устройство определится автоматически
texts = ["Пример текста", "Ещё один документ"]
embeddings = generator.encode_batch(texts)
print(f"Форма эмбеддингов: {embeddings.shape}")
```

#### 6.3.3. Кэширование эмбеддингов на диск

Генерация эмбеддингов для больших корпусов может занимать много времени. Кэширование позволяет сохранить результаты и не пересчитывать их при каждом запуске.

```python
import pickle
import os
from typing import Dict, List, Any
import numpy as np
from sentence_transformers import SentenceTransformer

class EmbeddingGenerator:
    """
    Класс для генерации эмбеддингов с автоматическим определением устройства.
    """
    def __init__(self, model_name: str = "all-MiniLM-L6-v2", device: str = None):
        import torch
        if device is None:
            device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model = SentenceTransformer(model_name, device=device)
        self.model_name = model_name
        self.dimension = self.model.get_sentence_embedding_dimension()
        print(f"Модель: {model_name}, размерность: {self.dimension}, устройство: {device}")

    def encode_batch(self, texts: List[str], batch_size: int = 32) -> np.ndarray:
        return self.model.encode(
            texts,
            batch_size=batch_size,
            show_progress_bar=True,
            normalize_embeddings=True,
            convert_to_numpy=True,
        )

class EmbeddingCache:
    """
    Класс для кэширования эмбеддингов на диск с использованием pickle.
    """
    def __init__(self, cache_dir: str = "./embedding_cache"):
        self.cache_dir = cache_dir
        os.makedirs(cache_dir, exist_ok=True)

    def get_cache_path(self, model_name: str, chunk_id: int) -> str:
        return os.path.join(self.cache_dir, f"{model_name}_{chunk_id}.pkl")

    def save_embeddings(self, model_name: str, chunk_id: int,
                        embeddings: np.ndarray, metadata: Dict = None):
        """Сохраняет эмбеддинги и метаданные на диск."""
        cache_path = self.get_cache_path(model_name, chunk_id)
        data = {"embeddings": embeddings, "metadata": metadata}
        with open(cache_path, "wb") as f:
            pickle.dump(data, f)

    def load_embeddings(self, model_name: str, chunk_id: int) -> Dict:
        """Загружает эмбеддинги с диска."""
        cache_path = self.get_cache_path(model_name, chunk_id)
        if os.path.exists(cache_path):
            with open(cache_path, "rb") as f:
                return pickle.load(f)
        return None

    def exists(self, model_name: str, chunk_id: int) -> bool:
        return os.path.exists(self.get_cache_path(model_name, chunk_id))

# Пример использования
generator = EmbeddingGenerator("all-MiniLM-L6-v2")  # device определится автоматически
cache = EmbeddingCache("./embedding_cache")

texts = ["Это текст 1", "Это текст 2"]
chunk_id = 1

if cache.exists(generator.model_name, chunk_id):
    data = cache.load_embeddings(generator.model_name, chunk_id)
    embeddings = data["embeddings"]
    print("Эмбеддинги загружены из кэша")
else:
    embeddings = generator.encode_batch(texts)
    cache.save_embeddings(generator.model_name, chunk_id, embeddings, {"texts": texts})
    print("Эмбеддинги сгенерированы и сохранены в кэш")

print(f"Форма эмбеддингов: {embeddings.shape}")
```

#### 6.3.4. Использование GPU для ускорения

Класс `EmbeddingGenerator` автоматически определяет доступное устройство:

- `cuda` — если есть NVIDIA GPU с поддержкой CUDA
- `mps` — для macOS с Metal (если доступно)
- `cpu` — во всех остальных случаях

```python
def get_best_device():
    import torch
    if torch.cuda.is_available():
        return "cuda"
    elif torch.backends.mps.is_available():
        return "mps"
    else:
        return "cpu"

print(f"Используется устройство: {get_best_device()}")
generator = EmbeddingGenerator(device=get_best_device())
```

#### 6.3.5. Как выбрать модель в зависимости от задачи

| Сценарий | Рекомендуемая модель | Размерность | Причина |
|----------|----------------------|-------------|---------|
| **Быстрый прототип** | all-MiniLM-L6-v2 | 384 | Максимальная скорость, минимальные требования |
| **Высокая точность (англ.)** | BAAI/bge-large-en-v1.5 | 1024 | Лучшее качество на MTEB |
| **Мультиязычный проект** | intfloat/multilingual-e5-large | 1024 | Поддержка 100+ языков, высокое качество |
| **Русский язык** | DeepPavlov/rubert-tiny | 312 | Специализирована на русский, быстро |
| **Облачное решение** | text-embedding-3-small | 1536 | Хорошее качество, низкая цена |

---

### 6.4. Оценка качества эмбеддингов

#### 6.4.1. Бенчмарк STS (Semantic Textual Similarity)

STS — это набор задач, где оценивается, насколько косинусное сходство эмбеддингов коррелирует с оценками людей (от 0 до 5) для пар предложений. Чем выше корреляция (Pearson или Spearman), тем лучше модель.

#### 6.4.2. MTEB (Massive Text Embedding Benchmark)

MTEB — это современный стандарт для оценки эмбеддингов. Он включает 58 задач из 8 категорий:

- Классификация
- Кластеризация
- Поиск (Retrieval)
- Семантическое сходство (STS)
- Суммаризация
- Переранжирование (Reranking)
- Бинарная классификация
- Поиск по параграфам

**Как интерпретировать результаты:** модели с MTEB‑score > 60 считаются хорошими, > 63 — отличными, > 65 — выдающимися.

#### 6.4.3. Тестирование на реальных данных

Для вашей конкретной задачи лучше всего провести A/B‑тест:

1. Выберите 2–3 модели эмбеддингов.
2. Сгенерируйте эмбеддинги для всех чанков.
3. Для 50–100 тестовых запросов выполните поиск и оцените качество (например, оценивая релевантность топ‑5 результатов вручную или с помощью LLM‑as‑a‑judge).
4. Сравните метрики (Recall@5, MRR) и скорость.

#### 6.4.4. Влияние L2‑нормализации

L2‑нормализация приводит длину вектора к 1:

$$
\hat{\mathbf{v}} = \frac{\mathbf{v}}{\|\mathbf{v}\|_2}
$$

После нормализации косинусное сходство становится эквивалентным скалярному произведению, что ускоряет поиск в некоторых библиотеках (FAISS). Большинство современных моделей уже выдают нормализованные векторы, если указать параметр `normalize_embeddings=True`.

---

### 6.5. Эксперимент: сравнение моделей эмбеддингов

В этом эксперименте мы используем данные из раздела 5 (`chunks_data.json`) для сравнения двух моделей.

```python
import json
import time
import numpy as np
from typing import List, Dict
from sentence_transformers import SentenceTransformer

# Используем классы, определённые выше
from embedding_generator import EmbeddingGenerator, EmbeddingCache

def load_chunks(json_path: str = "./chunks_data.json") -> List[Dict]:
    """Загружает чанки из JSON-файла (результат раздела 5)."""
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data

def generate_embeddings_for_chunks(chunks: List[Dict], model_name: str, cache: EmbeddingCache) -> np.ndarray:
    """Генерирует эмбеддинги для чанков с использованием кэша."""
    generator = EmbeddingGenerator(model_name)
    all_embeddings = []
    for i, chunk in enumerate(chunks):
        chunk_id = i
        if cache.exists(model_name, chunk_id):
            data = cache.load_embeddings(model_name, chunk_id)
            all_embeddings.append(data["embeddings"])
        else:
            text = chunk["text"]
            embedding = generator.encode_batch([text], batch_size=1)
            cache.save_embeddings(model_name, chunk_id, embedding, {"source": chunk.get("metadata", {}).get("source", "unknown")})
            all_embeddings.append(embedding)
    return np.vstack(all_embeddings)

def compute_cosine_similarity(query_emb: np.ndarray, doc_embs: np.ndarray) -> np.ndarray:
    """Вычисляет косинусное сходство между запросом и всеми документами."""
    # Если векторы нормализованы, косинусное сходство = скалярное произведение
    return np.dot(doc_embs, query_emb)

def search(query: str, generator: EmbeddingGenerator, doc_embs: np.ndarray,
           chunks: List[Dict], top_k: int = 5) -> List[Dict]:
    """Выполняет поиск по запросу и возвращает топ-k чанков."""
    query_emb = generator.encode_batch([query], batch_size=1)[0]
    scores = compute_cosine_similarity(query_emb, doc_embs)
    top_indices = np.argsort(scores)[::-1][:top_k]
    results = []
    for idx in top_indices:
        results.append({
            "text": chunks[idx]["text"],
            "score": float(scores[idx]),
            "metadata": chunks[idx].get("metadata", {})
        })
    return results

# --- Основной эксперимент ---
def compare_models(chunks_file: str = "./chunks_data.json"):
    chunks = load_chunks(chunks_file)
    print(f"Загружено {len(chunks)} чанков")

    # Список моделей для сравнения
    models = [
        ("all-MiniLM-L6-v2", "MiniLM (384d)"),
        ("BAAI/bge-base-en-v1.5", "BGE-base (768d)"),
    ]

    test_queries = [
        "налог на прибыль",
        "какие налоги платят самозанятые",
        "отчётность для ИП",
    ]

    results = {}
    cache = EmbeddingCache("./embedding_cache")

    for model_name, label in models:
        print(f"\n{'='*60}")
        print(f"Модель: {label}")
        print(f"{'='*60}")

        # Генерация эмбеддингов
        start_time = time.time()
        embeddings = generate_embeddings_for_chunks(chunks, model_name, cache)
        elapsed_time = time.time() - start_time

        print(f"Время генерации (с учётом кэша): {elapsed_time:.2f} сек")
        print(f"Размерность: {embeddings.shape[1]}")

        generator = EmbeddingGenerator(model_name)

        # Поиск по запросам
        for query in test_queries:
            print(f"\nЗапрос: {query}")
            top_chunks = search(query, generator, embeddings, chunks, top_k=3)
            for i, chunk in enumerate(top_chunks, 1):
                text_preview = chunk["text"][:100].replace("\n", " ")
                print(f"  {i}. score={chunk['score']:.4f}: {text_preview}...")

        results[label] = {
            "time": elapsed_time,
            "dimension": embeddings.shape[1]
        }

    return results

if __name__ == "__main__":
    # Предварительно убедитесь, что chunks_data.json существует (результат раздела 5)
    results = compare_models("./chunks_data.json")

    print("\n" + "="*60)
    print("ИТОГОВОЕ СРАВНЕНИЕ МОДЕЛЕЙ")
    print("="*60)
    for label, data in results.items():
        print(f"{label}: размерность={data['dimension']}, время={data['time']:.2f} сек")
```

**Пример вывода:**

```
Загружено 26 чанков

============================================================
Модель: MiniLM (384d)
============================================================
Время генерации (с учётом кэша): 0.45 сек
Размерность: 384

Запрос: налог на прибыль
  1. score=0.8234: Налог на прибыль организаций регулируется главой 25 НК РФ...
  2. score=0.7012: Ставка налога на прибыль составляет 20%...
  3. score=0.5421: Особенности расчёта для ИТ-компаний...

============================================================
Модель: BGE-base (768d)
============================================================
Время генерации (с учётом кэша): 1.23 сек
Размерность: 768

Запрос: налог на прибыль
  1. score=0.8912: Налог на прибыль организаций регулируется главой 25 НК РФ...
  2. score=0.7834: Ставка налога на прибыль составляет 20%...
  3. score=0.6011: Особенности расчёта для ИТ-компаний...
```

**Вывод:** BGE-base показывает более высокие оценки релевантности (score), но требует больше времени на генерацию эмбеддингов. MiniLM быстрее и подходит для прототипов, BGE — для продакшена, где качество важнее.

---

### 6.6. Контрольные вопросы

1. *В чём разница между косинусным сходством и евклидовым расстоянием, и когда какую метрику использовать?*  
   **Ответ:** Косинусное сходство измеряет угол между векторами и не зависит от их длины, поэтому оно лучше подходит для сравнения семантики текстов разной длины. Евклидово расстояние чувствительно к длине векторов и хорошо работает, когда важна абсолютная разница. Для нормализованных векторов эти метрики эквивалентны.

2. *Почему BGE-large показывает лучшее качество, чем MiniLM, на бенчмарке MTEB?*  
   **Ответ:** BGE-large имеет большую размерность (1024 против 384), что позволяет кодировать больше семантических нюансов. Кроме того, BGE обучался на более разнообразных данных с использованием продвинутых техник (contrastive learning, hard negative mining).

3. *Когда стоит использовать мультиязычные модели вместо специализированных русскоязычных?*  
   **Ответ:** Мультиязычные модели (например, E5, BGE-M3) показывают хорошие результаты на русском языке и поддерживают множество других языков. Их стоит использовать, если в корпусе есть документы на разных языках или если вы планируете масштабировать систему. Специализированные русские модели могут дать небольшой прирост качества, если корпус полностью на русском.

---

### 6.7. Задания

1. **Сравнение трёх моделей.** Расширьте эксперимент из раздела 6.5, добавив третью модель (например, `intfloat/multilingual-e5-base`). Сравните время генерации, размерность и качество поиска по 5 запросам. Постройте графики зависимости времени от размера батча.

2. **Класс для кэширования эмбеддингов.** Модифицируйте класс `EmbeddingCache`, чтобы он поддерживал:
   - Хранение в формате Parquet для больших данных.
   - Инкрементальное обновление (добавление новых документов без пересчёта старых).
   - Проверку целостности (хеш модели и версия данных).

3. **Визуализация эмбеддингов.** Выберите 500 случайных чанков из вашего датасета, сгенерируйте для них эмбеддинги с помощью любой модели и визуализируйте их в 2D с помощью UMAP. Покрасьте точки по темам (если есть метаданные с категориями). Опишите, что вы видите.

---

### 6.8. Список литературы

1. **Reimers, N., & Gurevych, I. (2019).** *Sentence-BERT: Sentence Embeddings using Siamese BERT-Networks*. – arXiv:1908.10084. Оригинальная статья о Sentence‑Transformers.
2. **MTEB Leaderboard.** – https://huggingface.co/spaces/mteb/leaderboard – результаты всех моделей на бенчмарке MTEB.
3. **BGE Models Documentation.** – https://huggingface.co/BAAI – модели от Пекинской академии.
4. **OpenAI Embeddings Documentation.** – https://platform.openai.com/docs/guides/embeddings – описание платных моделей.
5. **Sentence‑Transformers Documentation.** – https://www.sbert.net/ – официальная документация.
6. **Wang, L., et al. (2023).** *A Survey on Sentence Embeddings: Models and Applications*. – обзор современных подходов.



## Тема 7. Векторные базы данных

После того как мы сгенерировали эмбеддинги для всех чанков (тема 6), нам нужно сохранить их в специализированном хранилище, которое позволит выполнять быстрый поиск по запросу. Эта задача нетривиальна: при миллионах векторов размерностью 768+ простой линейный поиск (сканирование всех векторов) становится непозволительно медленным. Здесь на помощь приходят **векторные базы данных (Vector Databases)** — специализированные хранилища, оптимизированные для хранения и поиска многомерных векторов с использованием алгоритмов приближённого поиска ближайших соседей (ANN). В этой теме мы разберём, что такое векторные БД, как они устроены, сравним популярные решения и покажем практический код для работы с каждой из них, используя данные из предыдущих разделов.

---

### 7.1. Что такое векторная база данных

#### 7.1.1. Определение и назначение

**Векторная база данных (Vector Database)** — это система управления данными, предназначенная для хранения и эффективного поиска многомерных векторов (эмбеддингов). В отличие от реляционных БД, которые оптимизированы для точных запросов по структурированным данным (WHERE id = 5), векторные БД оптимизированы для поиска ближайших соседей (Nearest Neighbor Search): "найди 10 векторов, наиболее похожих на данный вектор".

#### 7.1.2. Отличие от реляционных и NoSQL БД

| Характеристика | Реляционные БД (PostgreSQL) | NoSQL (MongoDB) | Векторные БД (Qdrant, Milvus) |
|----------------|-----------------------------|-----------------|-------------------------------|
| **Тип данных** | Структурированные (таблицы) | Документы/ключ-значение | Векторы + метаданные |
| **Запросы** | Точные (SQL) | Гибкие (JSON) | Поиск по сходству (ANN) |
| **Индексы** | B-tree, Hash | Вторичные индексы | HNSW, IVF, PQ |
| **Скорость поиска по сходству** | Медленно (линейное сканирование) | Не поддерживается | Миллисекунды для миллионов векторов |
| **Фильтрация** | Мощная (WHERE, JOIN) | Ограниченная | Поддерживается (pre/post filtering) |

#### 7.1.3. Механизмы индексации (ANN-алгоритмы)

Алгоритмы приближённого поиска ближайших соседей (Approximate Nearest Neighbors, ANN) жертвуют небольшой точностью (recall) ради колоссального ускорения (в 100–1000 раз). Рассмотрим три основных подхода.

**HNSW (Hierarchical Navigable Small World)**

Графовый алгоритм, который строит многослойную иерархию графов малого мира.

```
[Верхний слой]   •——•——•         (очень разреженный)
                   |  |
[Средний слой]  •——•——•——•      (средняя плотность)
                   |  |  |
[Нижний слой]   •——•——•——•——•  (все векторы, плотный)

Поиск: начинается с верхнего слоя, затем спускается вниз, каждый раз находя ближайшие узлы.
```

**Математическая суть:** HNSW строит граф, где каждый узел соединён с несколькими ближайшими соседями (параметр `M`). При поиске алгоритм начинает с верхнего (самого разреженного) слоя и последовательно спускается вниз, используя жадный поиск на каждом слое.

**Параметры HNSW:**
- `M` — количество соединений на узел (обычно 16–64). Чем больше, тем выше точность, но больше памяти.
- `ef_construction` — размер динамического списка при построении (обычно 100–400). Чем больше, тем дольше построение, но выше качество индекса.
- `ef_search` — размер динамического списка при поиске (обычно 50–200). Чем больше, тем выше точность, но медленнее поиск.

**IVF (Inverted File Index)**

Кластеризационный подход: все векторы разбиваются на кластеры (с помощью k-means), затем для каждого кластера строится инвертированный список.

```
Все векторы → k-means → кластер 1, кластер 2, ..., кластер n
                              ↓
Поиск: определяем ближайшие кластеры (nprobe), ищем только внутри них.
```

**Параметры IVF:**
- `nlist` — количество кластеров (обычно 100–1000 для небольших баз, 1000–10000 для больших).
- `nprobe` — количество кластеров, в которых выполняется поиск (обычно 1–10). Чем больше, тем выше точность, но медленнее.

**PQ (Product Quantization)**

Сжатие векторов: вектор разбивается на подвекторы, каждый подвектор квантуется в ограниченный набор центроид. Это уменьшает память в 4–16 раз.

```
Вектор размерности 128: [a1, a2, ..., a128]
    ↓
Разбиение на 4 подвектора по 32 элемента: [a1..a32], [a33..a64], [a65..a96], [a97..a128]
    ↓
Каждый подвектор заменяется индексом ближайшей центроиды (например, из 256)
    ↓
Вместо 128 float (512 байт) храним 4 int (16 байт) — сжатие в 32 раза!
```

**Комбинации:** часто используют IVF + PQ (IVFPQ) для масштабирования до миллиардов векторов.

---

### 7.2. Обзор популярных векторных БД

| БД | Описание | Ключевые особенности | Когда использовать |
|----|----------|---------------------|-------------------|
| **Chroma** | Легковесная, встраиваемая БД на Python | • Простая установка (pip)<br>• Встроенные эмбеддинги (опционально)<br>• Поддержка метаданных и фильтрации<br>• Персистентное хранение | Прототипы, небольшие проекты, быстрое начало |
| **FAISS** | Библиотека от Meta, не является полноценной БД | • Высокая производительность (C++)<br>• Множество индексов (HNSW, IVF, PQ)<br>• Поддержка GPU<br>• Нет встроенного хранения метаданных | Высоконагруженные системы, кастомные решения |
| **Pinecone** | Облачное решение (SaaS) | • Полностью управляемый сервис<br>• Автоматическое масштабирование<br>• Высокая доступность<br>• Платный | Корпоративные системы, когда нет DevOps |
| **Weaviate** | Open‑source с гибридным поиском | • Встроенный векторный поиск + BM25<br>• Графовый слой (GraphQL)<br>• Модульная архитектура<br>• Автоматическое индексирование | Проекты, где нужен гибридный поиск (лексика + семантика) |
| **Milvus** | Высокомасштабируемая распределённая БД | • Поддержка миллиардов векторов<br>• GPU ускорение<br>• Высокая доступность<br>• Шардирование | Крупные корпоративные системы с миллиардами документов |
| **Qdrant** | Современная БД с rich API | • Поддержка фильтрации по метаданным<br>• Инкрементальные обновления<br>• Payload (метаданные) хранятся вместе с векторами<br>• Отличная документация | Продакшен-системы с частыми обновлениями |
| **LanceDB** | БД на основе Lance (columnar) | • Хранение на диске с быстрым доступом<br>• Нет ограничений на размер (работает с SSD)<br>• Интеграция с Pandas и PyArrow<br>• Нулевая задержка при запуске | Проекты с ограниченной оперативной памятью, большие датасеты |
| **PgVector** | Расширение PostgreSQL | • Интеграция с существующей SQL-базой<br>• Поддержка ACID<br>• Индексы (HNSW, IVF)<br>• Стандартный SQL | Проекты, уже использующие PostgreSQL, где важна согласованность данных |

---

### 7.3. Сравнительная таблица векторных БД

| БД | Лицензия | Скорость поиска | Фильтрация | Инкремент. обновления | Сложность внедрения | Масштабируемость |
|----|----------|-----------------|------------|----------------------|---------------------|------------------|
| **Chroma** | Apache 2.0 | Средняя | Да | Да | Низкая | До 1 млн векторов |
| **FAISS** | MIT | Очень высокая | Нет (только через ID) | Ограниченно | Средняя | Миллиарды векторов |
| **Pinecone** | Платная | Высокая | Да | Да | Низкая (SaaS) | Миллиарды векторов |
| **Weaviate** | BSD-3 | Высокая | Да | Да | Средняя | Миллиарды векторов |
| **Milvus** | Apache 2.0 | Очень высокая | Да | Да | Высокая | Миллиарды векторов |
| **Qdrant** | Apache 2.0 | Высокая | Да | Да | Средняя | Миллиарды векторов |
| **LanceDB** | Apache 2.0 | Высокая | Да | Да | Низкая | Миллиарды векторов |
| **PgVector** | PostgreSQL | Средняя | Да | Да | Средняя (SQL) | До 10 млн векторов |

---

### 7.4. Практика работы с Chroma

Chroma — идеальный выбор для прототипирования и небольших проектов. Он предоставляет простой Python API и хранит данные на диске.

#### Установка

```python
!pip install chromadb sentence-transformers
```

#### Полный код для работы с Chroma

```python
# ================================================================
# Chroma: Простая встраиваемая БД
# ================================================================

import chromadb
from chromadb.config import Settings
import json
from sentence_transformers import SentenceTransformer

def load_documents(json_path: str = "./extracted_data.json"):
    with open(json_path, "r", encoding="utf-8") as f:
        return json.load(f)

def clean_metadata(metadata: dict) -> dict:
    """Очищает метаданные от None и пустых значений."""
    cleaned = {}
    for key, value in metadata.items():
        if isinstance(value, list):
            if len(value) == 0:
                continue
            value = [v for v in value if v is not None]
            if len(value) == 0:
                continue
        elif value is None or value == "":
            continue
        cleaned[key] = value
    return cleaned

# 1. Инициализация
client = chromadb.PersistentClient(
    path="./chroma_db",
    settings=Settings(anonymized_telemetry=False)
)

collection = client.get_or_create_collection(
    name="documents",
    metadata={"hnsw:space": "cosine"}
)

# 2. Загрузка данных
documents = load_documents("./extracted_data.json")

texts = [doc["text"] for doc in documents]
metadatas = [clean_metadata(doc["metadata"]) for doc in documents]
ids = [f"doc_{i:04d}" for i in range(len(documents))]

# 3. Генерация эмбеддингов
model = SentenceTransformer("cointegrated/rubert-tiny2")
embeddings = model.encode(texts, normalize_embeddings=True, show_progress_bar=True).tolist()

# 4. Добавление
collection.add(documents=texts, embeddings=embeddings, metadatas=metadatas, ids=ids)
print(f"✅ Добавлено {len(documents)} документов.")

# 5. Поиск БЕЗ ФИЛЬТРА
query = "налог на прибыль"
query_emb = model.encode([query], normalize_embeddings=True).tolist()

results = collection.query(
    query_embeddings=query_emb,
    n_results=3
)

print(f"\n🔍 Найдено: {len(results['documents'][0])} документов")
for doc, meta in zip(results['documents'][0], results['metadatas'][0]):
    print(f"  - {doc[:150]}... (источник: {meta.get('source', 'unknown')})")

# 6. Поиск С ФИЛЬТРОМ
filtered_results = collection.query(
    query_embeddings=query_emb,
    n_results=3,
    where={"source": {"$contains": "report"}}  # фильтр по части имени
)

print(f"\n🔍 С фильтром по источнику:")
for doc, meta in zip(filtered_results['documents'][0], filtered_results['metadatas'][0]):
    print(f"  - {doc[:150]}... (источник: {meta.get('source', 'unknown')})")
```

---

### 7.5. Практика работы с FAISS

FAISS — это высокопроизводительная библиотека от Meta, но она не является полноценной БД: метаданные нужно хранить отдельно.

#### Установка

```python
!pip install faiss-cpu sentence-transformers
```

#### Полный код для FAISS

```python
# ================================================================
# FAISS: максимальная скорость, метаданные отдельно
# ================================================================

import faiss
import numpy as np
import json
from sentence_transformers import SentenceTransformer

def load_documents(json_path="./extracted_data.json"):
    with open(json_path, "r", encoding="utf-8") as f:
        return json.load(f)

# Загрузка данных
docs = load_documents()
texts = [d["text"] for d in docs]
metadatas = [d["metadata"] for d in docs]

if not texts:
    print("Нет документов для индексации")
    exit()

model = SentenceTransformer("cointegrated/rubert-tiny2")
embeddings = model.encode(texts, normalize_embeddings=True).astype('float32')
dim = embeddings.shape[1]
n_vectors = len(embeddings)

# Выбор типа индекса в зависимости от количества векторов
if n_vectors < 100:
    # Для малого числа документов используем точный поиск (Flat)
    index = faiss.IndexFlatIP(dim)  # IP = Inner Product (для нормализованных векторов даёт косинусное сходство)
    index.add(embeddings)
    print(f"Используется IndexFlatIP (точный поиск) для {n_vectors} векторов")
else:
    # Для больших данных – IVF
    nlist = min(100, max(1, n_vectors // 10))
    quantizer = faiss.IndexFlatIP(dim)
    index = faiss.IndexIVFFlat(quantizer, dim, nlist)
    index.train(embeddings)
    index.add(embeddings)
    index.nprobe = min(5, nlist)
    print(f"Используется IndexIVFFlat с nlist={nlist}, nprobe={index.nprobe}")

# Поиск
query = "налог на прибыль"
query_emb = model.encode([query], normalize_embeddings=True).astype('float32')

distances, indices = index.search(query_emb, min(3, n_vectors))

print("Результаты поиска FAISS:")
for idx, dist in zip(indices[0], distances[0]):
    if idx >= 0 and idx < len(texts):
        print(f"  - {metadatas[idx].get('source', 'unknown')}: {texts[idx][:150]}... (сходство: {dist:.4f})")
    else:
        print(f"  - индекс {idx} вне диапазона")
```

---

### 7.6. Практика работы с Pinecone (требуется API-ключ)

Pinecone — облачное решение. **Вам потребуется API-ключ**, который можно получить на сайте [pinecone.io](https://www.pinecone.io/).

#### Установка

```python
!pip install pinecone sentence-transformers
```

#### Получение API-ключа

1. Зарегистрируйтесь на [pinecone.io](https://www.pinecone.io/).
2. Создайте API-ключ в разделе "API Keys".
3. Добавьте ключ в Google Colab Secrets (🔑) как `PINECONE_API_KEY`.

#### Полный код для Pinecone

```python
# ================================================================
# PINECONE – облачный сервис (требуется API-ключ)
# ================================================================

import json
import time
from sentence_transformers import SentenceTransformer
import pinecone
from google.colab import userdata

# 1. Загрузка документов
def load_documents(json_path="./extracted_data.json"):
    with open(json_path, "r", encoding="utf-8") as f:
        return json.load(f)

# 2. Инициализация Pinecone
PINECONE_API_KEY = userdata.get('PINECONE_API_KEY')
pc = pinecone.Pinecone(api_key=PINECONE_API_KEY)

# 3. Параметры индекса
index_name = "rag-docs"
dimension = 312
metric = "cosine"
cloud = "aws"
region = "us-east-1"  # Стандартный регион для новых аккаунтов

# 4. Создание индекса (если не существует)
existing_indexes = [idx.name for idx in pc.list_indexes()]
if index_name not in existing_indexes:
    print(f"Создание индекса {index_name} в регионе {region}...")
    pc.create_index(
        name=index_name,
        dimension=dimension,
        metric=metric,
        spec=pinecone.ServerlessSpec(cloud=cloud, region=region)
    )
    # Ожидание готовности (до 2 минут)
    while not pc.describe_index(index_name).status['ready']:
        time.sleep(10)
        print("Ожидание готовности индекса...")
    print("✅ Индекс готов.")
else:
    print(f"ℹ️ Индекс {index_name} уже существует.")

index = pc.Index(index_name)

# 5. Загрузка данных и вставка
docs = load_documents("./extracted_data.json")
model = SentenceTransformer("cointegrated/rubert-tiny2")

batch_size = 100
total = len(docs)
for i in range(0, total, batch_size):
    batch = docs[i:i+batch_size]
    texts = [d["text"] for d in batch]
    emb = model.encode(texts, normalize_embeddings=True).tolist()
    metas = [d["metadata"] for d in batch]
    ids = [f"doc_{i+j}" for j in range(len(batch))]
    index.upsert(vectors=list(zip(ids, emb, metas)))
    print(f"Добавлено {len(batch)} из {total}")

print(f"✅ Всего добавлено {total} документов.")

# 6. Поиск
query = "налог на прибыль"
qe = model.encode([query], normalize_embeddings=True).tolist()
res = index.query(vector=qe, top_k=3, include_metadata=True)

print("\n🔍 Результаты поиска:")
if res['matches']:
    for match in res['matches']:
        src = match['metadata'].get('source', 'unknown')
        print(f"  - {src}: сходство {match['score']:.4f}")
else:
    print("  Ничего не найдено.")
```

---

### 7.7. Практика работы с Weaviate (требуется API-ключ)

Weaviate поддерживает гибридный поиск (BM25 + векторы). **Вам потребуется API-ключ** от облачного экземпляра Weaviate или локальный сервер.

#### Установка

```python
!pip install weaviate-client sentence-transformers
```

#### Получение API-ключа (для облачной версии)

1. Зарегистрируйтесь на [weaviate.io](https://weaviate.io/).
2. Создайте облачный кластер (бесплатный сэндвис).
3. Получите URL кластера и API-ключ.
4. Добавьте их в Google Colab Secrets (🔑) как:
   - `WEAVIATE_URL` = `https://your-cluster.weaviate.cloud`
   - `WEAVIATE_API_KEY` = `your-secret-key`

#### Полный код для Weaviate

```python
# ================================================================
# Weaviate – гибридный поиск (BM25 + вектор)
# ================================================================

import json
from sentence_transformers import SentenceTransformer
import weaviate
from weaviate.classes.config import Configure, Property, DataType
from weaviate.classes.query import MetadataQuery
from weaviate.classes.init import Auth
from google.colab import userdata

# ---- Загрузка секретов ----
WEAVIATE_URL = userdata.get('WEAVIATE_URL')
WEAVIATE_API_KEY = userdata.get('WEAVIATE_API_KEY')

if not WEAVIATE_URL or not WEAVIATE_API_KEY:
    raise ValueError(
        "❌ Секреты не найдены!\n"
        "Добавьте их в панели 🔑 Secrets:\n"
        "  - WEAVIATE_URL = https://your-cluster.weaviate.cloud\n"
        "  - WEAVIATE_API_KEY = ваш_секретный_ключ"
    )

# ---- Загрузка документов ----
def load_documents(json_path="./extracted_data.json"):
    with open(json_path, "r", encoding="utf-8") as f:
        return json.load(f)

# ---- Подключение ----
client = weaviate.connect_to_weaviate_cloud(
    cluster_url=WEAVIATE_URL,
    auth_credentials=Auth.api_key(WEAVIATE_API_KEY),
)

# ---- Создание коллекции ----
if client.collections.exists("Document"):
    client.collections.delete("Document")

collection = client.collections.create(
    name="Document",
    properties=[
        Property(name="text", data_type=DataType.TEXT),
        Property(name="source", data_type=DataType.TEXT),
        Property(name="author", data_type=DataType.TEXT),
    ],
    vectorizer_config=Configure.Vectorizer.none(),
)

# ---- Загрузка и вставка ----
docs = load_documents("./extracted_data.json")
model = SentenceTransformer("cointegrated/rubert-tiny2")

with collection.batch.fixed_size(batch_size=100) as batch:
    for doc in docs:
        text = doc["text"]
        emb = model.encode(text, normalize_embeddings=True).tolist()
        properties = {
            "text": text,
            "source": doc["metadata"].get("source", ""),
            "author": doc["metadata"].get("author", ""),
        }
        batch.add_object(properties=properties, vector=emb)

print(f"✅ Добавлено {len(docs)} документов.")

# ---- Векторный поиск ----
query = "налог на прибыль"
qe = model.encode(query, normalize_embeddings=True).tolist()

vector_results = collection.query.near_vector(
    near_vector=qe,
    distance=0.5,
    limit=3,
    return_properties=["text", "source"],
    return_metadata=MetadataQuery(distance=True)
)

print("\n🔍 Weaviate (векторный поиск):")
for obj in vector_results.objects:
    print(f"  - {obj.properties['source']}: {obj.properties['text'][:150]}... (distance: {obj.metadata.distance:.4f})")

# ---- Гибридный поиск ----
hybrid_results = collection.query.hybrid(
    query=query,
    alpha=0.5,  # баланс: 0 = только векторы, 1 = только BM25
    limit=3,
    return_properties=["text", "source"],
    return_metadata=MetadataQuery(score=True)
)

print("\n🔍 Weaviate (гибридный поиск):")
for obj in hybrid_results.objects:
    print(f"  - {obj.properties['source']}: {obj.properties['text'][:150]}... (score: {obj.metadata.score:.4f})")

client.close()
```

---

### 7.8. Практика работы с Qdrant

Qdrant работает локально и сохраняет данные на диск.

#### Установка

```python
!pip install qdrant-client sentence-transformers
```

#### Полный код для Qdrant

```python
# ================================================================
# Qdrant – гибкая фильтрация payload (локальный режим)
# ================================================================

import json
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct, Filter, FieldCondition, MatchValue

# ---- Загрузка данных ----
def load_documents(json_path="./extracted_data.json"):
    with open(json_path, "r", encoding="utf-8") as f:
        return json.load(f)

# ---- Подключение к локальной БД (хранилище на диске) ----
client = QdrantClient(path="./qdrant_data")  # данные сохранятся в папку qdrant_data

collection_name = "documents"

# ---- Пересоздание коллекции ----
client.recreate_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(size=312, distance=Distance.COSINE)
)

# ---- Загрузка документов ----
docs = load_documents("./extracted_data.json")
model = SentenceTransformer("cointegrated/rubert-tiny2")

# ---- Подготовка точек ----
points = []
for i, doc in enumerate(docs):
    emb = model.encode(doc["text"], normalize_embeddings=True).tolist()
    points.append(PointStruct(
        id=i,
        vector=emb,
        payload={
            "text": doc["text"],
            "source": doc["metadata"].get("source", ""),
            "author": doc["metadata"].get("author", ""),
        }
    ))

# ---- Вставка данных ----
client.upsert(collection_name=collection_name, points=points)
print(f"✅ Добавлено {len(points)} документов.")

# ---- Поиск без фильтра ----
query = "налог на прибыль"
qe = model.encode(query, normalize_embeddings=True).tolist()

results = client.search(
    collection_name=collection_name,
    query_vector=qe,
    limit=3,
    with_payload=True,
)

print("\n🔍 Qdrant (поиск без фильтра):")
for hit in results:
    print(f"  - {hit.payload['source']}: {hit.payload['text'][:150]}... (score: {hit.score:.4f})")

# ---- Поиск с фильтром ----
filtered_results = client.search(
    collection_name=collection_name,
    query_vector=qe,
    limit=3,
    query_filter=Filter(
        must=[
            FieldCondition(
                key="source",
                match=MatchValue(value="report_2024.pdf")  # замените на существующий файл
            )
        ]
    ),
    with_payload=True,
)

print("\n🔍 Qdrant (с фильтром по источнику):")
for hit in filtered_results:
    print(f"  - {hit.payload['source']}: {hit.payload['text'][:150]}... (score: {hit.score:.4f})")
```

---

### 7.9. Практика работы с Milvus (требуется локальный сервер)

Milvus — распределённая БД для миллиардов векторов. **Требуется запущенный сервер Milvus** (локально или в Docker).

#### Запуск Milvus (Docker)

```bash
docker run -d --name milvus-standalone -p 19530:19530 -p 9091:9091 milvusdb/milvus:latest
```

#### Установка

```python
!pip install pymilvus sentence-transformers
```

#### Полный код для Milvus

```python
# ================================================================
# Milvus: распределённый поиск (требуется сервер)
# ================================================================

from pymilvus import connections, Collection, CollectionSchema, FieldSchema, DataType, utility
from sentence_transformers import SentenceTransformer
import json

def load_documents(json_path="./extracted_data.json"):
    with open(json_path, "r", encoding="utf-8") as f:
        return json.load(f)

# 1. Подключение к серверу Milvus (локальному)
connections.connect("default", host="localhost", port="19530")
print("✅ Подключено к Milvus")

collection_name = "documents"

# 2. Удаление старой коллекции (если есть)
if utility.has_collection(collection_name):
    Collection(collection_name).drop()

# 3. Создание схемы
fields = [
    FieldSchema("id", DataType.INT64, is_primary=True, auto_id=True),
    FieldSchema("text", DataType.VARCHAR, max_length=65535),
    FieldSchema("source", DataType.VARCHAR, max_length=255),
    FieldSchema("embedding", DataType.FLOAT_VECTOR, dim=312),
]
schema = CollectionSchema(fields)
collection = Collection(collection_name, schema)

# 4. Загрузка данных
docs = load_documents("./extracted_data.json")
model = SentenceTransformer("cointegrated/rubert-tiny2")

data = []
for doc in docs:
    emb = model.encode(doc["text"], normalize_embeddings=True).tolist()
    data.append({
        "text": doc["text"],
        "source": doc["metadata"].get("source", ""),
        "embedding": emb,
    })

# 5. Вставка данных
collection.insert(data)
print(f"✅ Добавлено {len(docs)} документов.")

# 6. Создание индекса (обязательно для быстрого поиска)
collection.create_index(
    "embedding",
    {"metric_type": "COSINE", "index_type": "IVF_FLAT", "params": {"nlist": 128}}
)

# 7. Загрузка коллекции в память (обязательно перед поиском)
collection.load()

# 8. Поиск
query = "налог на прибыль"
qe = model.encode(query, normalize_embeddings=True).tolist()

results = collection.search(
    [qe],
    "embedding",
    {"metric_type": "COSINE", "params": {"nprobe": 10}},
    limit=3,
    output_fields=["text", "source"]
)

print("\n🔍 Milvus (векторный поиск):")
for hits in results:
    for hit in hits:
        text = hit.entity.get("text")
        source = hit.entity.get("source")
        print(f"  - {source}: {text[:150]}... (distance: {hit.distance:.4f})")
```

---

### 7.10. Практика работы с LanceDB (экономия RAM)

LanceDB хранит данные на диске и не загружает весь индекс в память.

#### Установка

```python
!pip install lancedb pandas sentence-transformers
```

#### Полный код для LanceDB

```python
# ================================================================
# LanceDB: данные на диске, индекс не в памяти
# ================================================================

import lancedb
import pandas as pd
from sentence_transformers import SentenceTransformer
import json

def load_documents(json_path="./extracted_data.json"):
    with open(json_path, "r", encoding="utf-8") as f:
        return json.load(f)

# 1. Подключение
db = lancedb.connect("./lancedb_data")

# 2. Загрузка данных
docs = load_documents("./extracted_data.json")
model = SentenceTransformer("cointegrated/rubert-tiny2")

data = []
for doc in docs:
    emb = model.encode(doc["text"], normalize_embeddings=True).tolist()
    data.append({
        "text": doc["text"],
        "source": doc["metadata"].get("source", ""),
        "vector": emb,
    })

df = pd.DataFrame(data)

# 3. Создание таблицы
table = db.create_table("documents", data=df, mode="overwrite")

# 4. Создание индекса (только если > 256 документов)
if len(docs) >= 256:
    table.create_index(
        metric="cosine",
        index_type="IVF_PQ",
        num_partitions=10,
        num_sub_vectors=16
    )
    print(f"✅ Создан индекс IVF_PQ для {len(docs)} документов")
else:
    print(f"ℹ️ Для {len(docs)} документов используется flat-поиск (без индекса)")

# 5. Поиск
query = "налог на прибыль"
qe = model.encode(query, normalize_embeddings=True).tolist()

results = table.search(qe).metric("cosine").limit(3).to_pandas()

print("\n🔍 LanceDB (поиск с диска):")
for _, row in results.iterrows():
    print(f"  - {row['source']}: {row['text'][:150]}... (distance: {row['_distance']:.4f})")
```

---

### 7.11. Практика работы с PgVector (требуется PostgreSQL)

PgVector — расширение PostgreSQL для векторного поиска. **Требуется установленный PostgreSQL с расширением pgvector**.

#### Установка PostgreSQL с pgvector (Docker)

```bash
docker run -d \
  --name postgres-pgvector \
  -e POSTGRES_PASSWORD=password \
  -e POSTGRES_DB=postgres \
  -p 5432:5432 \
  pgvector/pgvector:latest
```

#### Установка

```python
!pip install psycopg2-binary sentence-transformers pgvector
```

#### Полный код для PgVector

```python
# ================================================================
# PgVector: векторный поиск внутри PostgreSQL (требуется сервер)
# ================================================================

import psycopg2
from pgvector.psycopg2 import register_vector
from sentence_transformers import SentenceTransformer
import json

def load_documents(json_path="./extracted_data.json"):
    with open(json_path, "r", encoding="utf-8") as f:
        return json.load(f)

# 1. Подключение к PostgreSQL
conn = psycopg2.connect(
    dbname="postgres",
    user="postgres",
    password="password",
    host="localhost"
)

# 2. Регистрация типа vector для psycopg2
register_vector(conn)
cur = conn.cursor()

# 3. Создание таблицы и расширения
cur.execute("CREATE EXTENSION IF NOT EXISTS vector;")
cur.execute("DROP TABLE IF EXISTS documents;")
cur.execute("""
    CREATE TABLE documents (
        id SERIAL PRIMARY KEY,
        text TEXT,
        source TEXT,
        embedding vector(312)
    );
""")
conn.commit()

# 4. Вставка данных
docs = load_documents("./extracted_data.json")
model = SentenceTransformer("cointegrated/rubert-tiny2")

print(f"📥 Вставка {len(docs)} документов...")
for doc in docs:
    emb = model.encode(doc["text"], normalize_embeddings=True).tolist()
    cur.execute(
        "INSERT INTO documents (text, source, embedding) VALUES (%s, %s, %s)",
        (doc["text"], doc["metadata"].get("source", ""), emb)
    )
conn.commit()
print(f"✅ Добавлено {len(docs)} документов.")

# 5. Создание индекса HNSW для ускорения поиска
cur.execute("""
    CREATE INDEX ON documents
    USING hnsw (embedding vector_cosine_ops)
    WITH (m = 16, ef_construction = 64);
""")
conn.commit()
print("✅ Создан индекс HNSW.")

# 6. Поиск
query = "налог на прибыль"
qe = model.encode(query, normalize_embeddings=True).tolist()

cur.execute("""
    SELECT text, source, 1 - (embedding <=> %s) AS similarity
    FROM documents
    ORDER BY embedding <=> %s
    LIMIT 3;
""", (qe, qe))

results = cur.fetchall()

print("\n🔍 PgVector (SQL поиск):")
for text, source, similarity in results:
    print(f"  - {source}: {text[:150]}... (similarity: {similarity:.4f})")

# 7. Обязательно закрываем ресурсы
cur.close()
conn.close()
```

---

### 7.12. Сводная таблица: где нужен API-ключ

| БД | Требуется API-ключ | Где взять |
|----|-------------------|-----------|
| **Chroma** | ❌ Нет | Локальная установка |
| **FAISS** | ❌ Нет | Локальная установка |
| **Pinecone** | ✅ **ДА** | [pinecone.io](https://www.pinecone.io/) → API Keys |
| **Weaviate** | ✅ **ДА** (для облачной версии) | [weaviate.io](https://weaviate.io/) → облачный кластер |
| **Qdrant** | ❌ Нет | Локальная установка |
| **Milvus** | ❌ Нет (нужен локальный сервер) | Docker-контейнер |
| **LanceDB** | ❌ Нет | Локальная установка |
| **PgVector** | ❌ Нет (нужен PostgreSQL) | Локальный или Docker-контейнер |

---

### 7.13. Контрольные вопросы

1. *В чём разница между точным поиском (Flat) и приближённым (ANN), и когда стоит использовать каждый?*  
   **Ответ:** Точный поиск сканирует все векторы и даёт 100% точность, но медленный при больших объёмах. Подходит для датасетов < 10 000 векторов. ANN жертвует небольшой точностью (recall 95–99%) ради значительного ускорения (в 10–1000 раз). Подходит для продакшена с миллионами векторов.

2. *Какая векторная БД лучше всего подходит для гибридного поиска (векторный + лексический)?*  
   **Ответ:** Weaviate обладает встроенной поддержкой гибридного поиска (BM25 + векторы) с параметром `alpha` для баланса. Qdrant также поддерживает гибридный поиск, но Weaviate предлагает более богатый функционал в этой области.

3. *Какую БД выбрать для проекта с ограниченной оперативной памятью, но большим объёмом данных?*  
   **Ответ:** LanceDB хранит данные на диске и не загружает весь индекс в оперативную память, что делает его идеальным для проектов с ограниченной RAM. PgVector также может использовать индексы, хранящиеся на диске, но требует PostgreSQL.

---

### 7.14. Задания

1. **Построение векторной БД в Chroma.** Используя `extracted_data.json` из раздела 4, создайте коллекцию Chroma. Выполните 3 поисковых запроса с фильтрацией по источнику. Замерьте время выполнения каждого запроса.

2. **Сравнение FAISS и Qdrant.** Для одного и того же набора документов создайте индексы в FAISS и Qdrant. Сравните время поиска для 5 запросов. Сделайте вывод, какая БД быстрее и почему.

3. **Настройка гибридного поиска в Weaviate.** Используя Weaviate, настройте гибридный поиск с разными значениями `alpha` (0.0, 0.3, 0.5, 0.7, 1.0). Для каждого значения выполните 3 запроса и запишите результаты. Сделайте вывод, какое значение `alpha` даёт наилучшее качество.

---

### 7.15. Список литературы

1. **FAISS Documentation.** – https://github.com/facebookresearch/faiss
2. **Chroma Documentation.** – https://docs.trychroma.com/
3. **Pinecone Documentation.** – https://docs.pinecone.io/
4. **Weaviate Documentation.** – https://weaviate.io/developers/weaviate
5. **Qdrant Documentation.** – https://qdrant.tech/documentation/
6. **Milvus Documentation.** – https://milvus.io/docs
7. **LanceDB Documentation.** – https://lancedb.github.io/lancedb/
8. **PgVector Documentation.** – https://github.com/pgvector/pgvector

---

Этот раздел даёт полное практическое руководство по работе со всеми популярными векторными БД. Каждый пример кода использует `extracted_data.json` из раздела 4 и модель `cointegrated/rubert-tiny2`, что обеспечивает единый контекст и возможность сравнения. Для Pinecone и Weaviate указаны точные инструкции по получению API-ключей.


## Тема 8. Продвинутые архитектуры RAG

После изучения базовых компонентов RAG — от индексации и чанкинга до эмбеддингов и векторных баз данных — мы переходим к самому интересному: продвинутым архитектурам, которые превращают RAG из линейного пайплайна в интеллектуальную, адаптивную систему. Если классический RAG — это «запрос → поиск → генерация» без отклонений, то продвинутые архитектуры наделяют систему способностью **мыслить, рефлексировать и принимать решения**. В этой теме мы разберём четыре ключевых подхода: Agentic RAG, Adaptive RAG, Self‑RAG и Corrective RAG (CRAG), сравним их и покажем, как выбирать архитектуру под конкретную задачу.

---

### 8.1. Agentic RAG: LLM как автономный агент

#### 8.1.1. Идея агентного подхода

**Agentic RAG** — это парадигма, в которой LLM выступает не просто генератором ответа, а **автономным агентом**, способным самостоятельно планировать действия, использовать инструменты и принимать решения о том, когда и как искать информацию. Вместо однократного поиска агент может выполнять **многошаговые циклы** «мысль → действие → наблюдение» (ReAct — Reasoning + Acting).

**Ключевое отличие от классического RAG:**

| Характеристика | Классический RAG | Agentic RAG |
|----------------|------------------|-------------|
| **Количество шагов** | Один (поиск → генерация) | Множество (циклы ReAct) |
| **Принятие решений** | Отсутствует | Агент сам решает, что делать |
| **Использование инструментов** | Только векторный поиск | Поиск, веб-поиск, суммаризация, калькулятор и др. |
| **Адаптивность** | Фиксированный пайплайн | Динамическая стратегия |

**ReAct-паттерн** лежит в основе Agentic RAG:

```
Мысль (Thought):   "Мне нужно сравнить доходы Apple и Microsoft за 5 лет"
    ↓
Действие (Action): "Поиск: доходы Apple 2020-2024"
    ↓
Наблюдение (Observation): [результаты поиска по Apple]
    ↓
Мысль (Thought):   "Теперь нужно найти данные по Microsoft"
    ↓
Действие (Action): "Поиск: доходы Microsoft 2020-2024"
    ↓
Наблюдение (Observation): [результаты поиска по Microsoft]
    ↓
Мысль (Thought):   "У меня есть данные по обеим компаниям, можно сравнивать"
    ↓
Ответ: [сравнительный анализ]
```

#### 8.1.2. Компоненты Agentic RAG

Типичная Agentic RAG-система состоит из:

1. **Агент-планировщик (Planner)** — LLM, который разбивает сложный запрос на подзадачи и определяет последовательность действий. Например, для запроса «Сравни доходы Apple и Microsoft» агент планирует два поисковых запроса.

2. **Инструменты (Tools)** — набор функций, которые агент может вызывать:
   - **Векторный поиск** — по внутренней базе документов.
   - **Веб-поиск** — для актуальных данных (Tavily, Google Search API).
   - **Суммаризация** — сжатие длинных документов.
   - **Калькулятор** — для вычислений.
   - **SQL-запросы** — к结构化 базам данных.

3. **Память (Memory)** — хранение истории действий и наблюдений, чтобы агент мог ссылаться на предыдущие шаги.

4. **Рефлексия (Reflection)** — агент может оценивать качество полученной информации и решать, нужно ли уточнить запрос или выполнить дополнительный поиск.

#### 8.1.3. Реализация с LangGraph

LangGraph — это фреймворк для построения графовых агентских систем. В отличие от линейных цепочек LangChain, LangGraph позволяет моделировать **состояния и переходы** между ними, что идеально для Agentic RAG.

**Пример архитектуры LangGraph для Agentic RAG:**

```mermaid
flowchart TD
    START([Начало]) --> Agent[Агент-планировщик]
    Agent -->|"решение: искать"| Retrieve[Поиск в БД]
    Agent -->|"решение: веб-поиск"| WebSearch[Веб-поиск]
    Agent -->|"решение: ответ готов"| Generate[Генерация ответа]
    
    Retrieve --> Evaluate[Оценка релевантности]
    WebSearch --> Evaluate
    
    Evaluate -->|"информация достаточна"| Generate
    Evaluate -->|"информация неполная"| Rewrite[Переформулировка запроса]
    Rewrite --> Retrieve
    
    Generate --> END([Ответ])
```

**Ключевые узлы LangGraph**:

- **Workflow Agent** — структурированный граф с явным контролем потока.
- **React Agent** — автономный агент на основе ReAct с предварительно построенными инструментами.

#### 8.1.4. Сквозной пример: сравнение доходов компаний

**Запрос пользователя:** *«Сравни доходы Apple и Microsoft за последние 5 лет и выдели основные тренды»*.

**Шаги агента:**

1. **Планирование:** Агент распознаёт, что запрос требует двух отдельных поисков и сравнительного анализа.
2. **Действие 1:** Поиск по внутренней БД: «доходы Apple 2020-2024». Если документов нет — запускает веб-поиск через Tavily.
3. **Наблюдение 1:** Получены данные по Apple.
4. **Действие 2:** Поиск «доходы Microsoft 2020-2024».
5. **Наблюдение 2:** Получены данные по Microsoft.
6. **Проверка:** Агент оценивает, достаточно ли данных для сравнения. Если нет — выполняет дополнительные поиски.
7. **Генерация:** Формирует структурированный ответ с таблицей и выводами.

Преимущество такого подхода — **гибкость**: если первый поиск не дал результатов, агент может переформулировать запрос или обратиться к другому источнику.

---

### 8.2. Adaptive RAG: интеллектуальная адаптация к сложности запроса

#### 8.2.1. Идея адаптивности

**Adaptive RAG** — это архитектура, которая **анализирует сложность запроса** и динамически выбирает оптимальную стратегию ответа. Вместо того чтобы всегда выполнять поиск (что дорого и медленно) или всегда отвечать из памяти модели (что неточно), Adaptive RAG принимает решение на основе анализа запроса.

**Ключевая метафора:** Adaptive RAG «думает, прежде чем действовать». Если вопрос простой («Кто написал "Войну и мир"?»), система отвечает без поиска. Если вопрос сложный («Сравни налоговые режимы в разных странах ЕС»), система выполняет несколько поисковых запросов.

#### 8.2.2. Уровни сложности и стратегии

| Уровень сложности | Пример запроса | Стратегия | Действие |
|-------------------|----------------|-----------|----------|
| **Простой** | «Столица Франции» | Без поиска | LLM отвечает из своих знаний |
| **Средний** | «Какие налоги платят самозанятые в 2025 году?» | Один поиск | Векторный поиск + генерация |
| **Сложный** | «Сравни налоговые ставки в России, США и Германии» | Много поисков | Несколько запросов + синтез |

#### 8.2.3. Реализация: классификатор сложности запроса

Ключевой компонент Adaptive RAG — **классификатор сложности запроса (query complexity classifier)**. Это может быть:

- **Небольшая LLM** (например, T5 или DistilBERT), дообученная на размеченных данных классифицировать запросы по сложности.
- **Эвристика** — по длине запроса, наличию вопросительных слов, количеству ключевых сущностей.
- **Правила** — например, если запрос содержит «сравни», «отличие», «анализ» — назначается высокий уровень сложности.

**Архитектура Adaptive RAG**:

```mermaid
flowchart TD
    Q[Запрос пользователя] --> Classifier[Классификатор сложности]
    
    Classifier -->|"простой"| Direct[LLM без поиска]
    Classifier -->|"средний"| SimpleRAG[Один поиск → генерация]
    Classifier -->|"сложный"| ComplexRAG[Много поисков → синтез]
    
    Direct --> Response[Ответ]
    SimpleRAG --> Response
    ComplexRAG --> Response
```

#### 8.2.4. Преимущества Adaptive RAG

1. **Экономия токенов и времени** — простые запросы обрабатываются без затрат на поиск.
2. **Повышение качества** — сложные запросы получают более глубокую проработку.
3. **Масштабируемость** — система может обрабатывать смешанный трафик (простые и сложные вопросы) без потери эффективности.

**Реальные примеры использования:** системы поддержки клиентов (где 80% вопросов — простые FAQ) и исследовательские ассистенты (где требуются глубокие аналитические запросы).

---

### 8.3. Self‑RAG и Corrective RAG (CRAG): рефлексия и самокоррекция

Self‑RAG и CRAG — это две архитектуры, которые добавляют в RAG **механизмы самоконтроля и исправления ошибок**. Они решают одну и ту же проблему — низкое качество retrieved-документов — но разными способами.

#### 8.3.1. Self‑RAG: рефлексия через специальные токены

**Self‑RAG** (Self-Reflective Retrieval-Augmented Generation) — это модель, которая **генерирует специальные токены рефлексии (reflection tokens)** для оценки собственных действий и качества информации.

**Ключевая идея:** Self‑RAG — это **модель-центричный** подход, где одна LLM управляет поиском, генерацией и критикой. В отличие от Agentic RAG, где решения принимает внешний планировщик, здесь все механизмы встроены в саму модель.

**Типы рефлексивных токенов**:

| Токен | Назначение |
|-------|------------|
| **`[Retrieve]` / `[No Retrieval]`** | Решение, нужен ли поиск |
| **`[Relevant]` / `[Irrelevant]`** | Оценка релевантности найденных документов |
| **`[Support]` / `[No Support]`** | Проверка, подтверждает ли документ факты в ответе |
| **`[Useful]` / `[No Useful]`** | Оценка полезности документа для ответа |
| **`[Complete]`** | Сигнал, что ответ готов |

**Процесс Self‑RAG**:

1. **Решение о поиске:** Модель генерирует токен `[Retrieve]` или `[No Retrieval]`. Если `[No Retrieval]` — модель отвечает без внешних данных.
2. **Поиск и генерация:** Если `[Retrieve]`, модель получает Top‑K чанков и для каждого генерирует ответ с токенами рефлексии.
3. **Оценка:** Модель генерирует токены `[Relevant]`/`[Irrelevant]` для каждого чанка и `[Support]`/`[No Support]` для проверки фактов.
4. **Выбор лучшего:** Модель ранжирует кандидатов по токенам рефлексии и выбирает лучший ответ.

**Диаграмма Self‑RAG:**

```mermaid
flowchart TD
    Q[Запрос] --> Model[LLM]
    Model -->|"[Retrieve]"| Retrieve[Поиск документов]
    Model -->|"[No Retrieval]"| Generate[Генерация ответа]
    
    Retrieve --> Chunks[Top-K чанков]
    Chunks --> Eval1["[Relevant]/[Irrelevant]"]
    Eval1 --> Gen[Генерация для каждого чанка]
    Gen --> Eval2["[Support]/[No Support]"]
    Eval2 --> Rank[Ранжирование кандидатов]
    Rank --> Best[Выбор лучшего]
    Best --> Response[Ответ]
    
    Generate --> Response
```

**Преимущества Self‑RAG:**
- Модель сама контролирует качество, не требуя внешних оценщиков.
- Позволяет адаптивно регулировать количество поисков.
- Обеспечивает высокую фактическую точность (factual accuracy).

**Ограничения:**
- Требует дообучения модели на данных с рефлексивными токенами.
- Увеличивает вычислительные затраты (множественные вызовы модели для одного запроса).

#### 8.3.2. CRAG (Corrective RAG): коррекция через оценку релевантности

**CRAG** (Corrective Retrieval-Augmented Generation) — это архитектура, которая **оценивает качество найденных документов** и, при необходимости, запускает корректирующие действия.

**Ключевая идея:** CRAG добавляет **лёгкий оценщик релевантности (lightweight retrieval evaluator)**, который анализирует retrieved-документы и принимает решение: «использовать как есть», «исправить» или «искать заново».

**Компоненты CRAG**:

1. **Retriever** — стандартный поиск (векторный, гибридный или оба).
2. **Retrieval Evaluator** — модель (обычно небольшая LLM или классификатор), которая оценивает релевантность каждого документа и всей выборки.
3. **Действия (Actions)** — на основе оценки выбирается одно из действий:
   - **`Correct`** — документы релевантны → передаются в LLM без изменений.
   - **`Incorrect`** — документы нерелевантны → запускается веб-поиск или переформулировка запроса.
   - **`Ambiguous`** — неясно, релевантны ли документы → применяется дополнительная фильтрация.
4. **Корректирующий механизм:** При `Incorrect` система выполняет **веб-поиск** для дополнения информации или **переформулирует запрос** и выполняет повторный поиск.

**Архитектура CRAG**:

```mermaid
flowchart TD
    Q[Запрос] --> Retrieve[Поиск документов]
    Retrieve --> Evaluator[Retrieval Evaluator]
    
    Evaluator -->|"Correct"| Generate[Генерация ответа]
    Evaluator -->|"Ambiguous"| Filter[Дополнительная фильтрация]
    Evaluator -->|"Incorrect"| Correct[Корректирующие действия]
    
    Filter --> Generate
    Correct -->|"Веб-поиск"| WebSearch[Поиск в интернете]
    Correct -->|"Переформулировка"| Rewrite[Новый запрос]
    WebSearch --> Generate
    Rewrite --> Retrieve
    
    Generate --> Response[Ответ]
```

**Преимущества CRAG:**
- **Plug‑and‑play** — может быть добавлен к любой существующей RAG-системе.
- **Снижение галлюцинаций** — блокирует попадание нерелевантных документов в LLM.
- **Повышение надёжности** — особенно важен для высокорисковых сценариев (медицина, финансы).

**Ограничения:**
- Добавляет задержку (оценка релевантности требует времени).
- Зависит от качества retrieval evaluator.

#### 8.3.3. Self‑RAG vs CRAG: ключевые различия

| Критерий | Self‑RAG | CRAG |
|----------|----------|------|
| **Фокус** | Рефлексивная генерация | Корректирующий поиск |
| **Механизм** | Специальные токены рефлексии | Оценщик релевантности + действия |
| **Где происходит коррекция** | В процессе генерации (on-the-fly) | До генерации (pre‑generation) |
| **Требования к модели** | Дообучение на токенах рефлексии | Можно использовать готовые LLM |
| **Сложность** | Высокая (нужно дообучение) | Средняя (plug‑and‑play) |
| **Затраты** | Множество вызовов LLM | Один дополнительный вызов оценщика |

**Когда что использовать:**
- **Self‑RAG** — если у вас есть возможность дообучить модель и нужна максимальная точность фактов.
- **CRAG** — если вы хотите улучшить существующую RAG-систему без дообучения модели.
- **Комбинация** — CRAG может очистить и скорректировать retrieved-документы, а Self‑RAG — затем проверить и уточнить ответ.

---

### 8.4. Сравнение архитектур

#### 8.4.1. Сравнительная таблица

| Критерий | Naive RAG | Advanced RAG | Adaptive RAG | Self‑RAG | CRAG | Agentic RAG |
|----------|-----------|--------------|--------------|----------|------|-------------|
| **Сложность реализации** | Низкая | Средняя | Средняя | Высокая | Средняя | Высокая |
| **Качество ответа** | Среднее | Хорошее | Хорошее | Очень высокое | Высокое | Очень высокое |
| **Гибкость** | Низкая | Средняя | Высокая | Средняя | Средняя | Очень высокая |
| **Время ответа** | Низкое | Среднее | Низкое–Среднее | Высокое | Среднее | Высокое |
| **Стоимость токенов** | Низкая | Средняя | Низкая–Средняя | Высокая | Средняя | Высокая |
| **Самоконтроль** | Нет | Нет | Нет | Да (рефлексия) | Да (оценка) | Да (планирование) |
| **Использование инструментов** | Только поиск | Поиск + реранкинг | Поиск (выборочно) | Поиск | Поиск + веб-поиск | Множество инструментов |

#### 8.4.2. Диаграмма выбора архитектуры

```mermaid
flowchart TD
    Start([Какая задача?]) --> Q1{Нужна ли высокая<br>точность фактов?}
    
    Q1 -->|Нет| Q2{Сложные,<br>многошаговые запросы?}
    Q1 -->|Да| Q3{Есть возможность<br>дообучить модель?}
    
    Q2 -->|Нет| Naive[Naive RAG]
    Q2 -->|Да| Agentic[Agentic RAG]
    
    Q3 -->|Да| SelfRAG[Self‑RAG]
    Q3 -->|Нет| Q4{Нужно улучшить<br>существующую RAG?}
    
    Q4 -->|Да| CRAG[CRAG]
    Q4 -->|Нет| Advanced[Advanced RAG]
    
    Naive --> End[Выбрано]
    Agentic --> End
    SelfRAG --> End
    CRAG --> End
    Advanced --> End
```

#### 8.4.3. Рекомендации по выбору

| Сценарий | Рекомендуемая архитектура | Обоснование |
|----------|--------------------------|-------------|
| **Прототип, ограниченный бюджет** | Naive RAG | Минимальная сложность и стоимость |
| **Корпоративный поиск, средняя точность** | Advanced RAG | Хороший баланс качества и стоимости |
| **Поддержка клиентов (смешанные запросы)** | Adaptive RAG | Экономит токены на простых вопросах |
| **Медицина, юриспруденция (высокая точность)** | Self‑RAG или CRAG | Механизмы проверки фактов |
| **Исследования, аналитика (сложные запросы)** | Agentic RAG | Многошаговое планирование и инструменты |
| **Есть существующая RAG, нужно улучшить** | CRAG | Plug‑and‑play, не требует дообучения |

---

### 8.5. Контрольные вопросы

1. *В чём ключевое различие между Agentic RAG и Self‑RAG с точки зрения архитектуры?*  
   **Ответ:** Agentic RAG использует внешнего агента-планировщика (часто на базе LangGraph), который принимает решения о поиске и использовании инструментов. Self‑RAG — это модель-центричный подход, где одна LLM генерирует специальные токены рефлексии для самоконтроля.

2. *Когда Adaptive RAG выбирает стратегию «без поиска», и почему это выгодно?*  
   **Ответ:** Adaptive RAG использует классификатор сложности запроса. Для простых вопросов (например, фактологических, на которые модель знает ответ) стратегия «без поиска» экономит время и токены, снижая стоимость каждого запроса.

3. *В каких сценариях CRAG предпочтительнее Self‑RAG?*  
   **Ответ:** CRAG предпочтительнее, когда нет возможности дообучать модель (plug‑and‑play), и когда нужно быстро улучшить существующую RAG-систему. Self‑RAG требует дообучения модели на данных с рефлексивными токенами, но даёт более высокую точность фактов.

---

### 8.6. Задания

1. **Проектирование Agentic RAG для банка.** Спроектируйте Agentic RAG-систему для поддержки клиентов банка. Опишите:
   - Какие инструменты (tools) понадобятся агенту (поиск по документам, веб-поиск курсов валют, калькулятор кредитов и т.д.).
   - Пример диалога агента для запроса: *«Сравни условия ипотеки в нашем банке и в Сбербанке»*.
   - Какие узлы будут в LangGraph-графе.

2. **Сравнение Self‑RAG и CRAG.** Напишите эссе (1–2 страницы) на тему: *«Self‑RAG vs CRAG: различия, области применения и возможность комбинирования»*. В эссе обязательно затроньте:
   - Механизмы работы каждой архитектуры.
   - Когда какую выбирать.
   - Может ли CRAG и Self‑RAG работать вместе и как.

---

### 8.7. Список литературы

1. **Asai, A., et al. (2024).** *Self-RAG: Learning to Retrieve, Generate, and Critique through Self-Reflection*. – ICLR 2024. Оригинальная статья Self‑RAG.

2. **Jeong, S., et al. (2024).** *Adaptive-RAG: Learning to Adapt Retrieval-Augmented Large Language Models through Question Complexity*. – arXiv:2403.14403. Оригинальная статья Adaptive RAG.

3. **Yan, S., et al. (2024).** *Corrective Retrieval Augmented Generation*. – Работа, представляющая CRAG.

4. **Agentic RAG Survey (2025).** *From Traditional RAG to Agentic RAG: Evolution and Future Directions*. – arXiv:2501.09136v3.

5. **LangGraph Documentation.** – https://langchain-ai.github.io/langgraph/ – фреймворк для построения Agentic RAG.

6. **ReAct: Synergizing Reasoning and Acting in Language Models.** – ICLR 2023. Оригинальная статья о паттерне ReAct.

7. **IBM Tutorial: Build a self-RAG agent with IBM Granite LLMs.** – https://www.ibm.com/think/tutorials/build-self-rag-agent-langgraph-granite.



# Сквозной пример: Полный RAG-пайплайн для корпоративного ассистента

В этом примере мы построим RAG-систему для поддержки сотрудников компании. Система будет отвечать на вопросы по внутренней документации (политики, инструкции, регламенты). Мы пройдём все этапы:

1. **Извлечение и очистка текста** (Тема 4)
2. **Чанкинг** (Тема 5)
3. **Генерация эмбеддингов** (Тема 6)
4. **Построение векторной БД** (Тема 7)
5. **Поиск и генерация ответа** (Тема 1–2)
6. **Оценка качества** (Тема 3)
7. **Адаптивный RAG** (Тема 8)


In [ ]:
# ================================================================
# УСТАНОВКА БИБЛИОТЕК
# ================================================================

!pip install -q \
    pypdf \
    pdfplumber \
    sentence-transformers \
    chromadb \
    langchain-text-splitters \
    datasets \
    evaluate \
    sacrebleu \
    matplotlib \
    "numpy<2.1"

    # ================================================================
# СКВОЗНОЙ ПРИМЕР: ПОЛНЫЙ RAG-ПАЙПЛАЙН
#
# Объединяет все темы Лекции 5.1:
# - Извлечение текста из PDF (Тема 4)
# - Чанкинг (Тема 5)
# - Эмбеддинги (Тема 6)
# - Векторная БД (Тема 7)
# - Поиск и генерация (Тема 1-2)
# - Оценка качества (Тема 3)
# - Адаптивный RAG (Тема 8)
# ================================================================

import os
import json
import re
import time
import pickle
from pathlib import Path
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass

import numpy as np
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import pdfplumber
from pypdf import PdfReader
import evaluate

# --------------------------------------------
# ЧАСТЬ 1: ИЗВЛЕЧЕНИЕ ТЕКСТА (Тема 4)
# --------------------------------------------

def extract_pdf_text(file_path: str) -> str:
    """
    Извлекает текст из PDF с fallback между pdfplumber и pypdf.
    """
    text = ""
    try:
        with pdfplumber.open(file_path) as pdf:
            for page in pdf.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
        if text.strip():
            return text
    except Exception as e:
        print(f"pdfplumber не сработал для {file_path}: {e}")

    try:
        with open(file_path, "rb") as f:
            reader = PdfReader(f)
            for page in reader.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
    except Exception as e:
        print(f"Не удалось извлечь текст из {file_path}: {e}")

    return text.strip()


def clean_text(text: str) -> str:
    """Очищает текст от шума."""
    if not text:
        return ""
    text = re.sub(r'[\x00-\x08\x0B\x0C\x0E-\x1F\x7F]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def load_documents_from_folder(folder_path: str = "./documents") -> List[Dict]:
    """
    Загружает все PDF из папки и возвращает список документов с метаданными.
    """
    results = []
    pdf_files = list(Path(folder_path).glob("**/*.pdf"))

    if not pdf_files:
        print(f"⚠️ PDF файлы не найдены в {folder_path}")
        print("Создаю демонстрационные документы...")
        return create_demo_documents()

    print(f"📁 Найдено {len(pdf_files)} PDF файлов")

    for pdf_path in pdf_files:
        text = extract_pdf_text(str(pdf_path))
        if not text:
            continue
        cleaned_text = clean_text(text)
        results.append({
            "text": cleaned_text,
            "metadata": {
                "source": pdf_path.name,
                "file_path": str(pdf_path),
                "doc_id": pdf_path.stem,
            }
        })
        print(f"  ✅ Загружен: {pdf_path.name} ({len(cleaned_text)} символов)")

    return results


def create_demo_documents() -> List[Dict]:
    """
    Создаёт демонстрационные документы для примера.
    """
    docs = [
        {
            "text": """
            Налог на прибыль организаций регулируется главой 25 Налогового кодекса РФ.
            Ставка налога составляет 20% от прибыли. При этом 3% зачисляется в федеральный бюджет,
            17% — в региональный бюджет. Для IT-компаний предусмотрена льготная ставка 17%.
            Льгота действует при условии, что доля IT-доходов составляет не менее 70%.
            """,
            "metadata": {"source": "Налоговый кодекс РФ", "doc_id": "tax_code"}
        },
        {
            "text": """
            Самозанятые граждане уплачивают налог на профессиональный доход (НПД).
            Ставка налога составляет 4% при работе с физическими лицами и 6% при работе
            с юридическими лицами. Максимальный годовой доход для применения НПД — 2.4 млн рублей.
            Налог уплачивается ежемесячно до 25 числа следующего месяца.
            """,
            "metadata": {"source": "Закон о НПД", "doc_id": "npd_law"}
        },
        {
            "text": """
            Индивидуальные предприниматели обязаны вести книгу учёта доходов и расходов (КУДиР).
            Отчётность сдаётся в налоговую по месту регистрации. Срок сдачи декларации —
            30 апреля следующего года. УСН позволяет уплачивать налог по ставке 6% от доходов
            или 15% от доходов минус расходы.
            """,
            "metadata": {"source": "Инструкция для ИП", "doc_id": "ip_guide"}
        },
        {
            "text": """
            Страховые взносы во внебюджетные фонды уплачиваются всеми работодателями.
            В 2024 году ставка составляет 30% от фонда оплаты труда. Пенсионный фонд — 22%,
            Фонд социального страхования — 2.9%, Фонд обязательного медицинского страхования — 5.1%.
            Для малого бизнеса предусмотрена пониженная ставка 15% на сумму превышения МРОТ.
            """,
            "metadata": {"source": "Страховые взносы", "doc_id": "insurance"}
        }
    ]
    return docs


# --------------------------------------------
# ЧАСТЬ 2: ЧАНКИНГ (Тема 5)
# --------------------------------------------

class RecursiveTextSplitter:
    """Рекурсивный сплиттер для разбиения текста на чанки."""

    def __init__(self, chunk_size: int = 500, chunk_overlap: int = 50):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.separators = ["\n\n", "\n", ". ", "! ", "? ", ", ", " "]

    def split_document(self, text: str, metadata: Dict) -> List[Dict]:
        chunks_text = self._split_text(text)
        chunks_with_meta = []
        for i, chunk_text in enumerate(chunks_text):
            chunk_meta = metadata.copy()
            chunk_meta["chunk_index"] = i
            chunk_meta["chunk_length"] = len(chunk_text)
            chunks_with_meta.append({"text": chunk_text, "metadata": chunk_meta})
        return chunks_with_meta

    def _split_text(self, text: str) -> List[str]:
        if not text:
            return []
        chunks = []
        current_chunk = []
        current_len = 0
        segments = self._split_by_separators(text, self.separators)

        for segment in segments:
            seg_len = len(segment)
            if current_len + seg_len > self.chunk_size and current_chunk:
                chunks.append("".join(current_chunk).strip())
                overlap_text = self._get_overlap("".join(current_chunk), self.chunk_overlap)
                current_chunk = [overlap_text]
                current_len = len(overlap_text)
            current_chunk.append(segment)
            current_len += seg_len

        if current_chunk:
            chunks.append("".join(current_chunk).strip())
        return chunks

    def _split_by_separators(self, text: str, separators: List[str]) -> List[str]:
        if not text:
            return []
        separator = separators[0]
        remaining_seps = separators[1:]
        if not remaining_seps:
            return text.split(separator)
        parts = text.split(separator)
        result = []
        for i, part in enumerate(parts):
            if len(part) <= self.chunk_size:
                result.append(part)
            else:
                sub_parts = self._split_by_separators(part, remaining_seps)
                result.extend(sub_parts)
            if i < len(parts) - 1:
                result.append(separator)
        return result

    def _get_overlap(self, text: str, overlap_len: int) -> str:
        return text[-overlap_len:] if len(text) > overlap_len else text


def process_chunking(documents: List[Dict],
                     chunk_size: int = 500,
                     chunk_overlap: int = 50) -> List[Dict]:
    """Применяет чанкинг ко всем документам."""
    splitter = RecursiveTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    all_chunks = []
    for doc in documents:
        chunks = splitter.split_document(doc["text"], doc["metadata"])
        all_chunks.extend(chunks)
    print(f"📊 Создано {len(all_chunks)} чанков (размер={chunk_size}, overlap={chunk_overlap})")
    return all_chunks


# --------------------------------------------
# ЧАСТЬ 3: ЭМБЕДДИНГИ (Тема 6)
# --------------------------------------------

class EmbeddingGenerator:
    """Генератор эмбеддингов с автоматическим определением устройства."""

    def __init__(self, model_name: str = "cointegrated/rubert-tiny2", device: str = None):
        import torch
        if device is None:
            device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model = SentenceTransformer(model_name, device=device)
        self.model_name = model_name
        self.dimension = self.model.get_sentence_embedding_dimension()
        print(f"🧠 Модель: {model_name}, размерность: {self.dimension}, устройство: {device}")

    def encode_batch(self, texts: List[str], batch_size: int = 32) -> np.ndarray:
        return self.model.encode(
            texts,
            batch_size=batch_size,
            show_progress_bar=True,
            normalize_embeddings=True,
            convert_to_numpy=True,
        )


def generate_embeddings_for_chunks(chunks: List[Dict],
                                   model_name: str = "cointegrated/rubert-tiny2") -> np.ndarray:
    """Генерирует эмбеддинги для всех чанков."""
    generator = EmbeddingGenerator(model_name)
    texts = [chunk["text"] for chunk in chunks]
    embeddings = generator.encode_batch(texts, batch_size=32)
    print(f"✅ Сгенерировано {len(embeddings)} эмбеддингов размерности {embeddings.shape[1]}")
    return embeddings


# --------------------------------------------
# ЧАСТЬ 4: ВЕКТОРНАЯ БД (Тема 7)
# --------------------------------------------

def build_chroma_db(chunks: List[Dict], embeddings: np.ndarray, persist_dir: str = "./chroma_db") -> chromadb.Collection:
    """Строит векторную БД в Chroma."""
    client = chromadb.PersistentClient(
        path=persist_dir,
        settings=Settings(anonymized_telemetry=False)
    )

    # Удаляем старую коллекцию
    try:
        client.delete_collection("documents")
    except:
        pass

    collection = client.create_collection(
        name="documents",
        metadata={"hnsw:space": "cosine"}
    )

    texts = [chunk["text"] for chunk in chunks]
    metadatas = [chunk["metadata"] for chunk in chunks]
    ids = [f"chunk_{i:04d}" for i in range(len(chunks))]

    collection.add(
        documents=texts,
        embeddings=embeddings.tolist(),
        metadatas=metadatas,
        ids=ids
    )

    print(f"✅ Векторная БД построена: {len(chunks)} векторов")
    return collection


# --------------------------------------------
# ЧАСТЬ 5: ПОИСК И ГЕНЕРАЦИЯ (Тема 1-2)
# --------------------------------------------

def search_chunks(collection: chromadb.Collection,
                  query: str,
                  model: SentenceTransformer,
                  n_results: int = 3) -> List[Dict]:
    """Выполняет поиск в векторной БД."""
    query_emb = model.encode([query], normalize_embeddings=True).tolist()

    results = collection.query(
        query_embeddings=query_emb,
        n_results=n_results,
        include=["documents", "metadatas", "distances"]
    )

    top_chunks = []
    for doc, meta, dist in zip(
        results['documents'][0],
        results['metadatas'][0],
        results['distances'][0]
    ):
        top_chunks.append({
            "text": doc,
            "metadata": meta,
            "distance": dist
        })

    return top_chunks


def generate_response(query: str,
                      top_chunks: List[Dict],
                      use_llm: bool = True,
                      llm_func=None) -> str:
    """
    Генерирует ответ на основе найденных чанков.
    Если use_llm=False — возвращает простой форматированный ответ.
    """
    if not top_chunks:
        return "К сожалению, не удалось найти релевантную информацию."

    # Формируем контекст из чанков
    context = "\n\n".join([
        f"[Источник: {chunk['metadata'].get('source', 'unknown')}]\n{chunk['text']}"
        for chunk in top_chunks
    ])

    if use_llm and llm_func:
        # Используем LLM для генерации ответа
        prompt = f"""
        Ты — корпоративный ассистент. Ответь на вопрос, используя только предоставленный контекст.
        Если в контексте нет информации для ответа, скажи об этом честно.
        Укажи источники информации.

        Контекст:
        {context}

        Вопрос: {query}

        Ответ:
        """
        return llm_func(prompt)
    else:
        # Простой форматированный ответ (без LLM)
        response = f"**Вопрос:** {query}\n\n"
        response += f"**Найдено {len(top_chunks)} источников:**\n\n"
        for i, chunk in enumerate(top_chunks, 1):
            source = chunk['metadata'].get('source', 'unknown')
            response += f"**{i}. Источник: {source}**\n"
            response += f"{chunk['text'][:300]}...\n"
            response += f"*(релевантность: {1 - chunk['distance']:.3f})*\n\n"
        return response


# --------------------------------------------
# ЧАСТЬ 6: ОЦЕНКА КАЧЕСТВА (Тема 3)
# --------------------------------------------

def evaluate_search_quality(collection: chromadb.Collection,
                           model: SentenceTransformer,
                           test_queries: List[Dict]) -> Dict:
    """
    Оценивает качество поиска на тестовых запросах с известными ответами.
    """
    bleu = evaluate.load("sacrebleu")

    results = []

    for test in test_queries:
        query = test["query"]
        expected_keywords = test.get("keywords", [])

        top_chunks = search_chunks(collection, query, model, n_results=3)

        # Проверяем, содержатся ли ожидаемые ключевые слова в найденных чанках
        found_keywords = []
        for chunk in top_chunks:
            text = chunk["text"].lower()
            for kw in expected_keywords:
                if kw.lower() in text and kw not in found_keywords:
                    found_keywords.append(kw)

        recall = len(found_keywords) / len(expected_keywords) if expected_keywords else 1.0

        results.append({
            "query": query,
            "found_keywords": found_keywords,
            "expected_keywords": expected_keywords,
            "recall": recall,
            "top_sources": [chunk["metadata"].get("source", "unknown") for chunk in top_chunks],
            "top_distances": [chunk["distance"] for chunk in top_chunks],
        })

    avg_recall = np.mean([r["recall"] for r in results])

    print("\n" + "="*60)
    print("📊 ОЦЕНКА КАЧЕСТВА ПОИСКА")
    print("="*60)
    for r in results:
        status = "✅" if r["recall"] == 1.0 else "⚠️"
        print(f"{status} '{r['query']}': recall={r['recall']:.2%}, "
              f"найдено={r['found_keywords']}, ожидалось={r['expected_keywords']}")
    print(f"\nСредний Recall: {avg_recall:.2%}")
    print("="*60)

    return {"results": results, "avg_recall": avg_recall}


# --------------------------------------------
# ЧАСТЬ 7: АДАПТИВНЫЙ RAG (Тема 8)
# --------------------------------------------

class AdaptiveRAG:
    """
    Адаптивный RAG с классификацией сложности запроса.
    """

    def __init__(self,
                 collection: chromadb.Collection,
                 model: SentenceTransformer,
                 llm_func=None,
                 complexity_threshold: int = 30):
        self.collection = collection
        self.model = model
        self.llm_func = llm_func
        self.complexity_threshold = complexity_threshold

    def classify_query(self, query: str) -> str:
        """
        Классифицирует запрос по сложности.
        - 'simple': отвечаем без поиска (короткие, общие вопросы)
        - 'complex': используем полный RAG
        """
        # Эвристика: длина запроса + ключевые слова сложности
        words = query.split()
        word_count = len(words)

        complexity_keywords = ['сравни', 'отличие', 'анализ', 'детально', 'подробно',
                              'почему', 'каким образом', 'влияет', 'зависит']

        has_complex_keywords = any(kw in query.lower() for kw in complexity_keywords)

        if word_count < self.complexity_threshold and not has_complex_keywords:
            return "simple"
        else:
            return "complex"

    def answer(self, query: str) -> str:
        """Обрабатывает запрос с учётом классификации сложности."""
        complexity = self.classify_query(query)

        if complexity == "simple":
            # Простой вопрос: отвечаем без поиска (только LLM)
            print(f"🔹 Простой запрос (без поиска): '{query}'")
            if self.llm_func:
                prompt = f"Ты — корпоративный ассистент. Ответь кратко на вопрос: {query}"
                return self.llm_func(prompt)
            else:
                return f"❓ Простой вопрос: {query}\n(Для полного ответа используйте LLM)"
        else:
            # Сложный вопрос: используем полный RAG
            print(f"🔸 Сложный запрос (с поиском): '{query}'")
            top_chunks = search_chunks(self.collection, query, self.model, n_results=3)
            return generate_response(query, top_chunks, use_llm=bool(self.llm_func), llm_func=self.llm_func)


# --------------------------------------------
# ЧАСТЬ 8: ВИЗУАЛИЗАЦИЯ
# --------------------------------------------

def plot_results(results: Dict, save_path: str = "./rag_results_plot.png"):
    """Визуализирует результаты оценки."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # График 1: Recall по запросам
    recalls = [r["recall"] for r in results["results"]]
    queries = [r["query"][:30] + "..." if len(r["query"]) > 30 else r["query"] for r in results["results"]]

    axes[0].bar(queries, recalls, color=['green' if r == 1 else 'orange' for r in recalls])
    axes[0].set_ylabel('Recall')
    axes[0].set_title('Качество поиска по запросам')
    axes[0].set_ylim(0, 1.1)
    axes[0].axhline(y=1.0, color='green', linestyle='--', label='Идеальный recall')
    axes[0].legend()
    axes[0].tick_params(axis='x', rotation=45)

    # График 2: Распределение расстояний
    all_distances = []
    for r in results["results"]:
        all_distances.extend(r["top_distances"])

    axes[1].hist(all_distances, bins=20, alpha=0.7, color='blue', edgecolor='black')
    axes[1].set_xlabel('Косинусное расстояние (0 = близко)')
    axes[1].set_ylabel('Частота')
    axes[1].set_title('Распределение расстояний до найденных чанков')
    axes[1].axvline(x=np.mean(all_distances), color='red', linestyle='--', label=f'Среднее: {np.mean(all_distances):.3f}')
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"📊 График сохранён: {save_path}")


# --------------------------------------------
# ГЛАВНАЯ ФУНКЦИЯ
# --------------------------------------------

def main():
    """Запускает полный RAG-пайплайн."""
    print("\n" + "="*70)
    print("🚀 ЗАПУСК ПОЛНОГО RAG-ПАЙПЛАЙНА")
    print("="*70)

    # === ШАГ 1: Загрузка документов (Тема 4) ===
    print("\n📂 ШАГ 1: ЗАГРУЗКА ДОКУМЕНТОВ")
    print("-"*50)
    documents = load_documents_from_folder("./documents")
    print(f"✅ Загружено {len(documents)} документов")

    # === ШАГ 2: Чанкинг (Тема 5) ===
    print("\n✂️ ШАГ 2: ЧАНКИНГ")
    print("-"*50)
    chunks = process_chunking(documents, chunk_size=500, chunk_overlap=50)

    # === ШАГ 3: Эмбеддинги (Тема 6) ===
    print("\n🧠 ШАГ 3: ГЕНЕРАЦИЯ ЭМБЕДДИНГОВ")
    print("-"*50)
    model_name = "cointegrated/rubert-tiny2"
    embeddings = generate_embeddings_for_chunks(chunks, model_name)

    # === ШАГ 4: Векторная БД (Тема 7) ===
    print("\n🗄️ ШАГ 4: ПОСТРОЕНИЕ ВЕКТОРНОЙ БД")
    print("-"*50)
    collection = build_chroma_db(chunks, embeddings, "./chroma_db")

    # === ШАГ 5: Поиск и генерация (Тема 1-2) ===
    print("\n🔍 ШАГ 5: ПОИСК И ГЕНЕРАЦИЯ")
    print("-"*50)

    model = SentenceTransformer(model_name)

    test_queries = [
        "Какой налог платят самозанятые?",
        "Какая ставка налога на прибыль для IT-компаний?",
        "Что такое КУДиР и кто его должен вести?",
        "Какой процент страховых взносов платят работодатели?",
    ]

    print("\nРезультаты поиска:")
    for query in test_queries:
        print(f"\n📌 Вопрос: {query}")
        top_chunks = search_chunks(collection, query, model, n_results=3)

        for i, chunk in enumerate(top_chunks, 1):
            source = chunk["metadata"].get("source", "unknown")
            score = 1 - chunk["distance"]
            print(f"  {i}. [{source}] {chunk['text'][:120]}... (score={score:.3f})")

    # === ШАГ 6: Оценка качества (Тема 3) ===
    print("\n📊 ШАГ 6: ОЦЕНКА КАЧЕСТВА")
    print("-"*50)

    eval_queries = [
        {"query": "Какой налог платят самозанятые?", "keywords": ["НПД", "4%", "6%"]},
        {"query": "Ставка налога на прибыль", "keywords": ["20%", "IT-компаний", "17%"]},
        {"query": "Страховые взносы работодателей", "keywords": ["30%", "Пенсионный", "ФОТ"]},
    ]

    eval_results = evaluate_search_quality(collection, model, eval_queries)

    # === ШАГ 7: Адаптивный RAG (Тема 8) ===
    print("\n🔄 ШАГ 7: АДАПТИВНЫЙ RAG")
    print("-"*50)

    # Простая функция-заглушка для LLM (в реальности замените на Ollama или OpenAI)
    def dummy_llm(prompt: str) -> str:
        return f"🤖 (LLM ответ на: {prompt[:50]}...)"

    adaptive_rag = AdaptiveRAG(
        collection=collection,
        model=model,
        llm_func=dummy_llm,
        complexity_threshold=20
    )

    mixed_queries = [
        "Здравствуйте",
        "Какой налог платят самозанятые?",
        "Сравните налогообложение для самозанятых и ИП",
    ]

    print("\nТестирование адаптивного RAG:")
    for query in mixed_queries:
        print(f"\n📌 Запрос: '{query}'")
        response = adaptive_rag.answer(query)
        print(f"Ответ: {response[:200]}...")

    # === ШАГ 8: Визуализация ===
    print("\n📈 ШАГ 8: ВИЗУАЛИЗАЦИЯ")
    print("-"*50)
    plot_results(eval_results, "./rag_results_plot.png")

    # === ИТОГИ ===
    print("\n" + "="*70)
    print("✅ RAG-ПАЙПЛАЙН УСПЕШНО ЗАВЕРШЁН")
    print("="*70)
    print(f"📄 Документов: {len(documents)}")
    print(f"✂️ Чанков: {len(chunks)}")
    print(f"🧠 Размерность эмбеддингов: {embeddings.shape[1]}")
    print(f"🗄️ Векторная БД: Chroma ({len(collection.get()['ids'])} векторов)")
    print(f"📊 Средний Recall: {eval_results['avg_recall']:.2%}")
    print("="*70)


# --------------------------------------------
# ЗАПУСК
# --------------------------------------------

if __name__ == "__main__":
    main()



## 4. Итог: что покрывает этот пример

| Тема | Что демонстрируется |
|------|---------------------|
| **Тема 1** | Определение RAG, сквозной пример работы |
| **Тема 2** | Архитектура: индексация → поиск → генерация |
| **Тема 3** | Оценка качества: Recall@3, визуализация |
| **Тема 4** | Извлечение текста из PDF, очистка, метаданные |
| **Тема 5** | Рекурсивный чанкинг с параметрами |
| **Тема 6** | Генерация эмбеддингов, модель rubert-tiny2 |
| **Тема 7** | Векторная БД Chroma, поиск, расстояния |
| **Тема 8** | Адаптивный RAG с классификацией сложности |

---

## 5. Как использовать этот пример

### Для студентов
1. Скопируйте код в Jupyter Notebook или Colab.
2. Запустите все ячейки.
3. Изучите, как работают разные компоненты.
4. Измените параметры (chunk_size, model_name, threshold) и посмотрите, как меняются результаты.

### Для преподавателей
1. Используйте как демонстрацию на лекции.
2. Разбейте на части для практических занятий.
3. Попросите студентов модифицировать код (добавить новые метрики, другую БД).

### Для продакшена
1. Замените `dummy_llm` на реальную LLM (Ollama, OpenAI, vLLM).
2. Подключите реальные документы (папка `./documents`).
3. Добавьте логирование и мониторинг.

---

Этот сквозной пример объединяет **все 8 тем Лекции 5.1** в единый, работающий пайплайн. Студенты могут запустить его, изучить и модифицировать, получая полное представление о том, как строится современная RAG-система.


## Тема 9. Заключение и домашнее задание к Лекции 5.1

Мы прошли огромный путь — от определения RAG до продвинутых архитектур с саморефлексией. Теперь пришло время закрепить знания на практике. Это домашнее задание — ваш первый полноценный проект по построению RAG-системы, который охватывает все этапы: от загрузки документов до реализации адаптивного поиска. Выполнение всех пунктов даст вам навыки, необходимые для создания продакшен-систем, а дополнительные эксперименты позволят глубже понять компромиссы между качеством, скоростью и сложностью.

---

### 9.1. Итоги лекции

За время изучения Лекции 5.1 мы освоили фундаментальные принципы построения RAG-систем. Мы начали с определения RAG как гибридной архитектуры, объединяющей информационный поиск и генерацию текста, и разобрали проблемы, которые он решает: актуальность знаний, приватность данных, специализированные домены, прозрачность и экономическую эффективность.

Мы детально изучили архитектуру RAG-системы, выделив пять ключевых модулей: индексация (загрузка, очистка, чанкинг, эмбеддинги, сохранение в ВБД), поиск (векторный, лексический, гибридный), генерация (LLM), интеграция (формирование промпта) и обратная связь (сбор метрик). Мы разобрали этапы работы системы: offline-индексация и online-инференс, а также сравнили типы RAG-систем — от Naive до Agentic.

Особое внимание мы уделили методам оценки качества RAG. Мы изучили метрики поиска (Precision@k, Recall@k, MRR, MAP, NDCG), метрики генерации (Faithfulness, Answer Relevance, Context Relevance), а также комплексные фреймворки — RAGAS, ARES, TruLens, DeepEval. Мы поняли, что оценка RAG — это многокомпонентная задача, требующая раздельного анализа поиска и генерации.

В практической части мы освоили полный ETL-пайплайн: извлечение текста из PDF, очистку, разбиение на чанки (рекурсивный сплиттер), генерацию эмбеддингов с помощью sentence-transformers, работу с векторными базами данных (Chroma, FAISS, Pinecone, Qdrant и другие) и интеграцию с LLM через Ollama. Мы также узнали о продвинутых архитектурах: Adaptive RAG, Self‑RAG, CRAG и Agentic RAG, которые превращают линейный пайплайн в интеллектуальную систему с планированием, рефлексией и самокоррекцией.

Теперь, когда фундамент заложен, в Лекции 5.2 мы перейдём к полноценной реализации RAG-системы с подключением LLM, оптимизацией производительности, логированием, мониторингом и деплоем.

---

### 9.2. Обязательная часть домашнего задания

Выполните все семь пунктов. **Важно:** используйте данные и код из лекции (разделы 4–7) как основу, адаптируя их под свою тему.

#### Пункт 1. Сбор и подготовка датасета

- Выберите тему для вашей RAG-системы. Рекомендуемые темы:
  - **Техническая документация:** статьи по ML/DL, документация библиотек, руководства по Python.
  - **Юридические тексты:** законы, постановления, судебные решения.
  - **Медицинские тексты:** клинические рекомендации, статьи из PubMed.
  - **Корпоративная документация:** внутренние политики, инструкции, отчёты.
- Скачайте **10–20 документов** в формате PDF, DOCX или HTML. Можно использовать открытые источники:
  - arXiv.org (статьи по ML)
  - PubMed Central (медицинские статьи)
  - Федеральные законы РФ (https://www.consultant.ru/)
- Сохраните все документы в папку `./documents`.

#### Пункт 2. Очистка и подготовка текстов

- Используя код из раздела 4, извлеките текст из всех документов.
- Очистите текст: удалите спецсимволы, лишние пробелы, нормализуйте (приведите к нижнему регистру).
- Извлеките метаданные: имя файла, дата создания/изменения, автор (если есть).
- Сохраните результаты в `extracted_data.json` (как в разделе 4).

**Критерий успеха:** JSON-файл с полями `text` и `metadata` для каждого документа.

#### Пункт 3. Чанкинг с двумя параметрами

- Реализуйте рекурсивный чанкинг (раздел 5) для двух конфигураций:
  - **Конфигурация A:** `chunk_size=500`, `chunk_overlap=50`.
  - **Конфигурация B:** `chunk_size=1000`, `chunk_overlap=100`.
- Для каждой конфигурации подсчитайте:
  - Общее количество чанков.
  - Среднюю длину чанка.
  - Пример одного чанка из середины документа.
- Сравните результаты и сделайте вывод, какая конфигурация лучше подходит для ваших данных.

**Критерий успеха:** Два набора чанков в формате JSON с метаданными.

#### Пункт 4. Генерация эмбеддингов

- Выберите модель эмбеддингов из раздела 6. Обоснуйте свой выбор (например, `all-MiniLM-L6-v2` для прототипа, `BAAI/bge-base-en-v1.5` для точности, `intfloat/multilingual-e5-base` для мультиязычности).
- Сгенерируйте эмбеддинги для всех чанков (выбранной конфигурации).
- Реализуйте кэширование эмбеддингов (раздел 6.3.3), чтобы не пересчитывать их при повторных запусках.

**Критерий успеха:** Эмбеддинги сохранены в кэш, класс `EmbeddingGenerator` работает.

#### Пункт 5. Построение векторной базы данных

- Выберите векторную БД (рекомендуется Chroma для простоты или FAISS для высокой производительности).
- Создайте коллекцию и добавьте все чанки с их эмбеддингами и метаданными (раздел 7.4).
- Сохраните базу на диск (для Chroma — PersistentClient, для FAISS — сохраните индекс в файл).

**Критерий успеха:** Векторная БД сохранена на диск, готова к поиску.

#### Пункт 6. Функция поиска и тестирование

- Реализуйте функцию поиска, которая принимает запрос и возвращает топ‑3 чанка с их скорами.
- Протестируйте на **5–10 запросах**, релевантных вашей теме. Для каждого запроса выведите:
  - Текст запроса.
  - Топ‑3 чанка (первые 150 символов).
  - Скор (косинусное сходство).
  - Источник (имя файла).

**Критерий успеха:** Для каждого запроса выводятся релевантные чанки с источниками.

#### Пункт 7. Адаптивный RAG с классификатором

- Реализуйте **адаптивный RAG** (раздел 8.2):
  - Создайте классификатор сложности запроса (два варианта на выбор):
    - **Эвристический:** по длине запроса (например, `> 30 слов` → сложный).
    - **LLM-based:** используйте небольшую LLM (через Ollama или API) для классификации.
  - Для простых запросов (`simple`) — отвечайте без поиска, используя только LLM.
  - Для сложных запросов (`complex`) — используйте полный пайплайн (поиск → LLM с контекстом).
- Протестируйте на 5 запросах (2 простых, 3 сложных). Сравните ответы с и без поиска.

**Критерий успеха:** Система выбирает стратегию в зависимости от запроса, для простых — отвечает без поиска.

---

### 9.3. Дополнительные эксперименты (для повышенной оценки)

Выполните **не менее двух** экспериментов из списка (каждый +1 балл к итоговой оценке):

1. **Сравнение моделей эмбеддингов.** Возьмите 3 модели (например, MiniLM, BGE-base, multilingual-e5). Для 10 запросов сравните Recall@5. Постройте таблицу и сделайте вывод.

2. **Сравнение размеров чанков.** Для трёх размеров (200, 500, 1000) сравните качество поиска (Recall@5). Сделайте вывод, какой размер оптимален для ваших данных.

3. **Сравнение векторных БД.** Реализуйте поиск в Chroma и FAISS для одних и тех же данных. Сравните время поиска (10 запросов) и качество (совпадают ли топ‑5). Сделайте вывод, когда какую БД использовать.

4. **Фильтрация по метаданным.** Добавьте фильтрацию по источнику или дате. Покажите примеры запросов с фильтром и без него. Как меняется качество?

5. **Гибридный поиск (BM25 + эмбеддинги).** Реализуйте гибридный поиск (раздел 2.3, Лекция 5.2). Используйте `rank_bm25` для лексического поиска и объедините результаты с векторным поиском (взвешенная сумма). Сравните с чистым векторным.

6. **Семантический чанкинг.** Реализуйте семантический чанкинг (раздел 5.2.3) с использованием эмбеддингов и порогового значения. Сравните с рекурсивным чанкингом по качеству поиска.

---

### 9.4. Требования к отчёту и критерии оценки

#### Структура отчёта (PDF, 5–8 страниц)

1. **Титульный лист** — название работы, ФИО, дата.
2. **Постановка задачи** — какая тема, какие документы, что нужно сделать.
3. **Описание данных** — количество документов, форматы, источники.
4. **Подход и реализация**:
   - Извлечение и очистка текста (кратко, с примерами).
   - Чанкинг (сравнение конфигураций, таблица).
   - Эмбеддинги (выбор модели, обоснование).
   - Векторная БД (какую выбрали, почему).
   - Поиск (примеры запросов и результатов).
   - Адаптивный RAG (как работает классификатор).
5. **Эксперименты и результаты** (основные и дополнительные).
6. **Выводы** (что получилось, что было сложным, что можно улучшить).
7. **Код** (ссылка на GitHub репозиторий с Jupyter Notebook).

#### Критерии оценки (максимум 10 баллов)

| Критерий | Баллы |
|----------|-------|
| **Качество кода** (читаемость, структура, комментарии) | 25% (2.5) |
| **Полнота выполнения обязательных пунктов** (7 пунктов) | 30% (3.0) |
| **Анализ и выводы** (глубина, обоснованность) | 25% (2.5) |
| **Оформление отчёта** (структура, таблицы, графики) | 20% (2.0) |
| **Дополнительные эксперименты** (каждый +1 балл) | до +3 |

#### Формат сдачи

- **GitHub репозиторий** с:
  - Jupyter Notebook (с кодом и комментариями).
  - Папкой `documents/` с исходными файлами.
  - Файлами `extracted_data.json`, `chunks_data.json`.
  - README.md с инструкцией по запуску.
- **PDF-отчёт**, приложенный к репозиторию или отправленный отдельно.

---

### 9.5. Шаблон отчёта в Markdown

```markdown
# Отчёт по домашнему заданию: RAG-система для [ваша тема]

## 1. Постановка задачи
[Опишите, какую проблему решает ваша RAG-система, какие запросы она должна обрабатывать]

## 2. Данные
- Количество документов: 12
- Форматы: PDF, DOCX
- Источники: [например, статьи с arXiv]
- Тематика: [например, машинное обучение, NLP]

## 3. Извлечение и очистка
[Краткое описание, какие библиотеки использовали, какие проблемы были с PDF]

## 4. Чанкинг
| Конфигурация | chunk_size | chunk_overlap | Количество чанков | Средняя длина |
|--------------|------------|---------------|-------------------|---------------|
| A            | 500        | 50            | 127               | 478           |
| B            | 1000       | 100           | 64                | 956           |

**Вывод:** [Конфигурация А/B лучше, потому что...]

## 5. Эмбеддинги
- Модель: `all-MiniLM-L6-v2`
- Размерность: 384
- Почему выбрали: [скорость, качество, язык]

## 6. Векторная БД
- БД: Chroma
- Количество векторов: 127
- Путь сохранения: `./chroma_db`

## 7. Результаты поиска
| Запрос | Топ-1 чанк | Источник | Скор |
|--------|------------|----------|------|
| "..."  | "..."      | file.pdf | 0.87 |
...

## 8. Адаптивный RAG
- Классификатор: по длине запроса (порог 30 слов)
- Результаты:
  - Простые запросы: [примеры]
  - Сложные запросы: [примеры]

## 9. Эксперименты (обязательные и дополнительные)
[Таблицы, графики, сравнения]

## 10. Выводы
[Что получилось, что было сложно, что можно улучшить]

## 11. Ссылка на код
[GitHub репозиторий]
```

---

### 9.6. Рекомендации по выполнению

**Выбор датасета:**
- Начните с небольшого количества документов (10–15), чтобы быстрее итерировать.
- Используйте документы на одном языке (для простоты), или мультиязычные, если хотите проверить мультиязычные модели.
- Если нет своих документов, скачайте готовые датасеты: [SQuAD](https://rajpurkar.github.io/SQuAD-explorer/), [Natural Questions](https://ai.google.com/research/NaturalQuestions), или возьмите статьи с [arXiv](https://arxiv.org/).

**Настройка окружения:**
- Используйте виртуальное окружение (`conda` или `venv`).
- Установите все зависимости из файла `requirements.txt` (создайте его на основе импортов в коде).
- В Colab используйте T4 GPU для ускорения эмбеддингов.

**Отладка:**
- Начинайте с маленьких данных (1–2 документа), чтобы убедиться, что каждый этап работает.
- Используйте `print()` для проверки промежуточных результатов (длина текста, количество чанков, форма эмбеддингов).
- Если поиск возвращает нерелевантные результаты — проверьте чанкинг (возможно, чанки слишком большие) и модель эмбеддингов.

**Код:**
- Копируйте код из разделов 4–7, адаптируя его под свои данные.
- Добавляйте комментарии к каждому блоку кода.
- Сохраняйте промежуточные результаты (JSON, pickle) для ускорения экспериментов.

---

### 9.7. Контрольные вопросы для самопроверки

1. *Почему чанкинг является критическим этапом в RAG, и как размер чанка влияет на качество поиска?*  
   **Ответ:** Чанкинг определяет, какие фрагменты текста будут индексироваться и искаться. Слишком маленький чанк теряет контекст и не даёт достаточной информации для ответа. Слишком большой чанк содержит шум, размывает семантику и снижает точность поиска. Оптимальный размер зависит от модели эмбеддингов и типа документов.

2. *Какие преимущества даёт гибридный поиск (BM25 + эмбеддинги) по сравнению с чистым векторным поиском?*  
   **Ответ:** Гибридный поиск объединяет сильные стороны лексического (точные совпадения, ключевые слова) и семантического (синонимы, контекст) подходов. Он лучше работает для запросов с именами, кодами, номерами, а также для терминов, редко встречающихся в корпусе.

3. *В чём разница между Adaptive RAG и Self‑RAG?*  
   **Ответ:** Adaptive RAG анализирует сложность запроса и выбирает стратегию (без поиска, один поиск, много поисков). Self‑RAG — это модель-центричный подход, где LLM сама генерирует токены рефлексии для контроля качества поиска и генерации.

---

### 9.8. Список литературы

1. **Lewis, P., et al. (2020).** *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks*. – arXiv:2005.11401.
2. **Reimers, N., & Gurevych, I. (2019).** *Sentence-BERT: Sentence Embeddings using Siamese BERT-Networks*. – arXiv:1908.10084.
3. **Asai, A., et al. (2024).** *Self-RAG: Learning to Retrieve, Generate, and Critique through Self-Reflection*. – ICLR 2024.
4. **Jeong, S., et al. (2024).** *Adaptive-RAG: Learning to Adapt Retrieval-Augmented Large Language Models through Question Complexity*. – arXiv:2403.14403.
5. **LangChain Documentation.** – https://python.langchain.com/
6. **Chroma Documentation.** – https://docs.trychroma.com/
7. **Sentence-Transformers Documentation.** – https://www.sbert.net/

